# 🌱 GreenLedger — Campus Resource Optimisation (IC26-05-01)
### Reference campus: **Concordia University, Montréal** — real open data, nothing simulated

**In one line:** GreenLedger reads every campus meter, waste invoice and a live room camera, **recommends** what to change,
**controls** room AC/fan/lights from real occupancy, and **proves** with a ghost twin + placebo test whether a change really saved anything —
then reports the carbon honestly.

| Page (left sidebar) | What it shows |
|---|---|
| Overview | KPI cards with sparklines · aerial **campus digital twin** · campus insights · resource intelligence |
| Campus Digital Twin | real satellite map of both campuses, every building clickable |
| Live vs Ghost Twin | the real metered saving: live twin vs ghost twin, placebo test, trust test, action register |
| Buildings · Energy · Water · Waste Management | every meter and invoice, charts against the ghost twin |
| E-waste | type or upload your real devices → routes under India's E-Waste Rules 2022 |
| Carbon Footprint | Scopes 1-2-3 with sourced factors, download |
| Live Room Control | your camera → people → **fan / lights / AC switched live**, room live vs ghost twin |
| AI Recommendations | data-sized actions; approve / reject |
| Analytics | what-if sliders, alerts, discoveries |
| Reports · Settings | exports · data sources, models, every parameter |

**How to run (Google Colab):**
1. Click the 📁 folder icon and upload `DATA_OUTPUT.csv`, `WASTE_INVOICES.csv`, `WASTE_TYPE.csv`, `BUILDING_LIST.csv`
   (optional: your `ewaste_register.csv`). If you skip this, cell 2 downloads the four files from Concordia's open-data site.
2. Run the cells **one at a time, top to bottom** (Shift + Enter). Each prints a short, plain result.
3. The last cell prints two links — open the **Colab link** (or the **public link** for phones). Allow camera access on the Live Room page.

Total run time ≈ 3 minutes.

## Cell 1 · Install
Installs the libraries: app (Gradio), charts (Plotly), holiday calendar, statistics, gradient boosting (LightGBM),
people detection (Ultralytics YOLOv8) and thermal comfort (pythermalcomfort, ASHRAE 55).

In [ ]:
!pip -q install fastapi uvicorn plotly holidays scipy lightgbm ultralytics pythermalcomfort


## Cell 2 · Find the data files
Finds your uploaded CSV files (even if Colab renamed them) — or downloads them from Concordia's official open-data site.

🎤 **Tell the judges:** *"Everything comes from Concordia University's public open data — real meters and real waste invoices. We didn't simulate or fill in a single number."*

In [ ]:
import os, re, csv, glob, io, json, zipfile, datetime, warnings, urllib.request
import numpy as np
import pandas as pd
from scipy import stats
import holidays
import lightgbm as lgb
import plotly.graph_objects as go
warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)

SOURCE_URL = "https://opendata.concordia.ca/datasets/Facilities/"
FILES = ["DATA_OUTPUT.csv", "WASTE_INVOICES.csv", "WASTE_TYPE.csv", "BUILDING_LIST.csv"]
OPTIONAL = ["POINT_LIST.csv"]          # Concordia's official sensor list, used if available
OUT_DIR = os.environ.get("GL_OUT_DIR", "/content/greenledger_out" if os.path.isdir("/content") else "./greenledger_out")
DATA_DIR = os.environ.get("GL_DATA_DIR", "/content" if os.path.isdir("/content") else ".")
os.makedirs(OUT_DIR, exist_ok=True)


def find(name):
    """Find a file even if it was renamed on upload (e.g. 'DATA_OUTPUT (1).csv' or 'abc-DATA_OUTPUT.csv')."""
    stem = name[:-4].upper()
    for root in [DATA_DIR, ".", "/content", "/content/drive/MyDrive"]:
        if not root or not os.path.isdir(root):
            continue
        cands = glob.glob(os.path.join(root, "*.csv")) + glob.glob(os.path.join(root, "*", "*.csv"))
        hits = []
        for p in cands:
            s_ = re.sub(r"\(\d+\)$", "", os.path.splitext(os.path.basename(p))[0].upper().replace(" ", ""))
            if (s_ == stem or s_.endswith("-" + stem)) and os.path.getsize(p) > 0:
                hits.append(p)
        if hits:
            return sorted(hits, key=len)[0]
    return None


PATHS = {f: find(f) for f in FILES + OPTIONAL}
missing = [f for f in FILES if not PATHS[f]]
for f in missing + [o for o in OPTIONAL if not PATHS[o]]:   # try Concordia's official open-data site
    if True:
        try:
            dst = os.path.join(OUT_DIR, f)
            urllib.request.urlretrieve(SOURCE_URL + f, dst)
            PATHS[f] = dst
            print(f"Downloaded {f} from {SOURCE_URL}")
        except Exception as e:
            print(f"Could not download {f}: {e}")
missing = [f for f in FILES if not PATHS[f]]
assert not missing, (f"Missing {missing}. In Colab: click the folder icon on the left, upload the 4 CSV files, "
                     f"then run this cell again.")
for f, p in PATHS.items():
    print(f"{f:20s} → {p or 'not found (optional)'}")
print("\nData: Concordia University open data (Facilities), licence CC BY 4.0 — "
      "https://www.concordia.ca/web/open-data.html")


## Cell 3 · Read the meters
Concordia's building-automation export is many reports stacked in one file. This cell stitches them into one hourly table and names each sensor from Concordia's own API documentation.

| Meter | What it measures |
|---|---|
| ⚡ Loyola campus | electricity demand of the whole Loyola campus (kW) |
| ⚡ EV building | electricity demand of the Engineering & Visual Arts building |
| ⚡ EV + GM + MB + TD | electricity of four downtown buildings together |
| 🔥 EV + GM + MB / GM | natural gas (running counters that reset each month) |
| 💧 Municipal water | the city water supply meter (running counter) |
| 🌡️ Weather | outdoor temperature from the EV rooftop station |

In [ ]:
# ---- What each sensor is: Concordia's own API documentation (opendataConcordiaU/documentation,
#      facilities/consumption and facilities/environmental) ----
METER_INFO = {
    # point:  (system name, resource, plain name, buildings it covers, reading type, unit)
    "Point_12": ("DEMANDE.ELEC.MOYEN60S.BS", "electricity", "Loyola campus",            "LOY campus", "power", "kW"),
    "Point_7":  ("TOTKWEV.IC",               "electricity", "EV building",              "EV",          "power", "kW"),
    "Point_6":  ("TOTKWEV-MB.IC",            "electricity", "EV + GM + MB + TD buildings", "EV,GM,MB,TD", "power", "kW"),
    "Point_8":  ("MDI.001-650.IC",           "gas",         "EV + GM + MB buildings",   "EV,GM,MB",    "counter", "m³ (as documented)"),
    "Point_9":  ("MDICOR.GM",                "gas",         "GM building",              "GM",          "counter", "m³ (as documented)"),
    "Point_10": ("MDE.EFDP.IC",              "water",       "Municipal water supply",   "",            "counter", "m³"),
}
# campus totals without double counting: EV is inside EV+GM+MB+TD, GM is inside EV+GM+MB
CAMPUS_TOTAL = {"electricity": ["Point_12", "Point_6"], "gas": ["Point_8"], "water": ["Point_10"]}
WEATHER_POINT = "Point_3"     # MTX1-017.IC — outdoor temperature, EV-building weather station (°C)
RESOURCE = {
    "electricity": {"label": "Electricity", "unit": "kWh", "color": "#10B981", "icon": "⚡"},
    "gas":         {"label": "Natural gas", "unit": "m³",  "color": "#F97316", "icon": "🔥"},
    "water":       {"label": "Water",       "unit": "m³",  "color": "#3B82F6", "icon": "💧"},
    "waste":       {"label": "Waste",       "unit": "t",   "color": "#F59E0B", "icon": "🗑️"},
}

# ---- read Concordia's building-automation export (it is many reports stacked in one file) ----
rows, header = [], None
with open(PATHS["DATA_OUTPUT.csv"], errors="replace") as fh:
    for r in csv.reader(fh):
        if r and r[0] == "<>Date":
            header = ["date", "time"] + r[2:]
            continue
        if header and len(r) >= len(header) and "/" in r[0]:
            rows.append(r[:len(header)])
raw = pd.DataFrame(rows, columns=header)
stamp = pd.to_datetime(raw["date"] + " " + raw["time"].str.strip(), format="%m/%d/%Y %H:%M:%S")
raw = raw.drop(columns=["date", "time"]).apply(lambda s: pd.to_numeric(s.str.strip(), errors="coerce"))  # "No Data" → missing
raw.index = stamp.dt.floor("h").values
raw = raw[~raw.index.duplicated(keep="last")].sort_index()           # overlapping reports: keep the latest
HOURLY = raw.reindex(pd.date_range(raw.index.min(), raw.index.max(), freq="h"))
DAYS = pd.date_range(HOURLY.index.min().normalize(), HOURLY.index.max().normalize(), freq="D")

# optional: Concordia's official point list (adds building / description if the file is available)
if PATHS.get("POINT_LIST.csv"):
    try:
        _pl = pd.read_csv(PATHS["POINT_LIST.csv"])
        print("Official point list loaded:", list(_pl.columns))
    except Exception as e:
        print("POINT_LIST.csv could not be read:", e)

METERS = pd.DataFrame([{"point": p, "system name": v[0], "resource": v[1], "meter": v[2], "buildings": v[3] or "not stated",
                        "reading": "average power (kW)" if v[4] == "power" else "running total (resets each month/day)",
                        "first reading": HOURLY[p].first_valid_index(), "last reading": HOURLY[p].last_valid_index(),
                        "hours with data": f"{HOURLY[p].notna().mean():.0%}"} for p, v in METER_INFO.items()])
print(f"Hourly readings: {HOURLY.index.min():%d %b %Y} → {HOURLY.index.max():%d %b %Y}  ({len(HOURLY):,} hours)")
print(METERS.drop(columns=["point"]).to_string(index=False))


## Cell 4 · Clean the data (3 simple rules)
1. **Incomplete day** — fewer than 22 of 24 hours recorded → not used.
2. **Impossible jump** — one hour can't use more than the meter ever used in a whole day → that hour is removed.
3. **Not measuring** — the counter rose by the same amount every hour all day → it was estimating, not measuring.

🎤 *"We found real faults — the GM gas meter jumped 306,266 units in one hour; the water meter flat-lined for 5 weeks. We show every excluded day and why. We never fill gaps."*

In [ ]:
# ---- Turn hourly readings into one number per meter per day, with three plain quality rules ----
COMPLETE = 0.90      # a day counts only if ≥90% of its hours have a reading (22 of 24)
FAULTS = []          # every excluded day, with the reason — shown in the app


def daily_from_power(p):
    s = HOURLY[p]
    n = s.resample("D").count()
    return (s.resample("D").mean() * 24).where(n >= COMPLETE * 24).reindex(DAYS)   # average kW × 24 h = kWh


def hourly_use_from_counter(p):
    s = HOURLY[p]
    use = s.diff()
    reset = use < 0
    use[reset] = s[reset]                       # counter restarted at zero → the reading is the use since restart
    use = use.where(s.notna() & s.shift(1).notna())
    return use, reset


def daily_from_counter(p):
    use, reset = hourly_use_from_counter(p)
    # Rule A — impossible jump: one hour cannot use more than the most this meter ever used in a whole day
    while True:
        tot = use.resample("D").sum(min_count=1).dropna().sort_values(ascending=False)
        limit = np.where(use.index.normalize() == tot.index[0], tot.iloc[1], tot.iloc[0])
        bad = use > limit
        if not bad.any():
            break
        for t in use[bad].index:
            FAULTS.append({"point": p, "day": t.normalize(), "rule": "impossible jump",
                           "detail": f"{use[t]:,.0f} units in one hour at {t:%H:%M} — more than any whole day"})
        use[bad] = np.nan
    # Rule B — not measuring: every hourly step identical (±1 unit) all day = a fixed estimate, not a reading
    u = use[~reset]
    g = u.groupby(u.index.normalize())
    stuck = ((g.max() - g.min()) <= 1) & (g.count() >= COMPLETE * 24 - 1)
    for d in stuck[stuck].index:
        FAULTS.append({"point": p, "day": d, "rule": "not measuring",
                       "detail": f"same amount every hour ({g.median()[d]:,.0f}/h) — meter was not measuring"})
    n = use.resample("D").count()
    day = (use.resample("D").mean() * 24).where(n >= COMPLETE * 24).reindex(DAYS)
    day[day.index.isin(stuck[stuck].index)] = np.nan
    return day


DAILY = pd.DataFrame({p: (daily_from_power(p) if v[4] == "power" else daily_from_counter(p))
                      for p, v in METER_INFO.items()})
for p in DAILY:
    miss = DAILY[p].isna() & HOURLY[p].resample("D").count().reindex(DAYS).gt(0)
    for d in DAILY.index[miss]:
        if not any(f["point"] == p and f["day"] == d for f in FAULTS):
            FAULTS.append({"point": p, "day": d, "rule": "incomplete day", "detail": "fewer than 22 of 24 hours recorded"})
FAULTS = pd.DataFrame(FAULTS, columns=["point", "day", "rule", "detail"]).sort_values(["point", "day"])
FAULTS["meter"] = FAULTS.point.map(lambda p: f"{RESOURCE[METER_INFO[p][1]]['label']} · {METER_INFO[p][2]}")


def fault_periods(F):
    """Consecutive days with the same meter and rule → one line (easier to read)."""
    out = []
    for (m, rule), g in F.sort_values("day").groupby(["meter", "rule"]):
        days = list(g.day); det = list(g.detail); i = 0
        while i < len(days):
            j = i
            while j + 1 < len(days) and days[j + 1] - days[j] == pd.Timedelta(days=1):
                j += 1
            out.append({"meter": m, "from": days[i].date(), "to": days[j].date(), "days": j - i + 1, "rule": rule,
                        "detail": det[i] if i == j else det[i].split(" — ")[-1]})
            i = j + 1
    return pd.DataFrame(out).sort_values("from")

# daily outdoor temperature (EV weather station) + Quebec public holidays (official calendar via `holidays`)
TEMP = HOURLY[WEATHER_POINT].resample("D").mean().where(HOURLY[WEATHER_POINT].resample("D").count() >= COMPLETE * 24).reindex(DAYS)
QC_HOLIDAYS = holidays.Canada(subdiv="QC", years=sorted(set(DAYS.year)))
OFF_DAY = pd.Series([(d.dayofweek >= 5) or (d.date() in QC_HOLIDAYS) for d in DAYS], index=DAYS)

print("Good days per meter (out of", len(DAYS), "):")
for p in DAILY:
    print(f"  {RESOURCE[METER_INFO[p][1]]['icon']} {METER_INFO[p][2]:30s} {DAILY[p].notna().sum():4d} good days")
print("\nExcluded days by reason:")
print(FAULTS.groupby(["meter", "rule"]).size().rename("days").to_string())
print("\nThe problems (not counting incomplete days):")
print(fault_periods(FAULTS[FAULTS.rule != "incomplete day"]).to_string(index=False))


## Cell 5 · Learn "normal" — the ghost twin's brain
Two models are trained on one full year (every season once), using the day's **temperature**, **weekday/holiday** and **season**:
* the **ASHRAE change-point model** (explainable: "+1,233 kWh per °C colder"), and
* **gradient boosting (LightGBM)** — machine learning.

Each baseline month is hidden in turn and predicted by both; **whichever misses less becomes that meter's ghost twin.**
The change-point model is always kept to explain the building and to grade accuracy (ASHRAE Guideline 14).

🎤 *"We don't pick AI or physics by taste — each meter gets the model that predicts months it has never seen better."*

In [ ]:
# ---- "Expected use": how much each meter normally uses for a given temperature and day type ----
# ASHRAE change-point model (ASHRAE Guideline 14 / Inverse Modeling Toolkit, RP-1050):
#   daily use = base + (off-day change) + heating slope × max(0, Th − temp) + cooling slope × max(0, temp − Tc)
# The balance temperatures Th, Tc are found from the data (every whole °C observed is tried); the simplest
# model that fits is chosen by BIC. Slopes are kept ≥ 0 (colder can't reduce heating; hotter can't reduce cooling).
_tm = TEMP.notna().groupby(TEMP.index.to_period("M")).mean()
BASE_START = _tm[_tm >= COMPLETE].index[0].to_timestamp()               # first month with weather data
BASE_END = BASE_START + pd.DateOffset(years=1) - pd.Timedelta(days=1)    # one full year = every season once
MON_START = BASE_END + pd.Timedelta(days=1)
print(f"Baseline year: {BASE_START:%d %b %Y} → {BASE_END:%d %b %Y}   ·   Monitoring: {MON_START:%d %b %Y} → {DAYS[-1]:%d %b %Y}")


def _X(idx, th, tc):
    t = TEMP.reindex(idx).values
    cols = [np.ones(len(idx)), OFF_DAY.reindex(idx).values.astype(float)]
    if th is not None: cols.append(np.maximum(0, th - t))
    if tc is not None: cols.append(np.maximum(0, t - tc))
    return np.column_stack(cols)


def _ml_features(idx):
    idx = pd.DatetimeIndex(idx)
    return pd.DataFrame({"temp": TEMP.reindex(idx).values, "off": OFF_DAY.reindex(idx).values.astype(int),
                         "dow": idx.dayofweek, "doy_sin": np.sin(2 * np.pi * idx.dayofyear / 365.25),
                         "doy_cos": np.cos(2 * np.pi * idx.dayofyear / 365.25)}, index=idx)


def fit_model(y, kind="cp"):
    """kind='cp': ASHRAE change-point model · kind='ml': gradient boosting (LightGBM, library defaults)."""
    if kind == "ml":
        y = y.dropna(); y = y[TEMP.reindex(y.index).notna()]
        mdl = lgb.LGBMRegressor(random_state=0, verbosity=-1).fit(_ml_features(y.index), y.values)
        return {"kind": "ml", "mdl": mdl, "idx": y.index, "n": len(y)}
    y = y.dropna()
    y = y[TEMP.reindex(y.index).notna()]
    t = TEMP.reindex(y.index).values
    o = OFF_DAY.reindex(y.index).values.astype(float)
    v = y.values
    n = len(v)
    grid = [None] + sorted(set(np.round(t).astype(int)))
    best = None
    for th in grid:
        h = None if th is None else np.maximum(0, th - t)
        for tc in grid:
            if th is not None and tc is not None and tc < th:
                continue
            cols = [np.ones(n), o] + ([h] if h is not None else []) + ([np.maximum(0, t - tc)] if tc is not None else [])
            X = np.column_stack(cols)
            b = np.linalg.lstsq(X, v, rcond=None)[0]
            if len(b) > 2 and b[2:].min() < 0:
                continue
            sse = float(((X @ b - v) ** 2).sum())
            k = X.shape[1] + (th is not None) + (tc is not None)
            bic = n * np.log(sse / n) + k * np.log(n)
            if best is None or bic < best["bic"]:
                best = {"kind": "cp", "bic": bic, "th": th, "tc": tc, "b": b, "k": k, "n": n, "idx": y.index}
    return best


def expected(m, idx):
    idx = pd.DatetimeIndex(idx)
    if m["kind"] == "ml":
        return pd.Series(m["mdl"].predict(_ml_features(idx)), index=idx).clip(lower=0)
    return pd.Series(_X(idx, m["th"], m["tc"]) @ m["b"], index=idx).clip(lower=0)


def accuracy(y, m):
    yb = y.reindex(m["idx"])
    e = yb - expected(m, yb.index)
    cv = float(np.sqrt((e ** 2).sum() / (len(e) - m["k"])) / yb.mean() * 100)
    nmbe = float(e.sum() / (len(e) - m["k"]) / yb.mean() * 100)
    rho = float(e.autocorr(1))
    return cv, nmbe, rho


def describe_model(p, m):
    res, unit = METER_INFO[p][1], RESOURCE[METER_INFO[p][1]]["unit"]
    b = list(m["b"]); base, offd = b[0], b[1]; i = 2
    s = [f"On a normal weekday in mild weather it uses about {base:,.0f} {unit}/day; on weekends and holidays "
         f"{abs(offd):,.0f} {'less' if offd < 0 else 'more'}."]
    if m["th"] is not None:
        s.append(f"For every 1 °C below {m['th']} °C it uses {b[i]:,.0f} {unit} more (heating)."); i += 1
    if m["tc"] is not None:
        s.append(f"For every 1 °C above {m['tc']} °C it uses {b[i]:,.0f} {unit} more (cooling).")
    return " ".join(s)


# ---- Which model predicts better? Test both on months they never saw (leave-one-month-out) ----
# For every meter, each baseline month is hidden in turn and predicted by (a) the change-point model and
# (b) gradient boosting (LightGBM, library defaults, same inputs + season). The model with the smaller
# typical miss on hidden months is used for predictions (alerts, verification). The change-point model is
# always kept too, because its slopes EXPLAIN the building (used for the accuracy grade and recommendations).
BASE_MONTHS = list(pd.period_range(BASE_START, BASE_END, freq="M"))
MODEL, CP_MODEL, MODEL_KIND, EXPECTED, OOS, MISS, ACC, MODEL_CHECK = {}, {}, {}, {}, {}, {}, [], []
for p in DAILY:
    yb = DAILY[p].loc[BASE_START:BASE_END].where(TEMP.notna())
    oos = {"cp": [], "ml": []}; miss = {"cp": {}, "ml": {}}
    for mo in BASE_MONTHS:
        a, b = mo.start_time, mo.end_time.normalize()
        hold = yb.loc[a:b].dropna()
        train = yb.copy(); train.loc[a:b] = np.nan
        days = DAYS[(DAYS >= a) & (DAYS <= b) & TEMP.notna().values]
        for kind in ("cp", "ml"):
            m_ = fit_model(train, kind)
            oos[kind].append(expected(m_, days))
            if len(hold) >= COMPLETE * len(pd.date_range(a, b)):
                e = expected(m_, hold.index).sum()
                miss[kind][a] = float((e - hold.sum()) / e * 100)
    rms = {k: float(np.sqrt(np.mean(np.square(list(miss[k].values()))))) for k in miss}
    MODEL_KIND[p] = "ml" if rms["ml"] < rms["cp"] else "cp"
    OOS[p] = pd.concat(oos[MODEL_KIND[p]])
    MISS[p] = miss[MODEL_KIND[p]]
    CP_MODEL[p] = fit_model(yb, "cp")
    MODEL[p] = CP_MODEL[p] if MODEL_KIND[p] == "cp" else fit_model(yb, "ml")
    EXPECTED[p] = expected(MODEL[p], DAYS[TEMP.notna().values])
    MODEL_CHECK.append({"point": p, "Meter": f"{RESOURCE[METER_INFO[p][1]]['icon']} {METER_INFO[p][2]}",
                        "Hidden months": len(miss["cp"]), "Change-point typical miss %": round(rms["cp"], 1),
                        "Gradient boosting typical miss %": round(rms["ml"], 1),
                        "Used for predictions": "Gradient boosting (ML)" if MODEL_KIND[p] == "ml" else "Change-point"})
    m = CP_MODEL[p]
    cv, nmbe, rho = accuracy(DAILY[p], m)
    grade = "Good" if cv <= 15 and abs(nmbe) <= 5 else ("OK" if cv <= 30 and abs(nmbe) <= 10 else "Not reliable")
    ACC.append({"point": p, "Meter": f"{RESOURCE[METER_INFO[p][1]]['icon']} {RESOURCE[METER_INFO[p][1]]['label']} · {METER_INFO[p][2]}",
                "Baseline days": m["n"], "Typical daily error %": round(cv, 1), "Bias %": round(nmbe, 2),
                "Accuracy": grade, "Predictions by": MODEL_CHECK[-1]["Used for predictions"],
                "What the model learned": describe_model(p, m)})
ACC = pd.DataFrame(ACC)
MODEL_CHECK = pd.DataFrame(MODEL_CHECK)
RELIABLE = set(ACC[ACC.Accuracy != "Not reliable"].point)
print("Accuracy grade uses ASHRAE Guideline 14 limits on the change-point fit: Good = CV ≤ 15% & bias ≤ ±5% "
      "(monthly-data limit); OK = CV ≤ 30% & bias ≤ ±10% (hourly-data limit); otherwise Not reliable (no savings claimed).\n")
for _, r in ACC.iterrows():
    print(f"{r.Meter}\n   accuracy: {r.Accuracy} (typical daily error {r['Typical daily error %']}%) · predictions by: "
          f"{r['Predictions by']}\n   {r['What the model learned']}")
print("\nWhich model predicts hidden months better (typical monthly miss, lower is better):")
print(MODEL_CHECK.drop(columns="point").to_string(index=False))


## Cell 6 · Alerts and discoveries
After the baseline year, a meter is flagged when it uses **more** than its ghost twin by more than it ever did in the whole baseline year (each meter sets its own bar). Sustained **drops** are *discoveries* — good news, not counted until someone explains them.

In [ ]:
# ---- Unusual-change alerts ----
# Each meter sets its own bar from last year: we re-predict every baseline month with a model that did NOT
# see that month (leave-one-month-out), so the baseline errors are honest "surprises".
# ALERT     = after the baseline, a day (or 7-day stretch) used MORE than expected by more than the
#             biggest surprise of the whole baseline year.
# DISCOVERY = a 7-day stretch used LESS than expected by more than any 7 days of the baseline year
#             (good news nobody has explained yet — not counted as a saving until an action is linked).
# OOS (each baseline month predicted by a model that never saw it) comes from cell 5.
SCALES = {"1 day": 1, "7 days": 7}
RECORD, ALERTS, DISCOVERIES = {}, [], []
_expected_false = 0.0
for p in DAILY:
    y = DAILY[p]
    res_label, unit = RESOURCE[METER_INFO[p][1]]["label"], RESOURCE[METER_INFO[p][1]]["unit"]
    base_gap = (y - OOS[p]).loc[BASE_START:BASE_END]
    mon_gap = (y - EXPECTED[p]).loc[MON_START:]
    for name, w in SCALES.items():
        bg = base_gap.rolling(w, min_periods=w).sum().dropna()
        mg = mon_gap.rolling(w, min_periods=w).sum().dropna()
        hi, lo = float(bg.max()), float(bg.min())
        RECORD[(p, name)] = (hi, lo, len(bg))
        _expected_false += len(mg) / (len(bg) + 1)
        hits = mg[mg > hi]
        for d, v in hits.items():
            ALERTS.append({"point": p, "scale": name, "end": d, "start": d - pd.Timedelta(days=w - 1), "extra": v, "record": hi})
        if name == "7 days":
            for d, v in mg[mg < lo].items():
                DISCOVERIES.append({"point": p, "end": d, "start": d - pd.Timedelta(days=6), "less": -v, "record": -lo})


def _merge(df):
    """Join overlapping flagged windows into one event per meter."""
    out = []
    for p, g in df.sort_values("start").groupby("point", sort=False):
        cur = None
        for _, r in g.iterrows():
            if cur is not None and r.start <= cur["end"] + pd.Timedelta(days=1):
                cur["end"] = max(cur["end"], r.end); cur["rows"].append(r)
            else:
                if cur: out.append(cur)
                cur = {"point": p, "start": r.start, "end": r.end, "rows": [r]}
        if cur: out.append(cur)
    return out


def _qty(v, p):
    return f"{v:,.0f} {RESOURCE[METER_INFO[p][1]]['unit']}"


ALERT_TABLE = []
for ev in _merge(pd.DataFrame(ALERTS, columns=["point", "scale", "end", "start", "extra", "record"])):
    p = ev["point"]; worst = max(ev["rows"], key=lambda r: r.extra - r.record)
    y, e = DAILY[p].loc[ev["start"]:ev["end"]], EXPECTED[p].loc[ev["start"]:ev["end"]]
    extra = float((y - e).clip(lower=0).sum())
    ALERT_TABLE.append({
        "id": "", "meter": f"{RESOURCE[METER_INFO[p][1]]['icon']} {METER_INFO[p][2]}", "point": p,
        "from": ev["start"].date(), "to": ev["end"].date(), "extra use": _qty(extra, p),
        "extra %": round(extra / max(float(e.sum()), 1e-9) * 100, 1),
        "what happened": (f"{RESOURCE[METER_INFO[p][1]]['label']} at {METER_INFO[p][2]} used {_qty(worst.extra, p)} more than "
                          f"expected over {worst.scale} (ending {worst.end:%a %d %b}). The most it ever went over in the whole "
                          f"baseline year was {_qty(worst.record, p)}. This is new — worth a check."),
    })
ALERT_TABLE = pd.DataFrame(ALERT_TABLE, columns=["id", "meter", "point", "from", "to", "extra use", "extra %", "what happened"])
ALERT_TABLE["id"] = [f"AL-{i + 1:02d}" for i in range(len(ALERT_TABLE))]

DISCOVERY_TABLE = []
for ev in _merge(pd.DataFrame(DISCOVERIES, columns=["point", "end", "start", "less", "record"])):
    p = ev["point"]
    y, e = DAILY[p].loc[ev["start"]:ev["end"]], EXPECTED[p].loc[ev["start"]:ev["end"]]
    less = float((e - y).sum())
    DISCOVERY_TABLE.append({"meter": f"{RESOURCE[METER_INFO[p][1]]['icon']} {METER_INFO[p][2]}", "point": p,
                            "first seen": ev["rows"][0].end.date(), "from": ev["start"].date(), "to": ev["end"].date(),
                            "less than expected": _qty(less, p), "%": round(less / max(float(e.sum()), 1e-9) * 100, 1)})
DISCOVERY_TABLE = pd.DataFrame(DISCOVERY_TABLE, columns=["meter", "point", "first seen", "from", "to", "less than expected", "%"])
EXPECTED_FALSE_ALERTS = _expected_false

print(f"Alerts after the baseline: {len(ALERT_TABLE)}  (if nothing had changed, ≈{EXPECTED_FALSE_ALERTS:.1f} "
      f"flagged windows would be expected by chance)")
print(ALERT_TABLE[["id", "meter", "from", "to", "extra use", "extra %"]].to_string(index=False))
print(f"\nDiscoveries (sustained drops nobody has explained): {len(DISCOVERY_TABLE)}")
print(DISCOVERY_TABLE.drop(columns="point").to_string(index=False))


## Cell 7 · Waste
666 real hauling invoices — tonnes and cost per stream and building — and the **diversion rate** (compost + recycling ÷ total sorted). Bulky waste is shown but left out of the rate (Concordia: split unknown). Pickups with no weight are counted, not guessed.

In [ ]:
# ---- Waste: Concordia's real hauling invoices (Zero Waste program) ----
# Mass is in metric tonnes and Qty is a count of bins (Concordia API documentation, waste/invoices).
# Bulky waste is a mix of recycling and landfill in unknown proportions (Concordia's own definition),
# so it is reported separately and left out of the diversion rate. Pickups with no recorded weight are
# counted and shown — never filled in.
WASTE_TYPES = pd.read_csv(PATHS["WASTE_TYPE.csv"])
BUILDINGS = pd.read_csv(PATHS["BUILDING_LIST.csv"])
_w = pd.read_csv(PATHS["WASTE_INVOICES.csv"])
_w = _w.loc[:, [c for c in _w.columns if not c.startswith("Unnamed")]]
_w["Date"] = pd.to_datetime(_w["Date"], errors="coerce")
_w["tonnes"] = pd.to_numeric(_w["Mass"], errors="coerce")
_w["cost"] = pd.to_numeric(_w["Total_Cost"].astype(str).str.replace(r"[$,\s]", "", regex=True), errors="coerce")
WASTE = _w.dropna(subset=["Date"]).copy()
WASTE["month"] = WASTE["Date"].dt.to_period("M").dt.to_timestamp()
DIVERTED = ["Organics", "Paper", "Mixed_recycling"]                # sent to composting / recycling
STREAM_COLOR = {"Landfill": "#94A3B8", "Organics": "#22C55E", "Paper": "#38BDF8", "Mixed_recycling": "#A78BFA",
                "Bulky_waste": "#F59E0B"}
STREAM_LABEL = {"Landfill": "Landfill", "Organics": "Compost", "Paper": "Paper & cardboard",
                "Mixed_recycling": "Mixed recycling", "Bulky_waste": "Bulky (mixed, unknown split)"}

WASTE_MONTHLY = WASTE.pivot_table(index="month", columns="Waste_Type", values="tonnes", aggfunc="sum")
_weighed = WASTE[WASTE.tonnes.notna()].pivot_table(index="month", columns="Waste_Type", values="tonnes", aggfunc="count")
_full = _weighed.reindex(columns=["Landfill"] + DIVERTED).notna().all(axis=1)     # month has every stream weighed
_div = WASTE_MONTHLY.reindex(columns=DIVERTED).sum(axis=1)
_land = WASTE_MONTHLY.reindex(columns=["Landfill"]).sum(axis=1)
DIVERSION = (_div / (_div + _land) * 100).where(_full)
WASTE_BY_BUILDING = (WASTE.pivot_table(index="Building", columns="Waste_Type", values="tonnes", aggfunc="sum")
                     .fillna(0).round(1))
WASTE_BY_BUILDING["Total t"] = WASTE_BY_BUILDING.sum(axis=1)
WASTE_BY_BUILDING = WASTE_BY_BUILDING.sort_values("Total t", ascending=False)
WASTE_COST_PER_T = (WASTE[WASTE.tonnes > 0].groupby("Waste_Type").apply(lambda g: g.cost.sum() / g.tonnes.sum())).round(0)
NO_WEIGHT = int(WASTE.tonnes.isna().sum())

# before vs during the COVID closure, same 12-month baseline year as the meters
_b = WASTE[(WASTE.Date >= BASE_START) & (WASTE.Date <= BASE_END)]
WASTE_BASE = _b.groupby("Waste_Type").tonnes.sum().round(1)
_d = DIVERSION.loc[BASE_START:BASE_END].dropna()

print(f"Invoices: {len(WASTE)} ({WASTE.Date.min():%b %Y} → {WASTE.Date.max():%b %Y}), "
      f"{WASTE.Building.nunique()} buildings, {NO_WEIGHT} pickups with no recorded weight (left out, not guessed)")
print("\nTonnes in the baseline year:", WASTE_BASE.to_dict())
print(f"Diversion rate (compost + recycling ÷ those + landfill), months with every stream weighed: "
      f"baseline-year median {_d.median():.0f}%  ({len(_d)} months)")
print("\nCost per tonne ($ as invoiced):", WASTE_COST_PER_T.to_dict())
print("\nTonnes by building (all years):")
print(WASTE_BY_BUILDING.to_string())

# 12-month periods aligned with the meter baseline year (Mar → Feb)
WASTE["period"] = WASTE.Date.apply(lambda d: f"{d.year if d.month >= BASE_START.month else d.year - 1}–"
                                             f"{(d.year if d.month >= BASE_START.month else d.year - 1) + 1}")
_y = WASTE.pivot_table(index="period", columns="Waste_Type", values="tonnes", aggfunc="sum").fillna(0)
_y = _y[[c for c in ["Landfill"] + DIVERTED + ["Bulky_waste"] if c in _y]]
_cnt = WASTE.groupby("period").Date.agg(["min", "max"])
WASTE_YEARLY = _y.round(1)
_wc = WASTE[WASTE.tonnes > 0].pivot_table(index="period", columns="Waste_Type", values="tonnes", aggfunc="count")
_allw = _wc.reindex(index=_y.index, columns=["Landfill"] + DIVERTED).notna().all(axis=1)
for c in _y.columns:                                  # a stream never weighed in a period shows "—", not 0
    WASTE_YEARLY[c] = WASTE_YEARLY[c].where(_wc.reindex(index=_y.index).get(c, pd.Series(index=_y.index)).notna())
WASTE_YEARLY["Diversion %"] = (_y[DIVERTED].sum(axis=1) / (_y[DIVERTED].sum(axis=1) + _y["Landfill"]) * 100).round(0).where(_allw)
WASTE_YEARLY["Invoices from"] = _cnt["min"].dt.strftime("%b %Y")
WASTE_YEARLY["to"] = _cnt["max"].dt.strftime("%b %Y")
print("\nTonnes per 12-month period (Mar→Feb; partial periods marked by their invoice dates):")
print(WASTE_YEARLY.to_string())
print("Diversion % is shown only for periods where landfill, compost, paper and mixed recycling were all weighed.")
print("Note: 'Building' is where the bin or compactor sits — one compactor can serve several buildings.")


## Cell 7e · E-waste
Reads **your real asset register** (template provided) and routes every device the way India's **E-Waste (Management) Rules, 2022** require: keep in use, a *registered* refurbisher, or a *registered* recycler. Expected life comes from the CPCB EPR portal for each EEE code. Concordia's open data has no e-waste records, so nothing is shown until you upload a register — nothing is invented.

In [ ]:
# ---- E-waste: your real asset register → what to do with each device ----
# Nothing is invented. You upload a CSV of real devices (template written below). Rules follow India's
# E-Waste (Management) Rules, 2022: a bulk consumer (a university) may hand e-waste ONLY to a registered
# producer, refurbisher or recycler, and must keep records. "Expected life" is the average life the
# CPCB EPR portal lists for the device's EEE code (https://eprewastecpcb.in) — copy it into the CSV.
EWASTE_COLS = ["device_id", "device_type", "eee_code", "department", "location", "purchase_date",
               "expected_life_years", "condition", "weight_kg", "embodied_kgco2e"]
EWASTE_TEMPLATE = os.path.join(OUT_DIR, "ewaste_register_TEMPLATE.csv")
pd.DataFrame(columns=EWASTE_COLS).to_csv(EWASTE_TEMPLATE, index=False)
EWASTE_HELP = {
    "condition": "working / faulty / non-functional",
    "expected_life_years": "average life for the EEE code, from the CPCB EPR portal",
    "weight_kg": "optional — from the spec sheet or a scale",
    "embodied_kgco2e": "optional — manufacturer's product carbon footprint (PCF) sheet",
}
EWASTE_EOL_T_CO2E_PER_T = 0.02 / 0.907185    # EPA GHG Emission Factors Hub (2024), mixed electronics, recycled: 0.02 t CO2e / short ton
PATHS["ewaste_register.csv"] = find("ewaste_register.csv")


def ewaste_plan(df, today=None):
    """Flag devices and pick a route that is legal under the E-Waste (Management) Rules, 2022."""
    today = pd.Timestamp(today or datetime.date.today())
    d = df.copy()
    d["purchase_date"] = pd.to_datetime(d["purchase_date"], errors="coerce")
    for c in ["expected_life_years", "weight_kg", "embodied_kgco2e"]:
        d[c] = pd.to_numeric(d.get(c), errors="coerce")
    d["condition"] = d["condition"].astype(str).str.strip().str.lower()
    d["age_years"] = ((today - d["purchase_date"]).dt.days / 365.25).round(1)
    d["life_used_%"] = (d["age_years"] / d["expected_life_years"] * 100).round(0)

    def route(r):
        past = r["age_years"] >= r["expected_life_years"] if r["expected_life_years"] == r["expected_life_years"] else None
        if r["condition"] == "working" and past is False:
            return "Keep in use / redeploy", "working and within its expected life"
        if r["condition"] == "working":
            return "Registered refurbisher", "working but past its expected life — reuse before recycling"
        if r["condition"] == "faulty":
            return "Registered refurbisher (repair check)", "faulty — repair is assessed before recycling"
        if r["condition"] == "non-functional":
            return "Registered recycler", "non-functional — recover materials through an authorised recycler"
        return "Check record", "condition not recorded"
    rt = d.apply(route, axis=1, result_type="expand")
    d["route"], d["why"] = rt[0], rt[1]
    d["leaving campus"] = ~d["route"].isin(["Keep in use / redeploy", "Check record"])
    d["eol_kgco2e"] = (d["weight_kg"] / 1000 * EWASTE_EOL_T_CO2E_PER_T * 1000).where(d["leaving campus"])
    return d


EWASTE = None
if PATHS["ewaste_register.csv"]:
    EWASTE = ewaste_plan(pd.read_csv(PATHS["ewaste_register.csv"]))
    print(f"E-waste register: {len(EWASTE)} devices · {int(EWASTE['leaving campus'].sum())} to hand over to registered "
          f"refurbishers/recyclers · {EWASTE.loc[EWASTE['leaving campus'], 'weight_kg'].sum():,.0f} kg recorded")
    print(EWASTE.groupby("route").size().to_string())
else:
    print("No e-waste register uploaded yet. Concordia's open data has no e-waste records "
          f"(its invoices contain {int((WASTE.Waste_Type.str.lower() == 'e-waste').sum())} e-waste pickups).")
    print(f"→ Fill in the template with your department's real devices and upload it as ewaste_register.csv:\n   {EWASTE_TEMPLATE}")
    print("   Columns:", ", ".join(EWASTE_COLS))


## Cell 8 · Action register + "Did it work?" — contextual ghost twin + placebo-in-time
* **Ghost twin:** the building *without* the action, trained on the 12 months before and fed the **real** weather and calendar of every later day.
* **Placebo-in-time:** we pretend the action happened on every week of the year before (when nothing was done). Those fake "savings" show what noise looks like.
* **VERIFIED** only if the real change is outside the noise: 95% range excludes zero **and** p ≤ 5%. We count only the **low end**.

The register starts with one **ground-truth test** — the documented COVID-19 closure (13 March 2020) — to prove the engine on a real event. Approved recommendations and your own actions are added in the app.

🎤 *"A before/after dashboard says gas fell 40%. The ghost twin, fed the real spring weather, says that is normal — not proven, nothing counted."*

In [ ]:
# ---- Action register + "Did it work?" — contextual GHOST TWIN + PLACEBO-IN-TIME test ----
#   1. GHOST TWIN: a copy of the meter that never got the action. It is trained on the 12 months BEFORE the
#      action and then fed the REAL context of every later day (actual weather, weekday/holiday, season), so it
#      shows what the building would have used without the action. (IPMVP Option C / ASHRAE Guideline 14.)
#   2. saving = ghost twin − actual
#   3. PLACEBO-IN-TIME: we pretend the action happened on every week of the year before (when nothing was
#      done), hide that stretch, rebuild the ghost twin without it, and record the fake "saving". Those fakes
#      show what pure noise looks like for this meter.
#        • 95% range = ± t × (typical fake saving) × √(1 + 1/k), k = independent stretches in a year
#        • p-value  = share of fake savings at least as large as the real one (permutation test)
#   4. VERIFIED only if the whole 95% range excludes zero AND p ≤ 5%. We count the LOW end of the range.
ACTIONS_CSV = os.path.join(OUT_DIR, "actions.csv")
ACTION_COLS = ["action_id", "date", "point", "title", "type", "claimed_%", "source"]
if os.path.exists(ACTIONS_CSV):
    ACTIONS = pd.read_csv(ACTIONS_CSV, parse_dates=["date"])
else:
    # one real, documented event to start with (not a sustainability action — clearly labelled)
    ACTIONS = pd.DataFrame([{
        "action_id": "EV-01", "date": pd.Timestamp("2020-03-14"), "point": p,
        "title": "Ground-truth test — COVID-19: both campuses closed from midnight 13 March 2020",
        "type": "Ground-truth test (real, documented event) — proves the engine; not a sustainability saving",
        "claimed_%": np.nan,
        "source": "The Link, 13 Mar 2020: thelinknewspaper.ca/article/concordia-closed-until-march-29-due-to-covid-19-prevention"}
        for p in DAILY], columns=ACTION_COLS)
    ACTIONS["action_id"] = [f"EV-01{chr(97 + i)}" for i in range(len(ACTIONS))]
    ACTIONS.to_csv(ACTIONS_CSV, index=False)


def add_action(date, point, title, claimed_pct=None, source="logged in GreenLedger"):
    global ACTIONS
    aid = f"A-{len(ACTIONS) + 1:03d}"
    ACTIONS = pd.concat([ACTIONS, pd.DataFrame([{"action_id": aid, "date": pd.Timestamp(date), "point": point, "title": title,
                                                  "type": "Sustainability action", "claimed_%": claimed_pct,
                                                  "source": source}])], ignore_index=True)
    ACTIONS.to_csv(ACTIONS_CSV, index=False)
    return aid


WINDOWS = [28, 42, 56]          # 4, 6 or 8 whole weeks after the action (same weekday mix every time)
ELEC_KG_CO2_PER_KWH = 0.0005    # Hydro-Québec, 0.5 kg CO2/MWh for 2019 and 2020 ("CO2 emissions and Hydro-Québec electricity 1990-2020")
GAS_KG_CO2_PER_M3 = 1.926       # Québec MELCC GHG quantification guide, Table 5 (natural gas, CO2)
GAS_M3_PER_METER_UNIT = None    # Concordia's API documents m³. Sanity check (cell 10): at 1 unit = 1 m³ the EV+GM+MB meter
                                # alone would emit ~57,000 t CO₂/yr — far above everything else measured — so the unit is
                                # treated as unconfirmed. Set to 1.0 once Facilities confirms m³ (the app has a switch).


def co2_kg(point, amount):
    res = METER_INFO[point][1]
    if res == "electricity":
        return amount * ELEC_KG_CO2_PER_KWH
    if res == "gas" and GAS_M3_PER_METER_UNIT:
        return amount * GAS_M3_PER_METER_UNIT * GAS_KG_CO2_PER_M3
    return None


ALPHA = 0.05                       # 95% confidence — statistical convention
PLACEBO_STEP = 7                   # fake action dates one week apart: every date has the same weekday mix


def placebo_in_time(y_base, window, kind="cp"):
    """Fake action on every week of the baseline year → the 'saving' the ghost twin finds when nothing was done."""
    start, end = y_base.index.min(), y_base.index.max()
    fakes, d = [], start
    while d + pd.Timedelta(days=window - 1) <= end:
        b = d + pd.Timedelta(days=window - 1)
        hold = y_base.loc[d:b].dropna()
        if len(hold) >= COMPLETE * window:
            train = y_base.copy(); train.loc[d:b] = np.nan
            e = expected(fit_model(train, kind), hold.index)
            fakes.append(float((e.sum() - hold.sum()) / e.sum() * 100))
        d += pd.Timedelta(days=PLACEBO_STEP)
    return np.array(fakes)


def range_from_misses(misses, k=None):
    """Prediction interval from real misses. k = number of INDEPENDENT stretches (overlapping fakes share data)."""
    k = k or len(misses)
    typical = float(np.sqrt(np.mean(np.square(misses))))       # typical miss (root-mean-square, includes any bias)
    return float(stats.t.ppf(1 - ALPHA / 2, k - 1) * typical * np.sqrt(1 + 1 / k)), typical


def verify(point, date, window=42, other_events=None):
    T0 = pd.Timestamp(date).normalize()
    res, unit = METER_INFO[point][1], RESOURCE[METER_INFO[point][1]]["unit"]
    y = DAILY[point].where(TEMP.notna())
    base = y.loc[T0 - pd.Timedelta(days=365): T0 - pd.Timedelta(days=1)]
    post = y.loc[T0: T0 + pd.Timedelta(days=window - 1)]
    out = {"point": point, "date": T0, "window": window, "unit": unit}
    if post.notna().sum() < COMPLETE * window:
        return {**out, "verdict": "WAITING FOR DATA",
                "why": f"Needs {window} days of good readings after the action; {int(post.notna().sum())} available."}
    if base.notna().sum() < COMPLETE * 365:
        return {**out, "verdict": "NOT ENOUGH HISTORY",
                "why": f"Needs 12 months of good readings (with weather) before the action; "
                       f"{int(base.notna().sum())} days available."}
    kind = MODEL_KIND[point]
    cp = fit_model(base, "cp")
    cv, nmbe, rho = accuracy(base, cp)                       # ASHRAE Guideline 14 grade on the change-point fit
    ghost = cp if kind == "cp" else fit_model(base, kind)   # the meter's better predictor (cell 5)
    fakes = placebo_in_time(base, window, kind)
    k_ind = 365 // window
    if len(fakes) < k_ind:
        return {**out, "verdict": "NOT ENOUGH HISTORY", "why": "Too few complete weeks in the baseline year for the placebo test."}
    half, typical = range_from_misses(fakes, k_ind)
    yp = post.dropna()
    exp_ = expected(ghost, yp.index)
    saving = float((exp_.sum() - yp.sum()) / exp_.sum() * 100)
    lo, hi = saving - half, saving + half
    p = float((1 + np.sum(np.abs(fakes) >= abs(saving))) / (len(fakes) + 1))
    pre = y.loc[T0 - pd.Timedelta(days=window): T0 - pd.Timedelta(days=1)].mean()
    naive = float((pre - yp.mean()) / pre * 100)
    reliable = cv <= 30 and abs(nmbe) <= 10
    if not reliable:
        verdict = "MODEL NOT ACCURATE ENOUGH"
    elif lo > 0 and p <= ALPHA:
        verdict = "VERIFIED SAVING"
    elif hi < 0 and p <= ALPHA:
        verdict = "VERIFIED INCREASE"
    else:
        verdict = "NOT PROVEN"
    warn = []
    if other_events is not None:                         # IPMVP: a non-routine event in the baseline or the window
        clash = other_events[(other_events.point == point) & (other_events.date >= T0 - pd.Timedelta(days=365)) &
                             (other_events.date <= T0 + pd.Timedelta(days=window - 1))]
        if len(clash):
            warn.append("Another logged event on this meter falls inside the 12-month baseline or the measurement window "
                        f"({', '.join(clash.title.str[:50])}), so the change cannot be attributed to this action alone.")
            if verdict in ("VERIFIED SAVING", "VERIFIED INCREASE"):
                verdict = "CONFOUNDED"
    counted = max(0.0, lo / 100 * float(exp_.sum())) if verdict == "VERIFIED SAVING" else 0.0
    return {**out, "verdict": verdict, "saving_%": saving, "low_%": lo, "high_%": hi, "p": p, "naive_%": naive,
            "expected": float(exp_.sum()), "actual": float(yp.sum()), "counted": counted,
            "counted_co2_kg": co2_kg(point, counted) if counted else 0.0, "cv": cv, "rho": rho,
            "fakes": fakes, "typical_miss": typical, "kind": kind, "k_ind": k_ind,
            "days": len(yp), "model": ghost, "series_exp": exp_, "series_act": yp, "warnings": warn}


def explain(r):
    """One plain-English paragraph a judge can read."""
    name = f"{RESOURCE[METER_INFO[r['point']][1]]['label']} at {METER_INFO[r['point']][2]}"
    if "saving_%" not in r:
        return f"{r['verdict']}: {r['why']}"
    u = r["unit"]
    txt = (f"Over the {r['days']} days after {r['date']:%d %b %Y}, {name} used {r['actual']:,.0f} {u}. "
           f"Based on the 12 months before — adjusted for the actual weather and weekdays/holidays — it would have used "
           f"about {r['expected']:,.0f} {u} — that is its ghost twin ({'gradient-boosting' if r['kind'] == 'ml' else 'change-point'} model, "
           f"fed the real weather and calendar). That is a {abs(r['saving_%']):.1f}% {'drop' if r['saving_%'] > 0 else 'rise'}. "
           f"Placebo test: we pretended the action happened on {len(r['fakes'])} other dates last year; those fake "
           f"'savings' were typically {r['typical_miss']:.1f}%, so the 95% range is {r['low_%']:.1f}% to {r['high_%']:.1f}% "
           f"and only {r['p'] * 100:.0f}% of fakes were this large (p = {r['p']:.3f}). "
           f"A simple before/after comparison would have said {r['naive_%']:.1f}%. ")
    if r["verdict"] == "VERIFIED SAVING":
        txt += f"Both tests pass, so we count the LOW end: {r['counted']:,.0f} {u}."
        if r["counted_co2_kg"]:
            txt += f" That avoids about {r['counted_co2_kg']:,.0f} kg CO₂ on Québec's hydro-powered grid."
    elif r["verdict"] == "VERIFIED INCREASE":
        txt += "Use went UP beyond normal variation."
    elif r["verdict"] == "CONFOUNDED":
        txt += "The change is real, but another logged event overlaps it — nothing is counted for this action."
    elif r["verdict"] == "MODEL NOT ACCURATE ENOUGH":
        txt += (f"This meter's model error ({r['cv']:.0f}%) is above the ASHRAE Guideline 14 limit, so no saving is "
                f"claimed — only the raw numbers are shown.")
    else:
        txt += "At least one test fails, so the change could be normal variation. Nothing is counted."
    for w in r["warnings"]:
        txt += " ⚠ " + w
    return txt


VERIFIED = {}
for _, a in ACTIONS.iterrows():
    VERIFIED[a.action_id] = verify(a.point, a.date, 42, ACTIONS[ACTIONS.action_id != a.action_id])
    print(f"[{a.action_id}] {RESOURCE[METER_INFO[a.point][1]]['icon']} {METER_INFO[a.point][2]}: {VERIFIED[a.action_id]['verdict']}")
    print("    " + explain(VERIFIED[a.action_id]))


## Cell 9 · Can you trust it? (the fair test)
Normal months where we **know nothing was done**, each hidden from the model: how often does each method report a "saving"?

🎤 *"On months where nothing happened, a simple dashboard reported savings almost half the time. GreenLedger almost never does."*

In [ ]:
# ---- "Can you trust it?" — a fair test where we KNOW nothing was done ----
# For each of the 12 normal months in the baseline year, we pretend an action started on the 1st,
# hide that month from the model, and ask both methods whether it "saved" anything.
# A simple before/after dashboard (this month vs the 30 days before) vs. GreenLedger (cell 8 method).
TRUST = []
for p in RELIABLE:
    y = DAILY[p].where(TEMP.notna())
    miss = MISS[p]                                           # each month predicted by a model that never saw it (cell 5)
    for a, gl in miss.items():
        others = np.array([v for k, v in miss.items() if k != a])       # the range never uses the month being tested
        half, _ = range_from_misses(others)              # monthly stretches don't overlap → k = number of months
        post = y.loc[a:a + pd.offsets.MonthEnd(0)].dropna()
        prev = y.loc[a - pd.Timedelta(days=30): a - pd.Timedelta(days=1)].mean()
        TRUST.append({"point": p, "month": a, "naive_%": float((prev - post.mean()) / prev * 100),
                      "gl_%": gl, "gl_low": gl - half, "gl_high": gl + half})
TRUST = pd.DataFrame(TRUST)
TT = TRUST.dropna(subset=["gl_%", "naive_%"])
TRUST_SUMMARY = {
    "tests": len(TT),
    "naive_saving": int((TT["naive_%"] > 0).sum()),
    "naive_big": int((TT["naive_%"] >= 5).sum()),
    "gl_saving": int((TT.gl_low > 0).sum()),
    "gl_increase": int((TT.gl_high < 0).sum()),
}
TRUST_SUMMARY["naive_avg_abs"] = round(float(TT["naive_%"].abs().mean()), 1)
TRUST_SUMMARY["gl_avg_abs"] = round(float(TT["gl_%"].abs().mean()), 1)
print(f"{TRUST_SUMMARY['tests']} normal months on meters with an accurate model — nothing was done in any of them.")
print(f"  A simple before/after dashboard reported a 'saving' in {TRUST_SUMMARY['naive_saving']} of them "
      f"({TRUST_SUMMARY['naive_big']} of 5% or more); average apparent change {TRUST_SUMMARY['naive_avg_abs']}%.")
print(f"  GreenLedger verified a saving in {TRUST_SUMMARY['gl_saving']} and an increase in {TRUST_SUMMARY['gl_increase']}; "
      f"average apparent change {TRUST_SUMMARY['gl_avg_abs']}%.")


## Cell 9r · 💡 Recommendations — the optimisation layer
Turns each building's own data into actions, plus a **what-if simulator** (you pick: °C of setpoint change, % less on weekends, % less always-on load → kWh, m³, CO₂ from the building's own model and last year's real weather): weekend/holiday schedules (benchmarked against another Concordia meter that already does better), heating/cooling sensitivity per 1 °C (with last year's real weather), always-on base load, gas used regardless of weather, missing recycling/compost bins, cheaper waste streams. Every size is a what-if from real numbers — **potential, not verified** — and a person approves or rejects each one. Approved actions go into the register, so the ghost twin verifies them later.

🎤 *"Where's the optimisation? Here: recommendations sized from real data → a person approves → the ghost twin proves it. Recommend, act, verify."*

In [ ]:
# ---- 💡 Recommendations: what to do next — read straight from each building's own data ----
# No invented targets. Every size below is either
#   • an internal benchmark — "if this meter did what another Concordia meter ALREADY does", or
#   • a per-unit sensitivity — "each 1 °C / each 1 % is worth …", computed with last year's real weather.
# Each recommendation is marked "potential — not verified": once it is done, log it in the action register
# and GreenLedger verifies it after 4–8 weeks (cell 8). That closes the loop: recommend → act → verify.
LANDFILL_T_CO2E_PER_T = 0.58 / 0.907185   # EPA GHG Emission Factors Hub (2024), mixed MSW landfilled: 0.58 t CO2e per short ton
RECS = []


def _rec(group, area, where, finding, size, size_value, unit, action, point=None):
    RECS.append({"priority": group, "area": area, "where": where, "finding": finding, "potential (not verified)": size,
                 "_value": size_value, "unit": unit, "next step": action, "_point": point})


# 1) open alerts → fix now
for _, a in ALERT_TABLE.iterrows():
    _rec("1 · Fix now", RESOURCE[METER_INFO[a.point][1]]["label"], METER_INFO[a.point][2],
         a["what happened"], f"{a['extra use']} extra already used", np.nan, "", "Check what was running; log the fix as an action.", a.point)

BY_RES = {}
for p in RELIABLE:
    BY_RES.setdefault(METER_INFO[p][1], []).append(p)
BASE_DAYS = pd.date_range(BASE_START, BASE_END, freq="D")
N_OFF = int(OFF_DAY.loc[BASE_START:BASE_END].sum())

# 2) weekend / holiday schedules — internal benchmark
OFFDAY = {}
for p in RELIABLE:
    y = DAILY[p].loc[BASE_START:BASE_END]
    off = OFF_DAY.loc[BASE_START:BASE_END]
    W, O = float(y[~off].mean()), float(y[off].mean())
    OFFDAY[p] = {"weekday": W, "offday": O, "cut": (W - O) / W * 100}
for res, pts in BY_RES.items():
    best = max(pts, key=lambda q: OFFDAY[q]["cut"])
    for p in pts:
        d, bd = OFFDAY[p], OFFDAY[best]
        unit = RESOURCE[res]["unit"]
        if p == best:
            continue
        gap = (bd["cut"] - d["cut"]) / 100 * d["weekday"] * N_OFF
        _rec("2 · Quick win", RESOURCE[res]["label"], METER_INFO[p][2],
             f"On weekends and holidays it uses {abs(d['cut']):.0f}% {'less' if d['cut'] >= 0 else 'MORE'} than on weekdays. "
             f"{METER_INFO[best][2]} already cuts {bd['cut']:.0f}%.",
             f"≈ {gap:,.0f} {unit}/year if it matched {METER_INFO[best][2]}", gap, unit,
             "Review HVAC and lighting schedules for weekends and holidays.", p)

# 3) temperature sensitivity — each 1 °C of setpoint / balance temperature, with last year's real weather
for p in RELIABLE:
    m, res = CP_MODEL[p], METER_INFO[p][1]
    unit = RESOURCE[res]["unit"]
    t = TEMP.loc[BASE_START:BASE_END].dropna()
    exp_year = float(expected(m, t.index).sum())
    i = 2
    if m["th"] is not None:
        sh = m["b"][i]; i += 1
        per_deg = float(sh * np.clip(m["th"] - t, 0, 1).sum())
        share = float(sh * np.maximum(0, m["th"] - t).sum()) / exp_year * 100
        _rec("3 · Plan", RESOURCE[res]["label"], METER_INFO[p][2],
             f"{share:.0f}% of its use is driven by cold weather (starts below {m['th']} °C; +{sh:,.0f} {unit} per °C colder).",
             f"≈ {per_deg:,.0f} {unit}/year for each 1 °C lower heating balance point", per_deg, unit,
             "Lower heating setpoints / add night & weekend setbacks; then verify.", p)
    if m["tc"] is not None:
        sc = m["b"][i]
        per_deg = float(sc * np.clip(t - m["tc"], 0, 1).sum())
        share = float(sc * np.maximum(0, t - m["tc"]).sum()) / exp_year * 100
        _rec("3 · Plan", RESOURCE[res]["label"], METER_INFO[p][2],
             f"{share:.0f}% of its use is driven by warm weather (starts above {m['tc']} °C; +{sc:,.0f} {unit} per °C warmer).",
             f"≈ {per_deg:,.0f} {unit}/year for each 1 °C higher cooling balance point", per_deg, unit,
             "Raise cooling setpoints / widen the dead-band; then verify.", p)

# 4) base load (electricity, hourly data) — what runs 24/7
BASELOAD = {}
for p in [q for q in RELIABLE if METER_INFO[q][1] == "electricity"]:
    h = HOURLY[p].loc[BASE_START:BASE_END + pd.Timedelta(hours=23)]
    g = h.groupby(h.index.normalize())
    ok = g.count() >= COMPLETE * 24
    mn, mean = g.min()[ok], g.mean()[ok]
    kw = float(mn.median())
    BASELOAD[p] = {"kW": kw, "share": float((mn / mean).median() * 100), "kWh_year": kw * 8760}
    _rec("3 · Plan", "Electricity", METER_INFO[p][2],
         f"Never drops below about {kw:,.0f} kW, even at night — {BASELOAD[p]['share']:.0f}% of its average demand runs around the clock.",
         f"≈ {kw * 8760 / 100:,.0f} kWh/year for each 1% cut in this always-on load", kw * 8760 / 100, "kWh",
         "Night walk-through: ventilation, servers, lab equipment left on.", p)

# 5) gas used regardless of weather
for p in [q for q in RELIABLE if METER_INFO[q][1] == "gas"]:
    m = CP_MODEL[p]
    t = TEMP.loc[BASE_START:BASE_END].dropna()
    base_part = float((m["b"][0] + m["b"][1] * OFF_DAY.reindex(t.index).astype(float)).sum())
    share = base_part / float(expected(m, t.index).sum()) * 100
    _rec("3 · Plan", "Natural gas", METER_INFO[p][2],
         f"{share:.0f}% of its gas is used whatever the weather (≈ {m['b'][0]:,.0f} m³ on a mild weekday).",
         f"≈ {base_part / 100:,.0f} m³/year for each 1% cut in weather-independent gas", base_part / 100, "m³",
         "Check boiler lock-out in warm weather and hot-water schedules.", p)

# 6) waste — streams missing where landfill is invoiced, cost and CO2e per tonne moved
_wb = WASTE[WASTE.tonnes > 0].pivot_table(index="Building", columns="Waste_Type", values="tonnes", aggfunc="sum").fillna(0)
_wbase = (WASTE[(WASTE.Date >= BASE_START) & (WASTE.Date <= BASE_END) & (WASTE.Waste_Type == "Landfill")]
          .groupby("Building").tonnes.sum())
for bld, r in _wb.iterrows():
    if r.get("Landfill", 0) > 0:
        missing = [STREAM_LABEL[s_].lower() for s_ in DIVERTED if r.get(s_, 0) == 0]
        if missing:
            lt = float(_wbase.get(bld, 0))
            _rec("2 · Quick win", "Waste", f"{bld} bins",
                 f"{r['Landfill']:,.0f} t to landfill invoiced here (all years), but no {' / '.join(missing)} stream at this location.",
                 f"{lt:,.0f} t landfilled here in the baseline year ≈ {lt * LANDFILL_T_CO2E_PER_T:,.0f} t CO₂e; "
                 f"each tonne diverted avoids ≈ {LANDFILL_T_CO2E_PER_T:.2f} t CO₂e", lt, "t",
                 "Add the missing bins next to the landfill bins; track the diversion rate.")
_cheap = WASTE_COST_PER_T.drop(labels=["Landfill", "Bulky_waste"], errors="ignore")
_cheap = _cheap[_cheap < WASTE_COST_PER_T.get("Landfill", np.inf)]
if len(_cheap):
    s = _cheap.idxmin()
    diff = WASTE_COST_PER_T["Landfill"] - _cheap[s]
    _rec("2 · Quick win", "Waste", "campus", f"Landfill costs ${WASTE_COST_PER_T['Landfill']:,.0f}/t; {STREAM_LABEL[s].lower()} costs "
         f"${_cheap[s]:,.0f}/t (from the invoices).", f"${diff:,.0f} saved per tonne moved from landfill to {STREAM_LABEL[s].lower()}",
         diff, "$/t", "Better sorting signs / bin placement for recyclables.")
LANDFILL_BASE_T = float(WASTE_BASE.get("Landfill", 0))
LANDFILL_BASE_CO2E = LANDFILL_BASE_T * LANDFILL_T_CO2E_PER_T

RECS = pd.DataFrame(RECS).sort_values(["priority", "area", "_value"], ascending=[True, True, False]).reset_index(drop=True)
_wb_drop = RECS[(RECS.area == "Waste") & (RECS._value < 1)].index      # locations with no landfill in the baseline year
RECS = RECS.drop(_wb_drop).reset_index(drop=True)
RECS.insert(0, "id", [f"R-{i + 1:02d}" for i in range(len(RECS))])

# ---- human in the loop: every recommendation waits for a person to approve or reject it (with a reason) ----
DECISIONS_CSV = os.path.join(OUT_DIR, "recommendation_decisions.csv")
DECISIONS = pd.read_csv(DECISIONS_CSV) if os.path.exists(DECISIONS_CSV) else pd.DataFrame(
    columns=["time", "rec_id", "decision", "by", "reason", "action_id"])


def decide_rec(rec_id, decision, by="", reason="", action_date=None):
    """Approve → the action goes into the register (dated) so the ghost twin can verify it later. Reject → reason kept."""
    global DECISIONS
    r = RECS.set_index("id").loc[rec_id]
    aid = ""
    if decision == "approved" and isinstance(r["_point"], str) and action_date:
        aid = add_action(action_date, r["_point"], f"{r['next step']} ({rec_id})", None, f"recommendation {rec_id}")
    DECISIONS = pd.concat([DECISIONS, pd.DataFrame([{"time": datetime.datetime.now().strftime("%Y-%m-%d %H:%M"),
                                                     "rec_id": rec_id, "decision": decision, "by": by, "reason": reason,
                                                     "action_id": aid}])], ignore_index=True)
    DECISIONS.to_csv(DECISIONS_CSV, index=False)
    return aid
TOP_RECS = pd.concat([RECS[RECS.priority.str.startswith("1")].head(1),
                      RECS[RECS.priority.str.startswith("2") & (RECS.area != "Waste")].head(1),
                      RECS[(RECS.priority.str.startswith("3")) & (RECS.area == "Electricity")].head(1)])
print(f"{len(RECS)} recommendations ({RECS.priority.value_counts().sort_index().to_dict()})")
print(f"Landfill in the baseline year: {LANDFILL_BASE_T:,.0f} t ≈ {LANDFILL_BASE_CO2E:,.0f} t CO₂e (EPA factor)\n")
for _, r in RECS.iterrows():
    print(f"{r.id} [{r.priority}] {r.area} · {r['where']}\n    {r.finding}\n    → {r['potential (not verified)']} · {r['next step']}")


# ---- 🎚️ What-if simulator: YOU choose the scenario; the numbers come from each building's own model ----
# Uses the change-point slopes and LAST YEAR'S REAL WEATHER, campus meters only (no double counting:
# EV sits inside EV+GM+MB+TD, GM inside EV+GM+MB). These are projections — not savings until verified.
_T_BASE = TEMP.loc[BASE_START:BASE_END].dropna()
_OFF_BASE = OFF_DAY.reindex(_T_BASE.index).astype(bool)


def _slopes(m):
    b = list(m["b"]); i = 2; sh = sc = 0.0
    if m["th"] is not None:
        sh = b[i]; i += 1
    if m["tc"] is not None:
        sc = b[i]
    return sh, sc


def what_if(heat_down=0.0, cool_up=0.0, offday_cut=0.0, base_cut=0.0):
    """heat_down: °C lower heating balance point · cool_up: °C higher cooling balance point ·
    offday_cut: % less use on weekends/holidays · base_cut: % less always-on electricity."""
    rows = []
    for res in ("electricity", "gas"):
        for p in CAMPUS_TOTAL[res]:
            if p not in RELIABLE:
                continue
            m = CP_MODEL[p]; sh, sc = _slopes(m); t = _T_BASE
            heat = sh * float((np.maximum(0, m["th"] - t) - np.maximum(0, m["th"] - heat_down - t)).sum()) if m["th"] is not None else 0.0
            cool = sc * float((np.maximum(0, t - m["tc"]) - np.maximum(0, t - m["tc"] - cool_up)).sum()) if m["tc"] is not None else 0.0
            offd = float(expected(m, t.index)[_OFF_BASE.values].sum()) * offday_cut / 100
            base = BASELOAD[p]["kWh_year"] * base_cut / 100 if p in BASELOAD else 0.0
            rows.append({"point": p, "meter": METER_INFO[p][2], "resource": res, "unit": RESOURCE[res]["unit"],
                         "heating": heat, "cooling": cool, "weekends": offd, "base load": base,
                         "total": heat + cool + offd + base})
    W = pd.DataFrame(rows)
    kwh = float(W[W.resource == "electricity"].total.sum()); m3 = float(W[W.resource == "gas"].total.sum())
    co2_e = kwh * ELEC_KG_CO2_PER_KWH / 1000
    co2_g = m3 * (GAS_M3_PER_METER_UNIT or 0) * GAS_KG_CO2_PER_M3 / 1000 if GAS_M3_PER_METER_UNIT else None
    return W, kwh, m3, co2_e, co2_g


_W, _k, _m, _ce, _cg = what_if(1, 1, 10, 1)
print(f"\nWhat-if example (1 °C lower heating point, 1 °C higher cooling point, 10% less on weekends, 1% less always-on load), "
      f"with last year's weather: ≈ {_k / 1000:,.0f} MWh electricity and ≈ {_m:,.0f} m³ (meter units) of gas per year "
      f"→ {_ce:,.1f} t CO₂ from electricity. Projection only — verify after doing it.")


## Cell 10 · 🌍 Carbon report
GHG Protocol scopes with sourced factors: Scope 2 electricity (Hydro-Québec), Scope 3 landfill and e-waste (US EPA Emission Factors Hub), Scope 1 gas withheld until the meter unit is confirmed. Data gaps are not filled.

🎤 *"In Québec, landfill waste out-emits all metered electricity — on India's grid the same electricity would be ~1,400× more CO₂. Where you are changes what to fix first."*

In [ ]:
# ---- 🌍 Carbon report (GHG Protocol scopes) — only what can be measured and sourced ----
# Scope 1  natural gas burned on campus  → needs the gas meter unit confirmed (cell 8); until then: not computed
# Scope 2  purchased electricity         → metered kWh × Hydro-Québec 0.5 kg CO2/MWh (2019, 2020)
# Scope 3  waste sent to landfill        → invoiced tonnes × EPA landfill factor (cell 9r)
#          e-waste handed over           → kg × EPA mixed-electronics factor (cell 7e), if a register is uploaded
# Days with no good reading are NOT filled in; each line shows how complete the data was.
def _period(d):
    y0 = d.year if d.month >= BASE_START.month else d.year - 1
    return f"{pd.Timestamp(y0, BASE_START.month, 1):%b %Y} → {pd.Timestamp(y0 + 1, BASE_START.month, 1) - pd.Timedelta(days=1):%b %Y}"


CARBON_ROWS = []
for p in CAMPUS_TOTAL["electricity"]:
    s = DAILY[p].dropna()
    for per, g in s.groupby(s.index.map(_period)):
        days_in = len(pd.date_range(*[pd.Timestamp(x) for x in (g.index.min(), g.index.max())]))
        CARBON_ROWS.append({"period": per, "scope": "Scope 2 · electricity", "source": METER_INFO[p][2],
                            "amount": float(g.sum()), "unit": "kWh", "t CO2e": float(g.sum()) * ELEC_KG_CO2_PER_KWH / 1000,
                            "data": f"{len(g)} good days"})
for p in CAMPUS_TOTAL["gas"]:
    s = DAILY[p].dropna()
    for per, g in s.groupby(s.index.map(_period)):
        co2 = co2_kg(p, float(g.sum()))
        CARBON_ROWS.append({"period": per, "scope": "Scope 1 · natural gas", "source": METER_INFO[p][2],
                            "amount": float(g.sum()), "unit": "meter units (documented m³)",
                            "t CO2e": co2 / 1000 if co2 is not None else np.nan,
                            "data": f"{len(g)} good days" + ("" if co2 is not None else " · unit not confirmed → CO₂ not computed")})
_lw = WASTE[(WASTE.Waste_Type == "Landfill") & WASTE.tonnes.notna()]
for per, g in _lw.groupby(_lw.Date.map(_period)):
    CARBON_ROWS.append({"period": per, "scope": "Scope 3 · waste to landfill", "source": "hauling invoices",
                        "amount": float(g.tonnes.sum()), "unit": "t", "t CO2e": float(g.tonnes.sum()) * LANDFILL_T_CO2E_PER_T,
                        "data": f"{len(g)} weighed pickups"})
if EWASTE is not None and EWASTE["leaving campus"].any():
    kg = float(EWASTE.loc[EWASTE["leaving campus"], "weight_kg"].sum())
    CARBON_ROWS.append({"period": "register", "scope": "Scope 3 · e-waste", "source": "asset register",
                        "amount": kg, "unit": "kg", "t CO2e": kg / 1000 * EWASTE_EOL_T_CO2E_PER_T, "data": "end-of-life only"})
CARBON = pd.DataFrame(CARBON_ROWS)
CARBON_BY_PERIOD = CARBON.pivot_table(index="period", columns="scope", values="t CO2e",
                                      aggfunc=lambda s: s.sum(min_count=1)).round(1)
INDIA_GRID_KG_PER_KWH = 0.710   # CEA CO2 Baseline Database v21.0 (India) — for comparison only

def avoided_table():
    ty = ACTIONS.set_index("action_id")["type"]
    return pd.DataFrame([{"action": k, "type": "ground-truth test" if "Ground-truth" in str(ty.get(k, "")) else "action",
                          "meter": METER_INFO[v["point"]][2], "counted": v.get("counted", 0), "unit": v["unit"],
                          "kg CO2e avoided": v.get("counted_co2_kg") or 0.0, "verdict": v["verdict"]} for k, v in VERIFIED.items()])


def refresh_ewaste_carbon():
    """Add / update the e-waste line after a register is loaded in the app."""
    global CARBON, CARBON_BY_PERIOD
    CARBON = CARBON[CARBON.scope != "Scope 3 · e-waste"]
    if EWASTE is not None and EWASTE["leaving campus"].any() and EWASTE.loc[EWASTE["leaving campus"], "weight_kg"].notna().any():
        kg = float(EWASTE.loc[EWASTE["leaving campus"], "weight_kg"].sum())
        CARBON = pd.concat([CARBON, pd.DataFrame([{"period": "e-waste register", "scope": "Scope 3 · e-waste", "source": "asset register",
                                                   "amount": kg, "unit": "kg", "t CO2e": kg / 1000 * EWASTE_EOL_T_CO2E_PER_T,
                                                   "data": "end-of-life only (EPA)"}])], ignore_index=True)
    CARBON_BY_PERIOD = CARBON.pivot_table(index="period", columns="scope", values="t CO2e",
                                          aggfunc=lambda s: s.sum(min_count=1)).round(1)


AVOIDED = avoided_table()
CARBON_FACTORS = pd.DataFrame([
    ("Electricity (Scope 2)", f"{ELEC_KG_CO2_PER_KWH * 1000:.1f} kg CO₂ / MWh", "Hydro-Québec — CO2 emissions and Hydro-Québec electricity 1990–2020"),
    ("India grid (comparison only)", f"{INDIA_GRID_KG_PER_KWH} kg CO₂ / kWh", "CEA CO2 Baseline Database v21.0"),
    ("Natural gas (Scope 1)", f"{GAS_KG_CO2_PER_M3} kg CO₂ / m³ (used once meter unit is confirmed)", "Québec MELCC GHG quantification guide, Table 5"),
    ("Landfilled mixed waste (Scope 3)", f"{LANDFILL_T_CO2E_PER_T:.3f} t CO₂e / t", "US EPA GHG Emission Factors Hub 2024 (0.58 t / short ton)"),
    ("E-waste, recycled (Scope 3)", f"{EWASTE_EOL_T_CO2E_PER_T:.3f} t CO₂e / t", "US EPA GHG Emission Factors Hub 2024 (0.02 t / short ton)"),
], columns=["What", "Factor", "Source"])
print("t CO₂e by 12-month period (blank = not measured / not computable):")
print(CARBON_BY_PERIOD.to_string())
_e = CARBON[(CARBON.scope.str.contains("electricity")) & (CARBON.period == _period(BASE_START))].amount.sum()
print(f"\nThe surprise for judges: Québec's grid is almost all hydro, so {_e / 1e6:,.1f} GWh of metered electricity in the "
      f"baseline year is only {_e * ELEC_KG_CO2_PER_KWH / 1000:,.0f} t CO₂ — landfill waste is the biggest measured source. "
      f"On India's grid (CEA, {INDIA_GRID_KG_PER_KWH} kg/kWh) the same electricity would be {_e * INDIA_GRID_KG_PER_KWH / 1000:,.0f} t CO₂.")

# ---- Gas sanity check: the documented unit vs everything else we measured ----
FT3_TO_M3 = 0.0283168                       # exact unit conversion (1 cubic foot in m³)
_gb = sum(float(DAILY[p].loc[BASE_START:BASE_END].sum()) for p in CAMPUS_TOTAL["gas"])
GAS_DOC_T = _gb * GAS_KG_CO2_PER_M3 / 1000             # if 1 meter unit = 1 m³ (as documented)
GAS_FT3_T = _gb * FT3_TO_M3 * GAS_KG_CO2_PER_M3 / 1000   # if the meter were actually in cubic feet
_land = float(_cb_land := CARBON[(CARBON.period == _period(BASE_START)) & CARBON.scope.str.contains("landfill")]["t CO2e"].sum())
print(f"\nGas, baseline year: {_gb:,.0f} meter units. As documented (m³) → {GAS_DOC_T:,.0f} t CO₂; "
      f"even if the unit were cubic feet → {GAS_FT3_T:,.0f} t CO₂. Either way gas is the biggest source "
      f"(landfill {_land:,.0f} t, electricity {_e * ELEC_KG_CO2_PER_KWH / 1000:,.0f} t) — so gas is the lever, and its unit "
      f"must be confirmed before a tonne figure is published.")


def carbon_by_period(include_gas_documented=False):
    C = CARBON.copy()
    if include_gas_documented and not GAS_M3_PER_METER_UNIT:
        g = C.scope.str.contains("gas")
        C.loc[g, "t CO2e"] = C.loc[g, "amount"] * GAS_KG_CO2_PER_M3 / 1000
    return C.pivot_table(index="period", columns="scope", values="t CO2e", aggfunc=lambda s: s.sum(min_count=1)).round(1)


## Cell 11 · The six sentences for your pitch
All computed above from real data. Read these out on stage.

In [ ]:
# ---- The numbers to say to the judges (all computed above from real data) ----
PARAMETERS = pd.DataFrame([
    ("Data", "Concordia University open data (Facilities), CC BY 4.0", "Real meters, real invoices — nothing simulated"),
    ("Day counts if", "≥ 90% of its hours recorded", "Common data-completeness rule; days below it are listed, not filled in"),
    ("Baseline", "12 months before (every season once)", "IPMVP / ASHRAE practice: a full annual cycle"),
    ("Expected-use model", "ASHRAE change-point model (temperature + weekday/holiday)", "ASHRAE Guideline 14 / Inverse Modeling Toolkit"),
    ("Balance temperatures", "found from the data", "every whole °C observed is tried; best by BIC"),
    ("Model accuracy grade", "Good ≤ 15% / OK ≤ 30% error; bias ≤ ±5% / ±10%", "ASHRAE Guideline 14 calibration limits (monthly / hourly data)"),
    ("Measurement window", "4, 6 or 8 whole weeks", "whole weeks = same weekday mix before and after"),
    ("Ghost twin", "the meter's better model (change-point or gradient boosting)", "picked per meter by its typical miss on 12 hidden months"),
    ("Gradient boosting", "LightGBM, library defaults", "no tuning — nothing to cherry-pick"),
    ("Placebo-in-time", "fake action every week of the year before", "same weekday mix for every fake date"),
    ("95% range", "t × typical fake saving × √(1+1/k)", "standard prediction interval; k = independent stretches in a year"),
    ("Significance", "p ≤ 5% (permutation test)", "statistical convention; VERIFIED needs range AND p"),
    ("Counted saving", "low end of the 95% range", "never overstate — the brief's requirement"),
    ("Alert bar", "biggest out-of-sample surprise of the baseline year", "each meter sets its own bar; no tuned thresholds"),
    ("Holidays", "Québec public holidays", "official calendar (python `holidays`)"),
    ("Electricity CO₂", "0.5 kg CO₂ / MWh", "Hydro-Québec, CO2 emissions and Hydro-Québec electricity 1990–2020 (2019 & 2020)"),
    ("Natural gas CO₂", "1.926 kg CO₂ / m³ — used only once the meter unit is confirmed",
     "Québec MELCC GHG quantification guide, Table 5"),
    ("Landfill CO₂e", "0.639 t CO₂e per tonne", "US EPA GHG Emission Factors Hub 2024: 0.58 t per short ton, mixed MSW landfilled"),
    ("E-waste CO₂e", "0.022 t CO₂e per tonne (end of life)", "US EPA GHG Emission Factors Hub 2024: 0.02 t per short ton, mixed electronics recycled"),
    ("E-waste routes", "keep / registered refurbisher / registered recycler", "India E-Waste (Management) Rules, 2022 (bulk consumer duties)"),
    ("E-waste expected life", "from the CPCB EPR portal, per EEE code", "entered with each device — never assumed"),
    ("Room vacancy switch-off", "20 minutes", "ASHRAE 90.1 occupancy-sensor rule (changeable in the app)"),
    ("Room comfort band", "80% acceptability, 7-day mean outdoor temperature", "ASHRAE 55 adaptive model via pythermalcomfort"),
    ("Fan credit", "0.6 m/s → +1.2 °C", "ASHRAE 55 elevated-air-speed step"),
    ("People counting", "YOLOv8n (COCO), default confidence", "Ultralytics; class 'person' only"),
    ("Waste diversion", "(compost + paper + mixed recycling) ÷ (those + landfill)", "bulky waste excluded: Concordia says its split is unknown"),
], columns=["What", "Value", "Why / source"])

_ok = {k: v for k, v in VERIFIED.items() if "saving_%" in v}
_ver = [v for v in _ok.values() if v["verdict"] == "VERIFIED SAVING"]
_gas = [v for v in _ok.values() if METER_INFO[v["point"]][1] == "gas"]
_top_e = RECS[(RECS.area == "Electricity") & RECS.priority.str.startswith("2")].head(1)
HEADLINE = []
HEADLINE.append(f"1. One real campus, every resource: {DAILY.shape[1]} meters (electricity, gas, water) + {len(WASTE)} waste invoices, "
                f"{HOURLY.index.min():%b %Y}–{HOURLY.index.max():%b %Y} — plus a live camera for room-level control.")
HEADLINE.append(f"2. It optimises: {len(RECS)} data-driven recommendations, each sized from the building's own data"
                + (f" — e.g. {_top_e.iloc[0]['where']}: {_top_e.iloc[0]['finding']} ({_top_e.iloc[0]['potential (not verified)']})." if len(_top_e) else "."))
HEADLINE.append(f"3. Trust test on {TRUST_SUMMARY['tests']} normal months (nothing done): a before/after dashboard showed a 'saving' "
                f"{TRUST_SUMMARY['naive_saving']} times; GreenLedger certified {TRUST_SUMMARY['gl_saving']}.")
if _gas:
    g = max(_gas, key=lambda v: v["naive_%"])
    HEADLINE.append(f"4. Not overstated: a before/after dashboard says gas fell {g['naive_%']:.0f}% after March 2020; the ghost twin "
                    f"(real weather, real calendar) says {g['saving_%']:.0f}% and the placebo test says that is normal noise "
                    f"(p = {g['p']:.2f}). Nothing counted.")
if len(DISCOVERY_TABLE) and _ver:
    first = DISCOVERY_TABLE.sort_values("first seen").iloc[0]
    best = max(_ver, key=lambda v: v["saving_%"])
    HEADLINE.append(f"5. Ground-truth test: nobody told GreenLedger about the COVID-19 closure — it flagged the drop on "
                    f"{pd.Timestamp(first['first seen']):%d %b %Y}, and the ghost twin measured {METER_INFO[best['point']][2]} "
                    f"electricity {best['saving_%']:.0f}% below normal (p = {best['p']:.3f}). The same engine verifies real actions.")
_cb = CARBON[CARBON.period == _period(BASE_START)]
HEADLINE.append(f"6. Carbon, honestly: in the baseline year landfill waste ≈ {_cb[_cb.scope.str.contains('landfill')]['t CO2e'].sum():,.0f} t CO₂e "
                f"vs metered electricity ≈ {_cb[_cb.scope.str.contains('electricity')]['t CO2e'].sum():,.0f} t (Québec hydro). Gas CO₂ is "
                f"withheld until the meter unit is confirmed; {len(FAULTS)} bad meter-days were excluded with reasons.")
print("\n".join(HEADLINE))


## Cell 12 · Export the report
HTML report + ZIP "labelled dataset" (daily readings with ghost-twin value, fault reason and alert id; alerts; actions; verification; recommendations & decisions; carbon; waste; e-waste; room log; sources). Files are in the 📁 panel under `greenledger_out`.

In [ ]:
# ---- Exportable report + labelled dataset (for the judges / facilities team) ----
def labelled_dataset():
    rows = []
    for p in DAILY:
        f = FAULTS[FAULTS.point == p].set_index("day").rule
        df = pd.DataFrame({"date": DAYS, "meter": METER_INFO[p][2], "resource": METER_INFO[p][1],
                           "system_name": METER_INFO[p][0], "actual": DAILY[p].values,
                           "expected": EXPECTED[p].reindex(DAYS).values, "temp_C": TEMP.values,
                           "off_day": OFF_DAY.values, "period": np.where(DAYS <= BASE_END, "baseline", "monitoring")})
        df["data_fault"] = df.date.map(f).fillna("")
        df["alert"] = ""
        for _, a in ALERT_TABLE[ALERT_TABLE.point == p].iterrows():
            df.loc[(df.date >= pd.Timestamp(a["from"])) & (df.date <= pd.Timestamp(a["to"])), "alert"] = a["id"]
        rows.append(df)
    return pd.concat(rows, ignore_index=True)


def verification_table():
    out = []
    for _, a in ACTIONS.iterrows():
        r = VERIFIED.get(a.action_id) or verify(a.point, a.date, 42, ACTIONS[ACTIONS.action_id != a.action_id])
        out.append({"action": a.action_id, "date": a.date.date(), "meter": METER_INFO[a.point][2], "title": a.title,
                    "verdict": r["verdict"], "change %": round(r.get("saving_%", np.nan), 1), "p": round(r.get("p", np.nan), 3),
                    "95% low": round(r.get("low_%", np.nan), 1), "95% high": round(r.get("high_%", np.nan), 1),
                    "naive %": round(r.get("naive_%", np.nan), 1), "counted": round(r.get("counted", 0), 0),
                    "unit": r["unit"], "explanation": explain(r)})
    return pd.DataFrame(out)


def build_report():
    stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M")
    zpath = os.path.join(OUT_DIR, f"greenledger_concordia_{stamp}.zip")
    V = verification_table()
    with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
        z.writestr("readings_daily_labelled.csv", labelled_dataset().to_csv(index=False))
        z.writestr("alerts.csv", ALERT_TABLE.drop(columns="point").to_csv(index=False))
        z.writestr("discoveries.csv", DISCOVERY_TABLE.drop(columns="point").to_csv(index=False))
        z.writestr("data_faults.csv", FAULTS.to_csv(index=False))
        z.writestr("actions.csv", ACTIONS.to_csv(index=False))
        z.writestr("verification.csv", V.to_csv(index=False))
        z.writestr("model_accuracy.csv", ACC.drop(columns="point").to_csv(index=False))
        z.writestr("trust_test.csv", TRUST.to_csv(index=False))
        z.writestr("waste_monthly_tonnes.csv", WASTE_MONTHLY.to_csv())
        z.writestr("waste_yearly.csv", WASTE_YEARLY.to_csv())
        z.writestr("parameters_and_sources.csv", PARAMETERS.to_csv(index=False))
        z.writestr("recommendations.csv", RECS.drop(columns=["_value", "_point"]).to_csv(index=False))
        z.writestr("recommendation_decisions.csv", DECISIONS.to_csv(index=False))
        z.writestr("carbon_inventory.csv", CARBON.to_csv(index=False))
        z.writestr("carbon_factors.csv", CARBON_FACTORS.to_csv(index=False))
        z.writestr("model_choice.csv", MODEL_CHECK.drop(columns="point").to_csv(index=False))
        z.writestr("room_control_log.csv", pd.DataFrame(ROOM["log"] if "ROOM" in globals() else []).to_csv(index=False))
        if EWASTE is not None:
            z.writestr("ewaste_plan.csv", EWASTE.to_csv(index=False))
        z.writestr("README.txt", "GreenLedger — reference campus: Concordia University (Montréal).\n"
                   "Source: Concordia University open data (Facilities), CC BY 4.0. Nothing simulated or filled in.\n"
                   "readings_daily_labelled.csv: one row per meter per day — actual, expected, data_fault reason, alert id.\n")
    tbl = lambda d: d.to_html(index=False, border=0, classes="t")
    html = f"""<html><head><meta charset="utf-8"><title>GreenLedger report</title><style>
body{{font-family:Inter,Arial,sans-serif;max-width:1000px;margin:30px auto;color:#1f2937;line-height:1.5}}
h1{{color:#047857}} .t{{border-collapse:collapse;width:100%;font-size:13px}} .t td,.t th{{border-bottom:1px solid #e5e7eb;padding:6px;text-align:left}}
.t th{{background:#f3f4f6}} li{{margin:6px 0}}</style></head><body>
<h1>GreenLedger — campus resource report</h1>
<p>Reference campus: <b>Concordia University, Montréal</b> · Concordia open data (CC BY 4.0) · generated {datetime.datetime.now():%d %b %Y %H:%M}</p>
<h2>Headlines</h2><ul>{''.join(f'<li>{h}</li>' for h in HEADLINE)}</ul>
<h2>Recommendations (potential — verified only after they are done)</h2>{tbl(RECS[['id', 'priority', 'area', 'where', 'finding', 'potential (not verified)', 'next step']])}
<h2>Carbon inventory (t CO₂e; blank = not measured / not computable)</h2>{CARBON_BY_PERIOD.reset_index().to_html(index=False, border=0, classes='t', na_rep='—')}
{tbl(CARBON_FACTORS)}
<h2>Did it work? (action register — ghost twin + placebo-in-time)</h2>{tbl(V[['action', 'date', 'meter', 'title', 'verdict', 'change %', '95% low', '95% high', 'p', 'naive %', 'counted', 'unit']])}
<h2>Alerts</h2>{tbl(ALERT_TABLE[['id', 'meter', 'from', 'to', 'extra use', 'what happened']])}
<h2>Discoveries (unexplained sustained drops)</h2>{tbl(DISCOVERY_TABLE.drop(columns='point'))}
<h2>Waste per 12-month period (tonnes)</h2>{WASTE_YEARLY.to_html(border=0, classes='t')}
<h2>Model accuracy</h2>{tbl(ACC.drop(columns='point'))}
<h2>Which model is the ghost twin?</h2>{tbl(MODEL_CHECK.drop(columns='point'))}
{'<h2>E-waste plan</h2>' + tbl(EWASTE[['device_id', 'device_type', 'department', 'age_years', 'expected_life_years', 'condition', 'route', 'why']]) if EWASTE is not None else ''}
{'<h2>Room control log</h2>' + tbl(pd.DataFrame(ROOM['log'])) if 'ROOM' in globals() and ROOM['log'] else ''}
<h2>Parameters and sources</h2>{tbl(PARAMETERS)}
</body></html>"""
    hpath = os.path.join(OUT_DIR, f"greenledger_report_{stamp}.html")
    open(hpath, "w").write(html)
    return hpath, zpath


REPORT_HTML, REPORT_ZIP = build_report()
print("Report:", REPORT_HTML)
print("Labelled dataset (ZIP):", REPORT_ZIP)


## Cell 13 · 🎥 Live room engine — live twin vs ghost twin of one room
* **People** counted on every camera frame with **YOLOv8n** (median of the last 3 frames, so one bad frame can't flip a switch).
* **Comfort** from the **ASHRAE 55 adaptive model** (pythermalcomfort) for this week's real outdoor weather (Open-Meteo):
  cooler than comfortable → fan & AC off · comfortable → nothing · warmer than neutral → **fan ON** · too warm even with a fan → AC at the warmest comfortable °C.
  If no room temperature is typed, the live outdoor temperature is used and the screen says so.
* **Empty** for 20 min (**ASHRAE 90.1**) → everything **OFF** (demo timing: 30 s).
* **Live twin** = the room as GreenLedger runs it · **Ghost twin** = the same room where people switch things on but nobody switches them off.
  Saving = ghost − live over time × each device's nameplate kW (from its label) — an estimate for one room; campus savings are verified on meters.
* **Switching:** smart plug / Home Assistant webhook (`GL_DEVICE_WEBHOOK`) if set; otherwise a **virtual switch** that shows exactly what would switch.

In [ ]:
# ---- 🎥 Live room: camera → people → fan / AC / lights, with a LIVE TWIN vs GHOST TWIN of the room ----
# • People: YOLOv8n on every camera frame; the count is the median of the last 3 frames (removes flicker).
# • Comfort: ASHRAE 55 adaptive model (pythermalcomfort) for this week's outdoor weather (Open-Meteo):
#       colder than the comfort band ............ AC off, fan off
#       comfortable, below neutral .............. no fan, no AC
#       above neutral, fan keeps it comfortable . FAN ON (0.6 m/s air speed widens the band by 1.2 °C)
#       too warm even with a fan ................ AC on at the warmest comfortable whole °C + fan
#   Room temperature: typed in (thermometer / AC display); if empty, the live outdoor temperature is used
#   and the screen says so.
# • Empty for the vacancy time (ASHRAE 90.1: 20 min; shorter in demo mode) → everything off.
# • LIVE TWIN  = the room as GreenLedger runs it.
#   GHOST TWIN = the same room without GreenLedger: devices go on when people need them, but nobody
#                switches them off when the room empties (the case occupancy controls exist for).
#   Saving = ghost − live, integrated over time, using each device's nameplate kW (from its label).
#   This is an ESTIMATE for one room; campus savings are verified on real meters in "Act & Prove".
# • Switching: a smart plug / Home Assistant webhook (GL_DEVICE_WEBHOOK) if configured; otherwise a VIRTUAL
#   switch — the dashboard shows exactly what would be switched, and says so.
import threading as _th, urllib.parse, collections, shutil
try:
    from ultralytics import YOLO
    YOLO_PATH = os.path.join(OUT_DIR, "yolov8n.pt")           # saved now → no download needed during the demo
    if not os.path.exists(YOLO_PATH):
        _m = YOLO("yolov8n.pt")
        shutil.copy(getattr(_m, "ckpt_path", None) or "yolov8n.pt", YOLO_PATH)
    YOLO_MODEL = YOLO(YOLO_PATH)
except Exception as e:
    YOLO_MODEL = None
    print("Camera detection unavailable:", e)
try:
    from pythermalcomfort.models import adaptive_ashrae
except Exception as e:
    adaptive_ashrae = None
    print("pythermalcomfort unavailable:", e)

VACANCY_MIN_DEFAULT = 20      # ASHRAE 90.1 occupancy-sensor rule (minutes)
DEMO_VACANCY_MIN = 0.5        # demo mode: 30 s, so judges can watch it switch off
CONF_DEFAULT = 0.5            # detection confidence shown on screen and adjustable (Ultralytics' own default is 0.25)
FAN_AIR_SPEED = 0.6           # m/s — ASHRAE 55 first elevated-air-speed step (+1.2 °C)
STILL_AIR = 0.1               # m/s — still air in ASHRAE 55
BEE_DEFAULT_AC_C = 24         # India BEE default AC setting, for comparison
DEVICES = ("Fan", "Lights", "AC")
DEVICE_WEBHOOK = os.environ.get("GL_DEVICE_WEBHOOK", "")
ROOM_LOG_CSV = os.path.join(OUT_DIR, "room_control_log.csv")
WEATHER = {"city": None, "now": None, "trm": None, "fetched": None, "error": None}


def _new_room():
    return {"lock": _th.Lock(), "last_person": None, "people": 0, "raw": 0, "counts": collections.deque(maxlen=3),
            "live": {d: False for d in DEVICES}, "ghost": {d: False for d in DEVICES}, "setpoint": None,
            "history": [], "log": [], "temp_source": ""}


ROOM = _new_room()


def fetch_weather(city):
    """Current outdoor temperature + 7-day mean (ASHRAE 55 prevailing mean outdoor temperature), Open-Meteo."""
    try:
        g = json.loads(urllib.request.urlopen(
            "https://geocoding-api.open-meteo.com/v1/search?count=1&name=" + urllib.parse.quote(city), timeout=8).read())
        r0 = g["results"][0]
        w = json.loads(urllib.request.urlopen(
            f"https://api.open-meteo.com/v1/forecast?latitude={r0['latitude']}&longitude={r0['longitude']}"
            "&current=temperature_2m&daily=temperature_2m_mean&past_days=7&forecast_days=1&timezone=auto", timeout=8).read())
        past = [t for t in w["daily"]["temperature_2m_mean"][:7] if t is not None]
        WEATHER.update(city=f"{r0['name']}, {r0.get('country', '')}", now=float(w["current"]["temperature_2m"]),
                       trm=float(np.mean(past)), fetched=datetime.datetime.now(), error=None)
    except Exception as e:
        WEATHER.update(error=str(e))
    return WEATHER


def comfort(trm):
    """ASHRAE 55 adaptive: (lower, neutral, upper at still air, upper with a fan) for 80% acceptability."""
    r = adaptive_ashrae(tdb=25, tr=25, t_running_mean=trm, v=STILL_AIR)
    rf = adaptive_ashrae(tdb=25, tr=25, t_running_mean=trm, v=FAN_AIR_SPEED)
    g = (lambda o, k: getattr(o, k)) if not isinstance(r, dict) else (lambda o, k: o[k])
    return float(g(r, "tmp_cmf_80_low")), float(g(r, "tmp_cmf")), float(g(r, "tmp_cmf_80_up")), float(g(rf, "tmp_cmf_80_up"))


def decide(people, vacant_min, room_temp, trm, vacancy_limit):
    if people == 0:
        if vacant_min is not None and vacant_min >= vacancy_limit:
            return ({d: False for d in DEVICES}, None, f"Room empty for {vacant_min:.1f} min (≥ {vacancy_limit:g} min) → everything OFF.")
        left = max(0.0, vacancy_limit - (vacant_min or 0))
        return (None, None, f"No one in view — switching off in {left * 60:.0f} s unless someone comes back.")
    if room_temp is None or trm is None or adaptive_ashrae is None:
        return ({"Lights": True, "Fan": None, "AC": None}, None,
                f"{people} people present → lights ON. Type the room temperature (or load the weather) for the fan/AC decision.")
    lo, neutral, hi, hi_fan = comfort(trm)
    if room_temp < lo:
        return ({"Lights": True, "Fan": False, "AC": False}, None,
                f"{people} present · {room_temp:.1f} °C is cooler than the comfort band ({lo:.1f}–{hi:.1f} °C) → fan and AC OFF.")
    if room_temp <= neutral:
        return ({"Lights": True, "Fan": False, "AC": False}, None,
                f"{people} present · {room_temp:.1f} °C is comfortable (neutral {neutral:.1f} °C) → no fan, no AC needed.")
    if room_temp <= hi_fan:
        return ({"Lights": True, "Fan": True, "AC": False}, None,
                f"{people} present · {room_temp:.1f} °C is above neutral ({neutral:.1f} °C); a fan keeps it comfortable up to "
                f"{hi_fan:.1f} °C → FAN ON, AC OFF.")
    sp = float(np.floor(hi_fan))
    return ({"Lights": True, "Fan": True, "AC": True}, sp,
            f"{people} present · {room_temp:.1f} °C is too warm even with a fan → AC ON at {sp:.0f} °C + fan "
            f"(warmest comfortable setting; BEE default is {BEE_DEFAULT_AC_C} °C).")


def send_command(device, on, setpoint=None):
    if not DEVICE_WEBHOOK:
        return "virtual switch (no smart plug connected)"
    try:
        req = urllib.request.Request(DEVICE_WEBHOOK, method="POST", headers={"Content-Type": "application/json"},
                                     data=json.dumps({"device": device, "on": bool(on), "setpoint": setpoint}).encode())
        urllib.request.urlopen(req, timeout=5)
        return "sent to smart switch"
    except Exception as e:
        return f"switch error: {e}"


def _log(now, dev, action, reason, how):
    ROOM["log"].append({"time": now.strftime("%H:%M:%S"), "device": dev, "action": action,
                        "people": ROOM["people"], "reason": reason, "how": how})
    pd.DataFrame(ROOM["log"]).to_csv(ROOM_LOG_CSV, index=False)


def apply(want, setpoint, reason, auto, now):
    for dev in DEVICES:
        w = want.get(dev)
        if w is None:
            continue
        if w:
            ROOM["ghost"][dev] = True                  # without GreenLedger, people still switch things ON
        changed = ROOM["live"][dev] != w or (dev == "AC" and w and setpoint != ROOM["setpoint"])
        if not changed:
            continue
        if auto:
            how = send_command(dev, w, setpoint if dev == "AC" else None)
            ROOM["live"][dev] = w
            if dev == "AC":
                ROOM["setpoint"] = setpoint if w else None
            _log(now, dev, ("ON" + (f" @ {setpoint:.0f} °C" if dev == "AC" and setpoint else "")) if w else "OFF", reason, how)
        elif not ROOM["log"] or ROOM["log"][-1]["reason"] != reason:
            _log(now, dev, "suggest " + ("ON" if w else "OFF"), reason, "auto-control is off")


def room_kw(states, kw):
    return sum(float(kw.get(d) or 0) for d in DEVICES if states[d])


def process_frame(frame, room_temp=None, auto=True, vacancy_limit=VACANCY_MIN_DEFAULT, outdoor_manual=None,
                  conf=CONF_DEFAULT, kw=None):
    """One camera frame → annotated frame + decision; updates the live and ghost twins of the room."""
    now = datetime.datetime.now()
    kw = kw or {}
    annotated, raw = frame, 0
    if frame is not None and YOLO_MODEL is not None:
        bgr = np.ascontiguousarray(np.asarray(frame)[:, :, :3][:, :, ::-1])     # camera gives RGB; YOLO expects BGR
        r = YOLO_MODEL(bgr, classes=[0], conf=float(conf), verbose=False)[0]   # class 0 = person
        raw = int(len(r.boxes))
        annotated = np.ascontiguousarray(r.plot()[:, :, ::-1])
    with ROOM["lock"]:
        ROOM["counts"].append(raw)
        people = int(np.median(ROOM["counts"]))
        ROOM["people"], ROOM["raw"] = people, raw
        if people > 0:
            ROOM["last_person"] = now
        vacant = vacancy_limit if ROOM["last_person"] is None else (now - ROOM["last_person"]).total_seconds() / 60
        if room_temp is not None:
            rt, ROOM["temp_source"] = room_temp, "room thermometer"
        elif WEATHER["now"] is not None:
            rt, ROOM["temp_source"] = WEATHER["now"], "live OUTDOOR temperature (no room temperature typed)"
        else:
            rt, ROOM["temp_source"] = None, "none"
        trm = WEATHER["trm"] if WEATHER["trm"] is not None else outdoor_manual
        want, sp, reason = decide(people, vacant, rt, trm, float(vacancy_limit))
        if want:
            apply(want, sp, reason, auto, now)
        ROOM["history"].append({"time": now, "people": people, "live_kw": room_kw(ROOM["live"], kw),
                                "ghost_kw": room_kw(ROOM["ghost"], kw),
                                "live_on": sum(ROOM["live"].values()), "ghost_on": sum(ROOM["ghost"].values())})
        ROOM["history"] = ROOM["history"][-1800:]          # last hour at one frame every 2 s
    return annotated, people, raw, reason


def room_savings():
    """Integrate ghost − live over the session: kWh (if nameplate kW given) and device-minutes switched off."""
    H = pd.DataFrame(ROOM["history"])
    if len(H) < 2:
        return 0.0, 0.0, H
    dt = H.time.diff().dt.total_seconds().fillna(0).clip(upper=60) / 3600       # hours; gaps > 1 min not counted
    kwh = float(((H.ghost_kw - H.live_kw).shift(1).fillna(0) * dt).sum())
    dev_min = float(((H.ghost_on - H.live_on).shift(1).fillna(0) * dt * 60).sum())
    return kwh, dev_min, H


print("Room control ready:", "camera detection ON (YOLOv8n)" if YOLO_MODEL else "camera detection OFF",
      "·", "comfort model ON (ASHRAE 55 adaptive)" if adaptive_ashrae else "comfort model OFF",
      "·", "smart switch via webhook" if DEVICE_WEBHOOK else "virtual switch (no smart plug connected)")
fetch_weather("Bengaluru")
print("Live weather:", f"{WEATHER['city']} {WEATHER['now']} °C now, 7-day mean {WEATHER['trm']:.1f} °C"
      if WEATHER["trm"] is not None else f"not reachable ({WEATHER['error']}) — type the outdoor 7-day mean in the app")
# self-test on a real photo that ships with Ultralytics, then reset the room
try:
    from ultralytics.utils import ASSETS
    import cv2
    _img = cv2.imread(str(ASSETS / "bus.jpg"))[:, :, ::-1]
    _a, _n, _raw, _why = process_frame(_img, room_temp=29.5, auto=True, outdoor_manual=24)
    print(f"Self-test on Ultralytics sample photo: {_raw} people detected → {_why}")
except Exception as e:
    print("Self-test skipped:", e)
ROOM = _new_room()


## Cell 14 · The web app page
One HTML page in the style of your Lovable design — cream background, dark-forest sidebar, DM Sans / Manrope, KPI cards with sparklines, an aerial campus twin and a dark *Campus insights* panel. It contains **no data**: every number is fetched from the API in the next cell.

In [ ]:
# ---- The web app page (one HTML file: Lovable-style layout, Chart.js charts, Leaflet aerial map) ----
# Fonts: DM Sans + Manrope · charts: Chart.js 4.4.1 · map: Leaflet 1.9.4 + Esri World Imagery · icons: Lucide.
# All numbers are fetched from the API in the next cell — nothing in this page is hard-coded data.
WEB_HTML = r'''<!doctype html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>GreenLedger — Campus Resource Optimisation</title>
<link rel="preconnect" href="https://fonts.googleapis.com">
<link href="https://fonts.googleapis.com/css2?family=DM+Sans:wght@400;500;600;700&family=Manrope:wght@400;500;600;700;800&display=swap" rel="stylesheet">
<link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css">
<script src="https://cdn.jsdelivr.net/npm/chart.js@4.4.1/dist/chart.umd.js"></script>
<script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script>
<script src="https://unpkg.com/lucide@0.460.0/dist/umd/lucide.min.js"></script>
<style>
:root{--radius:10px;--bg:oklch(96.2% .017 99);--ivory:oklch(98.4% .007 99);--fg:oklch(29% .029 159);--forest:oklch(33.4% .051 173);
--forest-dark:oklch(29.1% .044 174);--sage:oklch(86.5% .047 127);--pale-sage:oklch(92.8% .026 123);--lime:oklch(91.9% .177 117);
--muted:oklch(55.8% .027 148);--border:oklch(89.3% .022 126);--amber:oklch(51% .095 77);--coral:oklch(50% .13 30);
--sb-active:oklch(41.4% .05 171);--shadow:0 3px 10px oklch(29% .029 159/.04)}
*{box-sizing:border-box}html,body{margin:0;height:100%}
body{background:var(--bg);color:var(--fg);font-family:'DM Sans',system-ui,sans-serif;font-size:14px}
h1,h2,h3,.disp{font-family:Manrope,'DM Sans',sans-serif;letter-spacing:-.02em;margin:0}
a{color:inherit}
.app{display:grid;grid-template-columns:250px 1fr;min-height:100vh}
/* sidebar */
.sb{background:var(--forest);color:#fff;display:flex;flex-direction:column;position:sticky;top:0;height:100vh;padding:18px 14px}
.brand{display:flex;align-items:center;gap:10px;padding:6px 8px 22px;font:800 22px Manrope,system-ui,sans-serif;letter-spacing:-.02em}
.brand .dot{width:6px;height:6px;border-radius:50%;background:var(--lime);margin:-14px 0 0 -6px}
.sb .lbl{font:600 10.5px 'DM Sans',system-ui,sans-serif;letter-spacing:.14em;opacity:.7;padding:8px 10px}
.nav a{display:flex;align-items:center;gap:11px;padding:9px 11px;border-radius:8px;text-decoration:none;font-weight:500;font-size:13.5px;color:#e9efe9;margin:1px 0;cursor:pointer}
.nav a:hover{background:oklch(38% .05 172)}
.nav a.on{background:var(--sb-active);color:var(--lime);box-shadow:inset 0 0 0 1px oklch(48% .05 171)}
.nav a svg{width:16px;height:16px}
.nav .grp{font:600 10px 'DM Sans',system-ui,sans-serif;letter-spacing:.14em;opacity:.55;padding:14px 11px 4px}
.sbfoot{margin-top:auto;border-top:1px solid oklch(42% .045 172);padding:14px 8px 4px;font-size:12px;line-height:1.5}
.sbfoot b{display:flex;align-items:center;gap:8px;font-size:13px}.sbfoot b:before{content:"";width:8px;height:8px;border-radius:50%;background:var(--lime)}
.sbfoot .s{opacity:.8}.sbfoot .t{opacity:.55;margin-top:10px;font-size:11px}
/* top bar */
.main{min-width:0}
.top{display:flex;align-items:center;gap:14px;padding:12px 28px;border-bottom:1px solid var(--border);background:var(--bg);position:sticky;top:0;z-index:500}
.top .ttl{font:600 15px Manrope,system-ui,sans-serif;margin-right:auto}
.chip{display:inline-flex;align-items:center;gap:6px;border:1px solid var(--border);background:var(--ivory);border-radius:8px;padding:6px 10px;font-size:12.5px;font-weight:500}
.search{display:flex;align-items:center;gap:8px;border-left:1px solid var(--border);padding-left:14px;color:var(--muted)}
.search input{border:0;background:transparent;font:inherit;color:var(--fg);outline:none;width:150px}
.seg{display:inline-flex;border:1px solid var(--border);border-radius:9px;padding:3px;background:var(--ivory)}
.seg button{border:0;background:transparent;padding:6px 12px;border-radius:7px;font:500 12.5px 'DM Sans',system-ui,sans-serif;color:var(--fg);cursor:pointer}
.seg button.on{background:var(--forest);color:#fff}
.bell{position:relative}.bell i{position:absolute;top:-2px;right:-2px;width:7px;height:7px;border-radius:50%;background:var(--coral)}
.av{width:30px;height:30px;border-radius:50%;background:var(--sage);display:flex;align-items:center;justify-content:center;font:700 11px 'DM Sans',system-ui,sans-serif}
.page{padding:22px 28px 40px;max-width:1500px}
.ph{display:flex;align-items:center;gap:12px;margin-bottom:16px}
.ph .back{display:inline-flex;align-items:center;gap:5px;border:1px solid var(--border);border-radius:7px;padding:4px 9px;font-size:12px;background:var(--ivory);cursor:pointer;color:var(--muted)}
.ph h1{font-size:26px;font-weight:600}.ph .tag{border:1px solid var(--sage);color:var(--forest);background:var(--pale-sage);border-radius:999px;padding:2px 9px;font-size:11px;font-weight:600}
.eyebrow{font:600 10.5px 'DM Sans',system-ui,sans-serif;letter-spacing:.14em;text-transform:uppercase;color:var(--muted);display:flex;align-items:center;gap:6px}
.sub{color:var(--muted);font-size:13px;margin:6px 0 18px}
.hero{display:flex;justify-content:space-between;align-items:flex-end;margin-bottom:18px}
.hero h1{font-size:34px;font-weight:600;margin:6px 0 4px}
.btn{display:inline-flex;align-items:center;gap:7px;border:1px solid var(--border);background:var(--ivory);border-radius:8px;padding:8px 13px;font:600 13px 'DM Sans',system-ui,sans-serif;cursor:pointer;color:var(--fg);text-decoration:none}
.btn.pri{background:var(--forest);color:#fff;border-color:var(--forest)}.btn.pri:hover{background:var(--forest-dark)}
.btn.sm{padding:5px 10px;font-size:12px}.btn svg{width:15px;height:15px}
.btn.warn{color:var(--coral)}
/* cards */
.grid{display:grid;gap:14px}.g4{grid-template-columns:repeat(4,minmax(0,1fr))}.g3{grid-template-columns:repeat(3,minmax(0,1fr))}.g2{grid-template-columns:repeat(2,minmax(0,1fr))}
.g5{grid-template-columns:repeat(5,minmax(0,1fr))}
.split{display:grid;grid-template-columns:minmax(0,1fr) 340px;gap:16px;align-items:start}
.card{background:var(--ivory);border:1px solid var(--border);border-radius:var(--radius);box-shadow:var(--shadow);padding:16px 18px}
.card.dark{background:var(--forest);color:#fff;border-color:var(--forest)}
.card.sage{background:var(--pale-sage);border-color:var(--sage)}
.card h3{font-size:15px;font-weight:600}
.kpi .row{display:flex;justify-content:space-between;align-items:center}.kpi .lab{font-weight:600;font-size:13px}
.kpi .v{font:700 30px Manrope,system-ui,sans-serif;letter-spacing:-.03em;margin:14px 0 2px}.kpi .v small{font:500 13px 'DM Sans',system-ui,sans-serif;margin-left:6px;opacity:.7}
.kpi .d{font-size:11.5px;display:flex;align-items:center;gap:5px;opacity:.85}.kpi .d b{font-weight:700}
.kpi canvas{height:46px!important;margin-top:10px}.kpi .foot{font-size:10.5px;opacity:.6;margin-top:4px}
.ct{font:600 10.5px 'DM Sans',system-ui,sans-serif;letter-spacing:.14em;text-transform:uppercase;color:var(--muted);margin-bottom:10px}
.dark .ct{color:var(--sage)}
.pill{display:inline-flex;align-items:center;gap:5px;border:1px solid var(--border);background:var(--bg);border-radius:7px;padding:4px 9px;font-size:12px;font-weight:500;cursor:pointer}
.pill.on{background:var(--pale-sage);border-color:var(--sage)}.pill svg{width:13px;height:13px}
.status{font-size:10.5px;font-weight:600;border-radius:999px;padding:2px 9px;border:1px solid}
.st-Optimal{color:var(--forest);background:var(--pale-sage);border-color:var(--sage)}
.st-Warning{color:var(--amber);background:oklch(95% .04 85);border-color:oklch(86% .07 85)}
.st-Critical{color:var(--coral);background:oklch(95% .03 30);border-color:oklch(85% .07 30)}
.st-Unrated,.st-No{color:var(--muted);background:var(--bg);border-color:var(--border)}
.kv{display:grid;grid-template-columns:auto 1fr;gap:5px 14px;font-size:12.5px;margin-top:10px}.kv span{color:var(--muted)}.kv b{font-weight:600;text-align:right}
/* insights dark panel */
.ins{background:var(--forest);color:#fff;border-radius:var(--radius);padding:18px}
.ins .ring{display:flex;align-items:center;gap:14px;margin:8px 0 14px}
.ring svg{width:78px;height:78px}.ins .item{border-top:1px solid oklch(42% .045 172);padding:11px 0}
.ins .k{font:600 10px 'DM Sans',system-ui,sans-serif;letter-spacing:.12em;color:var(--sage);text-transform:uppercase}.ins .n{font:700 15px Manrope,system-ui,sans-serif;margin-top:3px}
.ins .s{font-size:11.5px;opacity:.75}.ins .opp{display:flex;gap:9px;margin:8px 0;font-size:12.5px;font-weight:600}
.ins .opp:before{content:"";flex:0 0 6px;height:6px;border-radius:50%;background:var(--coral);margin-top:6px}
.ins a.more{display:block;text-align:center;border-top:1px solid oklch(42% .045 172);padding-top:10px;margin-top:6px;font-size:12px;text-decoration:none;opacity:.85}
/* map */
.mapwrap{position:relative;border-radius:8px;overflow:hidden;border:1px solid var(--border)}
#map,.map{height:520px;background:#1d2a24}
.mapfoot{display:flex;justify-content:space-between;font-size:11px;color:var(--muted);padding:8px 2px 0}
.lbl-pin{background:#fff;color:var(--fg);border-radius:6px;padding:3px 7px;font:600 11px 'DM Sans',system-ui,sans-serif;box-shadow:0 2px 6px #0003;white-space:nowrap}
.pin{width:30px;height:30px;border-radius:50%;background:var(--forest);border:2px solid #fff;display:flex;align-items:center;justify-content:center;color:#fff;box-shadow:0 3px 8px #0005}
.pin svg{width:15px;height:15px}.pin.Warning{background:var(--amber)}.pin.Critical{background:var(--coral)}.pin.No,.pin.Unrated{background:#56645c}
.leaflet-popup-content{font-family:'DM Sans',system-ui,sans-serif;font-size:12.5px}
.leaflet-tooltip{background:transparent;border:0;box-shadow:none;padding:0}.leaflet-tooltip:before{display:none}
.far .leaflet-tooltip{display:none}
/* tables */
table.t{width:100%;border-collapse:collapse;font-size:12.5px}table.t th{text-align:left;font:600 10.5px 'DM Sans',system-ui,sans-serif;letter-spacing:.1em;text-transform:uppercase;color:var(--muted);border-bottom:1px solid var(--border);padding:8px}
table.t td{border-bottom:1px solid var(--border);padding:8px;vertical-align:top}
.scroll{max-height:420px;overflow:auto}
.verdict{border-radius:var(--radius);padding:16px 18px;border:1px solid}
.v-ok{background:var(--pale-sage);border-color:var(--sage)}.v-no{background:oklch(96% .03 85);border-color:oklch(86% .07 85)}
.v-bad{background:oklch(95% .03 30);border-color:oklch(85% .07 30)}.v-wait{background:var(--ivory);border-color:var(--border)}
.verdict .big{font:700 22px Manrope,system-ui,sans-serif}.verdict p{margin:8px 0 0;line-height:1.6;font-size:13px}
.stat{font:700 24px Manrope,system-ui,sans-serif}.stat+div{font-size:11.5px;color:var(--muted)}
.rec{background:var(--ivory);border:1px solid var(--border);border-radius:var(--radius);margin-bottom:10px}
.rec .h{display:flex;align-items:center;gap:14px;padding:14px 16px;cursor:pointer}
.rec .ic{width:36px;height:36px;border-radius:8px;background:var(--pale-sage);display:flex;align-items:center;justify-content:center;flex:0 0 36px}
.rec .ic svg{width:17px;height:17px}.rec .t1{font:600 10px 'DM Sans',system-ui,sans-serif;letter-spacing:.12em;color:var(--muted);text-transform:uppercase}
.rec .t2{font:600 15px Manrope,system-ui,sans-serif;margin:2px 0}.rec .t3{font-size:12px;color:var(--muted)}.rec .body{padding:0 16px 16px 66px;display:none;line-height:1.6}
.rec.open .body{display:block}
.field{display:flex;flex-direction:column;gap:4px;font-size:11.5px;color:var(--muted);font-weight:600}
.field input,.field select,textarea{border:1px solid var(--border);background:#fff;border-radius:7px;padding:7px 9px;font:500 13px 'DM Sans',system-ui,sans-serif;color:var(--fg)}
.row{display:flex;gap:10px;align-items:flex-end;flex-wrap:wrap}
.note{font-size:12px;color:var(--muted);line-height:1.55}
.msg{font-size:12.5px;font-weight:600;color:var(--forest);margin-top:8px}.msg.err{color:var(--coral)}
.dev{display:flex;align-items:center;gap:12px}.dev .sw{width:42px;height:24px;border-radius:999px;background:#cfd6cf;position:relative;transition:.2s}
.dev .sw:after{content:"";position:absolute;top:3px;left:3px;width:18px;height:18px;border-radius:50%;background:#fff;transition:.2s}
.dev .sw.on{background:var(--forest)}.dev .sw.on:after{left:21px}
.cam{background:#111;border-radius:8px;width:100%;aspect-ratio:4/3;object-fit:cover}
input[type=range]{width:100%;accent-color:oklch(33.4% .051 173)}
.badge{font-size:10.5px;font-weight:700;border-radius:999px;padding:2px 8px}
.b-open{background:oklch(95% .04 85);color:var(--amber)}.b-approved{background:var(--pale-sage);color:var(--forest)}.b-rejected{background:oklch(95% .03 30);color:var(--coral)}
.hide{display:none}
@media(max-width:1100px){.g4,.g5{grid-template-columns:repeat(2,minmax(0,1fr))}.split{grid-template-columns:1fr}.app{grid-template-columns:1fr}.sb{position:relative;height:auto}}
</style>
</head>
<body>
<div class="app">
<aside class="sb">
  <div class="brand"><i data-lucide="leaf" style="color:var(--lime)"></i>GreenLedger<span class="dot"></span></div>
  <div class="lbl">WORKSPACE</div>
  <nav class="nav" id="nav"></nav>
  <div class="sbfoot"><b>Real open data</b><div class="s">Concordia University · CC BY 4.0. Live room uses your camera.</div>
  <div class="t">IC26-05-01 · Climate Action</div></div>
</aside>
<div class="main">
  <div class="top"><div class="ttl" id="ttl">Overview</div>
    <span class="chip"><i data-lucide="building-2" style="width:14px"></i>Concordia University · reference campus</span>
    <div class="search"><i data-lucide="search" style="width:15px"></i><input id="q" placeholder="Search buildings" list="blist"><datalist id="blist"></datalist></div>
    <span class="chip" style="background:var(--forest);color:#fff;border-color:var(--forest)">Real data</span>
    <a class="bell" href="#/analytics" title="Alerts"><i data-lucide="bell" style="width:18px"></i><i id="belldot"></i></a>
    <div class="av">GL</div>
  </div>
  <div class="page" id="page">Loading…</div>
</div>
</div>
<script>
const NAV=[["overview","Overview","layout-dashboard"],["twin","Campus Digital Twin","map"],["prove","Live vs Ghost Twin","git-compare-arrows"],
["buildings","Buildings","building"],["energy","Energy","zap"],["water","Water","droplet"],["waste","Waste Management","recycle"],
["ewaste","E-waste","cpu"],["carbon","Carbon Footprint","cloud"],["room","Live Room Control","video"],
["recs","AI Recommendations","lightbulb"],["analytics","Analytics","line-chart"],["reports","Reports","file-text"],["settings","Settings","settings-2"]];
const $=s=>document.querySelector(s), page=()=>$("#page");
const api=(u,o)=>fetch(u,o).then(r=>r.json());
const post=(u,b)=>api(u,{method:"POST",headers:{"Content-Type":"application/json"},body:JSON.stringify(b||{})});
const f0=v=>v==null||isNaN(v)?"—":Math.round(v).toLocaleString();
const f1=v=>v==null||isNaN(v)?"—":(+v).toLocaleString(undefined,{maximumFractionDigits:1,minimumFractionDigits:1});
const fk=v=>v==null||isNaN(v)?"—":(Math.abs(v)>=1e4?(v/1000).toFixed(1)+"k":f0(v));
const esc=s=>String(s??"").replace(/[&<>"]/g,c=>({"&":"&amp;","<":"&lt;",">":"&gt;",'"':"&quot;"}[c]));
const C={forest:"#24503f",sage:"#b7cc99",lime:"#cde46a",amber:"#9a6b1f",coral:"#ad4530",muted:"#7f8f86",grid:"#e3e6da",water:"#3a7a8a",ghost:"#97a59c"};
Chart.defaults.font.family="'DM Sans',system-ui,sans-serif";Chart.defaults.color="#6d7c73";Chart.defaults.font.size=11;
let charts=[],MAP=null,S=null;
function kill(){charts.forEach(c=>c.destroy());charts=[];if(MAP){MAP.remove();MAP=null}stopCam()}
function chart(el,cfg){const c=new Chart(typeof el=="string"?$(el):el,cfg);charts.push(c);return c}
function icons(){try{lucide.createIcons()}catch(e){}}
function head(icon,title,sub){return `<div class="ph"><span class="back" onclick="location.hash='#/overview'">← Overview</span>
<i data-lucide="${icon}" style="width:20px"></i><h1>${title}</h1><span class="tag">Real data</span></div>${sub?`<div class="sub">${sub}</div>`:""}`}
const axes=(yt)=>({x:{grid:{display:false},ticks:{maxTicksLimit:10}},y:{grid:{color:C.grid},title:{display:!!yt,text:yt}}});
function spark(el,data,dark){chart(el,{type:"line",data:{labels:data.map((_,i)=>i),datasets:[{data,borderColor:dark?C.lime:C.forest,borderWidth:1.6,pointRadius:0,tension:.35,fill:true,backgroundColor:dark?"rgba(205,228,106,.12)":"rgba(36,80,63,.07)",spanGaps:true}]},
options:{plugins:{legend:{display:false},tooltip:{enabled:false}},scales:{x:{display:false},y:{display:false}},responsive:true,maintainAspectRatio:false}})}
function ring(p){const r=32,c=2*Math.PI*r,d=c*(Math.max(0,Math.min(100,p))/100);return `<svg viewBox="0 0 80 80"><circle cx="40" cy="40" r="${r}" stroke="oklch(42% .045 172)" stroke-width="7" fill="none"/>
<circle cx="40" cy="40" r="${r}" stroke="${C.lime}" stroke-width="7" fill="none" stroke-dasharray="${d} ${c}" transform="rotate(-90 40 40)" stroke-linecap="round"/>
<text x="40" y="41" text-anchor="middle" fill="#fff" font-family="Manrope,system-ui,sans-serif" font-weight="800" font-size="15">${Math.round(p)}%</text><text x="40" y="54" text-anchor="middle" fill="#cfe0c9" font-size="7">TRUST</text></svg>`}
function insights(s){const i=s.insights;return `<div class="ins"><div class="ct" style="color:var(--sage)">Campus insights</div>
<div class="ring">${ring(i.ring)}<div><div class="n">${i.ring_label}</div><div class="s">${esc(i.ring_text)}</div></div></div>
<div class="item"><div class="k">Highest energy consumer</div><div class="n">${esc(i.top.name)}</div><div class="s">${i.top.value} · ${s.month}</div></div>
${i.best?`<div class="item"><div class="k">Best weekend schedule</div><div class="n">${esc(i.best.name)}</div><div class="s">${i.best.value}</div></div>`:""}
<div class="item"><div class="k">Potential resource savings</div><div class="n">${i.potential.value}</div><div class="s">${esc(i.potential.text)}</div></div>
<div class="item"><div class="k">Priority opportunities</div>${i.opps.map(o=>`<div class="opp"><div>${esc(o.title)}<div class="s" style="font-weight:400">${esc(o.where)} · ${esc(o.value)}</div></div></div>`).join("")}</div>
<a class="more" href="#/recs">All recommendations →</a></div>`}
// ---------------- MAP ----------------
const RES_ICON={electricity:"zap",gas:"flame",water:"droplet"};
async function campusMap(el,onpick,layer,fit){fit=fit||"SGW";const B=await api("/api/buildings");
MAP=L.map(el,{zoomControl:true,attributionControl:true});
const sat=L.tileLayer("https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",{maxZoom:19,attribution:"Imagery © Esri, Maxar, Earthstar Geographics"}).addTo(MAP);
const osm=L.tileLayer("https://{s}.tile.openstreetmap.org/{z}/{x}/{y}.png",{maxZoom:19,attribution:"© OpenStreetMap"});
L.control.layers({"Aerial":sat,"Streets":osm}).addTo(MAP);
const pts=[];B.forEach(b=>{const show=layer=="all"||(layer=="waste"?b.waste_t>0:b.meters.some(m=>m.res==layer));
 if(!show)return;const ic=b.waste_t>0&&!b.meters.length?"recycle":(b.meters[0]?RES_ICON[b.meters[0].res]:"building");
 const st=b.status.split(" ")[0];const m=L.marker([b.lat,b.lon],{icon:L.divIcon({className:"",html:`<div class="pin ${st}"><i data-lucide="${ic}"></i></div>`,iconSize:[30,30],iconAnchor:[15,15]})}).addTo(MAP);
 m.bindTooltip(`<span class="lbl-pin">${esc(b.code)} · ${esc(b.name)}</span>`,{direction:"bottom",offset:[0,14],opacity:1,permanent:true});
 b._m=m;
 m.on("click",()=>onpick&&onpick(b));pts.push([b.lat,b.lon])});
const lab=()=>MAP.getContainer().classList.toggle("far",MAP.getZoom()<17);MAP.on("zoomend",lab);
const fp=B.filter(b=>fit=="all"||b.campus==fit).map(b=>[b.lat,b.lon]);if(fp.length)MAP.fitBounds(fp,{padding:[40,40]});lab();setTimeout(icons,50);MAP.on("zoomend moveend",()=>setTimeout(icons,10));return B}
function buildingCard(b){return `<div class="card"><div class="row" style="justify-content:space-between"><div class="eyebrow">${b.campus} · ${esc(b.code)}</div><span class="status st-${b.status.split(" ")[0]}">${b.status}</span></div>
<h3 style="margin-top:6px">${esc(b.name)}</h3><div class="note">${esc(b.address)}</div>
<div class="kv">${b.meters.length?b.meters.map(m=>`<span>${m.res}</span><b>${esc(m.name)}</b>`).join(""):"<span>Meters</span><b>none in the open data</b>"}
<span>Waste invoiced</span><b>${b.waste_t?f1(b.waste_t)+" t":"—"}</b>${b.landfill_t?`<span>of which landfill</span><b>${f1(b.landfill_t)} t</b>`:""}</div></div>`}
// ---------------- PAGES ----------------
const P={};
P.overview=async()=>{const s=await api("/api/summary");S=s;$("#belldot").style.display=s.alerts?"block":"none";
const K=s.kpis.map(k=>`<div class="card kpi ${k.dark?"dark":k.accent?"sage":""}"><div class="row"><span class="lab">${k.label}</span><i data-lucide="${{energy:"zap",water:"droplet",waste:"recycle",carbon:"cloud"}[k.key]}" style="width:17px;color:${k.dark?C.lime:C.forest}"></i></div>
<div class="v">${k.unit=="%"?f0(k.value):fk(k.value)}<small>${k.unit}</small></div>
<div class="d">${k.delta!=null?`<b>${k.delta>0?"▲":"▼"} ${f1(Math.abs(k.delta))}${k.unit=="%"?" pts":"%"}</b>`:""} ${esc(k.delta_label)}</div>
<canvas id="sp_${k.key}"></canvas><div class="foot">${k.key=="waste"?"diversion by 12-month period":k.key=="carbon"?"t CO₂e by 12-month period":"last 12 months · real meter data"}</div></div>`).join("");
const I=s.intel;
page().innerHTML=`<div class="hero"><div><div class="eyebrow"><i data-lucide="leaf" style="width:13px"></i>A more sustainable campus</div><h1>Campus overview</h1>
<div class="note">Resource performance, in perspective — every reading is real (Concordia University open data), latest full month: <b>${s.month}</b>.</div></div>
<a class="btn" href="#/reports">View reports →</a></div>
<div class="grid g4">${K}</div>
<div class="split" style="margin-top:16px"><div class="card"><div class="row" style="justify-content:space-between;align-items:center"><div><h3>Campus Digital Twin</h3><div class="note">Concordia University · both campuses · real coordinates</div></div><span class="note">● Real data</span></div>
<div class="row" style="margin:12px 0;justify-content:space-between"><div class="row" id="layers">${[["all","All","layers"],["electricity","Energy","zap"],["gas","Gas","flame"],["water","Water","droplet"],["waste","Waste","recycle"]].map(([k,l,i])=>`<span class="pill ${k=="all"?"on":""}" data-k="${k}"><i data-lucide="${i}"></i>${l}</span>`).join("")}</div>
<div class="seg" id="ocamp"><button class="on" data-c="SGW">SGW (downtown)</button><button data-c="LOY">Loyola</button><button data-c="all">Both</button></div></div>
<div class="mapwrap"><div id="map"></div></div><div class="mapfoot"><span>Campus resource overview · status vs ghost twin (${s.month})</span><span>Aerial: Esri World Imagery</span></div></div>
${insights(s)}</div>
<div class="row" style="justify-content:space-between;margin:26px 0 12px"><div><h2 style="font-size:20px">Resource intelligence</h2><div class="note">Efficiency, environmental impact, and opportunities.</div></div><a class="btn sm" href="#/analytics">Analytics →</a></div>
<div class="grid g5">
<div class="card"><div class="ct">⚡ Energy intelligence</div><div class="stat">${f1(I.energy.value)} MWh</div><div>${s.month} · ${I.energy.delta!=null?f1(I.energy.delta)+"% vs ghost twin":""}</div>
<div class="kv"><span>Verified (your actions)</span><b>${f0(I.energy.verified_mwh)} MWh</b><span>Ground-truth test</span><b>${f0(I.energy.groundtruth_mwh)} MWh</b></div></div>
<div class="card"><div class="ct">💧 Water flow</div><div class="stat">${f0(I.water.value)} m³</div><div>${s.month} · last year ${f0(I.water.ly)} m³</div>
<div class="note" style="margin-top:8px">${I.water.reliable?"":"Model not accurate enough for savings claims — raw readings only."} ${I.water.stuck_days} days excluded (meter not measuring).</div></div>
<div class="card"><div class="ct">☁ Carbon impact</div><div class="stat">${f0(I.carbon.base_t)} t CO₂e</div><div>electricity + landfill, baseline year</div>
<div class="note" style="margin-top:8px">Gas (unit unconfirmed): ${f0(I.carbon.gas_lo)}–${f0(I.carbon.gas_hi)} t. Same electricity on India's grid: ${f0(I.carbon.india_t)} t.</div></div>
<div class="card"><div class="ct">♻ Waste management</div><div class="stat">${f0(I.waste.tonnes)} t</div><div>invoiced, baseline year</div>
<div class="kv">${Object.entries(I.waste.split).map(([k,v])=>`<span>${esc(k)}</span><b>${f0(v)} t</b>`).join("")}</div></div>
<div class="card"><div class="ct">🎥 Live room</div><div class="stat">${I.room.people} people</div><div>${I.room.on.length?I.room.on.join(", ")+" ON":"all devices off"}</div>
<div class="note" style="margin-top:8px">${f1(I.room.dev_min)} device-minutes switched off this session.</div><a class="btn sm" style="margin-top:8px" href="#/room">Open camera →</a></div>
</div>`;
s.kpis.forEach(k=>spark("#sp_"+k.key,k.spark,k.dark));
let layer="all",camp="SGW";const draw=async()=>{if(MAP){MAP.remove();MAP=null}await campusMap("map",b=>{L.popup().setLatLng([b.lat,b.lon]).setContent(buildingCard(b)).openOn(MAP)},layer,camp)};
document.querySelectorAll("#ocamp button").forEach(x=>x.onclick=()=>{document.querySelectorAll("#ocamp button").forEach(y=>y.classList.remove("on"));x.classList.add("on");camp=x.dataset.c;draw()});
document.querySelectorAll("#layers .pill").forEach(p=>p.onclick=()=>{document.querySelectorAll("#layers .pill").forEach(x=>x.classList.remove("on"));p.classList.add("on");layer=p.dataset.k;draw()});
await draw();icons()};
P.twin=async()=>{const s=S||await api("/api/summary");page().innerHTML=head("map","Campus Digital Twin")+`<div class="split"><div class="card">
<div class="row" style="justify-content:space-between"><div><h3>Campus Digital Twin</h3><div class="note">Concordia University · SGW (downtown) and Loyola campuses · click a building</div></div>
<div class="seg" id="camp"><button class="on" data-c="SGW">SGW (downtown)</button><button data-c="LOY">Loyola</button><button data-c="all">Both</button></div></div>
<div class="mapwrap" style="margin-top:12px"><div id="map" style="height:640px"></div></div><div class="mapfoot"><span>Pin colour = status vs ghost twin · ${s.month}</span><span>Aerial: Esri World Imagery</span></div></div>
<div><div class="eyebrow">Selection</div><div id="sel" class="note" style="margin:6px 0 14px">Entire campus · click a building on the map.</div>${insights(s)}</div></div>`;
const pick=b=>{$("#sel").innerHTML=buildingCard(b);MAP.setView([b.lat,b.lon],18)};const B=await campusMap("map",pick,"all","SGW");
if(window.PICK){const b=B.find(x=>window.PICK.startsWith(x.code+" ·"));window.PICK=null;if(b)pick(b)}
document.querySelectorAll("#camp button").forEach(x=>x.onclick=()=>{document.querySelectorAll("#camp button").forEach(y=>y.classList.remove("on"));x.classList.add("on");
const pts=B.filter(b=>x.dataset.c=="all"||b.campus==x.dataset.c).map(b=>[b.lat,b.lon]);MAP.fitBounds(pts,{padding:[40,40]})});icons()};
P.prove=async()=>{const [acts,tr]=await Promise.all([api("/api/actions"),api("/api/trust")]);const T=tr.summary;
page().innerHTML=head("git-compare-arrows","Live twin vs Ghost twin","The <b>live twin</b> is what the meter really did. The <b>ghost twin</b> is the same building without the action — trained on the 12 months before and fed the real weather and calendar. The gap is the real saving; placebo tests on 47 fake dates show what noise looks like.")+
`<div class="grid g3" style="grid-template-columns:2fr 1fr 1fr"><div class="card"><div class="ct">Before you trust any of this — normal months where nothing was done</div><canvas id="trust" height="90"></canvas></div>
<div class="card"><div class="ct">Simple before/after dashboard</div><div class="stat" style="color:var(--amber)">${T.naive_saving} / ${T.tests}</div><div>months it reported a "saving" that never happened</div></div>
<div class="card dark"><div class="ct">GreenLedger</div><div class="stat" style="color:var(--lime)">${T.gl_saving} / ${T.tests}</div><div style="color:#cfe0c9">false savings certified (expected ≈ 5% by chance)</div></div></div>
<div class="card" style="margin-top:16px"><div class="row" style="justify-content:space-between"><div class="field" style="min-width:420px">Action / event<select id="act">${acts.map(a=>`<option value="${a.id}">${a.id} · ${esc(a.meter)} · ${a.date} — ${esc(a.title.slice(0,60))}</option>`).join("")}</select></div>
<div class="seg" id="win"><button data-w="28">4 weeks</button><button class="on" data-w="42">6 weeks</button><button data-w="56">8 weeks</button></div></div>
<div style="height:380px;margin-top:14px"><canvas id="tw"></canvas></div><div id="vd" style="margin-top:12px"></div>
<div class="grid g4" id="vs" style="margin-top:12px"></div><div class="grid g2" style="margin-top:12px"><div class="card"><div class="ct">Placebo-in-time · fake action dates last year</div><div style="height:240px"><canvas id="hist"></canvas></div></div>
<div class="card"><div class="ct">Log a real action</div><div class="row"><div class="field" style="flex:2">Meter<select id="nm">${["Point_12","Point_7","Point_6","Point_8","Point_9","Point_10"].map(p=>`<option value="${p}">${p}</option>`).join("")}</select></div>
<div class="field" style="flex:1">Date<input id="nd" placeholder="2020-05-01"></div></div><div class="field" style="margin-top:8px">What was done<input id="nt" placeholder="e.g. HVAC weekend schedule tightened"></div>
<div class="row" style="margin-top:10px"><button class="btn pri" id="nb">Log & verify</button></div><div id="nmsg" class="msg"></div>
<div class="note" style="margin-top:10px">Needs 12 months of readings before the date and 4–8 weeks after. Approved recommendations land here automatically.</div></div></div></div>
<div class="card" style="margin-top:16px"><div class="ct">Action register</div><div class="scroll"><table class="t"><tr><th>ID</th><th>Date</th><th>Meter</th><th>What</th><th>Type</th><th>Verdict</th><th>Counted</th></tr>
${acts.map(a=>`<tr><td>${a.id}</td><td>${a.date}</td><td>${esc(a.meter)}</td><td>${esc(a.title)}</td><td class="note">${esc(a.type)}</td><td><b>${a.verdict}</b></td><td>${a.counted?f0(a.counted)+" "+a.unit:"—"}</td></tr>`).join("")}</table></div></div>`;
const ms=await api("/api/meters");$("#nm").innerHTML=ms.map(m=>`<option value="${m.point}">${m.res} · ${esc(m.name)}</option>`).join("");
chart("#trust",{type:"bar",data:{labels:["Simple before/after dashboard","GreenLedger"],datasets:[{data:[T.naive_saving,T.gl_saving],backgroundColor:[C.amber,C.forest],borderRadius:6}]},
options:{indexAxis:"y",plugins:{legend:{display:false}},scales:{x:{max:T.tests,title:{display:true,text:`months out of ${T.tests} where a "saving" was reported`}},y:{grid:{display:false}}}}});
let w=42;const load=async()=>{const r=await api(`/api/verify/${$("#act").value}?window=${w}`);charts.filter(c=>["tw","hist"].includes(c.canvas.id)).forEach(c=>c.destroy());charts=charts.filter(c=>!["tw","hist"].includes(c.canvas.id));
const cls={"VERIFIED SAVING":"v-ok","VERIFIED INCREASE":"v-bad","NOT PROVEN":"v-no","CONFOUNDED":"v-no"}[r.verdict]||"v-wait";
$("#vd").innerHTML=`<div class="verdict ${cls}"><div class="big">${{"VERIFIED SAVING":"✅","VERIFIED INCREASE":"⛔","NOT PROVEN":"🤔","CONFOUNDED":"⚠️"}[r.verdict]||"⏳"} ${r.verdict}</div><p>${esc(r.explain)}</p>${/Ground-truth/.test(r.type)?`<p class="note"><b>Ground-truth test:</b> a real, documented event used to prove the engine — not a sustainability saving.</p>`:""}</div>`;
if(r["saving_%"]==null){$("#vs").innerHTML="";return}
$("#vs").innerHTML=[["Simple before/after",f1(r["naive_%"])+"%","what most dashboards report"],["Ghost twin says",f1(r["saving_%"])+"%",`95% range ${f1(r["low_%"])} … ${f1(r["high_%"])}%`],["Placebo p-value",(+r.p).toFixed(3),r.fakes.length+" fake dates"],["Counted (real, low end)",f0(r.counted)+" "+r.unit,r.counted_co2_kg?f0(r.counted_co2_kg)+" kg CO₂ avoided":"only if verified"]].map(x=>`<div class="card"><div class="ct">${x[0]}</div><div class="stat">${x[1]}</div><div>${x[2]}</div></div>`).join("");
const lab=[...r.pre_dates,...r.dates],n0=r.pre_dates.length,pad=a=>[...Array(n0).fill(null),...a];
chart("#tw",{type:"line",data:{labels:lab,datasets:[
{label:"band hi",data:pad(r.band_hi),borderWidth:0,pointRadius:0,fill:false},{label:"Ghost twin ± noise (95%)",data:pad(r.band_lo),borderWidth:0,pointRadius:0,fill:"-1",backgroundColor:"rgba(151,165,156,.22)"},
{label:"Ghost twin (no action)",data:[...r.pre_ghost,...r.ghost],borderColor:C.ghost,borderDash:[6,4],borderWidth:2,pointRadius:0},
{label:"Live twin (actual meter)",data:[...r.pre_actual,...r.actual],borderColor:C.forest,borderWidth:2.6,pointRadius:0},
{label:"Cumulative saving",data:pad(r.cum),borderColor:C.amber,borderWidth:2,pointRadius:0,yAxisID:"y2"}]},
options:{maintainAspectRatio:false,interaction:{mode:"index",intersect:false},plugins:{legend:{labels:{filter:i=>i.text!="band hi",boxWidth:14}}},
scales:{x:{grid:{display:false},ticks:{maxTicksLimit:10}},y:{grid:{color:C.grid},title:{display:true,text:r.unit+"/day"}},y2:{position:"right",grid:{display:false},title:{display:true,text:"cumulative "+r.unit}}}}});
const lo=Math.min(...r.fakes,r["saving_%"])-3,hi=Math.max(...r.fakes,r["saving_%"])+3,nb=20,wd=(hi-lo)/nb,cnt=Array(nb).fill(0);r.fakes.forEach(v=>cnt[Math.min(nb-1,Math.floor((v-lo)/wd))]++);
const labs=cnt.map((_,i)=>(lo+wd*(i+.5)).toFixed(1)),ri=Math.min(nb-1,Math.floor((r["saving_%"]-lo)/wd));
const mk=cnt.map((_,i)=>i==ri?Math.max(...cnt):null);
chart("#hist",{type:"bar",data:{labels:labs,datasets:[{label:"fake dates",data:cnt,backgroundColor:C.sage,borderRadius:3,grouped:false},{label:"real result",data:mk,backgroundColor:"rgba(36,80,63,.85)",barPercentage:.35,grouped:false}]},
options:{maintainAspectRatio:false,plugins:{legend:{display:false},tooltip:{callbacks:{title:t=>"≈ "+t[0].label+"% apparent saving"}}},scales:{x:{title:{display:true,text:`apparent saving % on fake dates · dark bar = the real result (${f1(r["saving_%"])}%)`},grid:{display:false}},y:{title:{display:true,text:"fake dates"}}}}})};
$("#act").onchange=load;document.querySelectorAll("#win button").forEach(b=>b.onclick=()=>{document.querySelectorAll("#win button").forEach(x=>x.classList.remove("on"));b.classList.add("on");w=+b.dataset.w;load()});
$("#nb").onclick=async()=>{const r=await post("/api/actions",{point:$("#nm").value,date:$("#nd").value,title:$("#nt").value});$("#nmsg").className="msg"+(r.error?" err":"");$("#nmsg").textContent=r.error||`✔ Logged ${r.id} — ${r.verdict}`;if(!r.error)setTimeout(()=>P.prove(),900)};
const ver=acts.filter(a=>a.verdict=="VERIFIED SAVING").sort((a,b)=>(b.saving||0)-(a.saving||0));if(ver.length)$("#act").value=ver[0].id;await load();icons()};
P.buildings=async()=>{const [ms,B]=await Promise.all([api("/api/meters"),api("/api/buildings")]);
page().innerHTML=head("building","Buildings","Every real meter, its status against its ghost twin this month, and what its model learned.")+
`<div class="grid g3">${ms.map(m=>`<div class="card"><div class="row" style="justify-content:space-between"><i data-lucide="${RES_ICON[m.res]}" style="width:18px"></i><span class="status st-${m.status}">${m.status}</span></div>
<h3 style="margin-top:8px">${esc(m.name)}</h3><div class="note">${m.res} · buildings ${esc(m.buildings)}</div>
<div class="kv"><span>Per day (${S?S.month:"latest"})</span><b>${f0(m.per_day)} ${m.unit}</b><span>vs ghost twin</span><b>${m.delta==null?"—":f1(m.delta)+"%"}</b>
<span>Model accuracy</span><b>${m.accuracy} (${f1(m.cv)}%)</b><span>Ghost twin model</span><b>${esc(m.model)}</b><span>Alerts</span><b>${m.alerts}</b></div>
<div class="note" style="margin-top:8px">${esc(m.learned)}</div><button class="btn sm" style="margin-top:10px" onclick="showMeter('${m.point}')">History →</button></div>`).join("")}</div>
<div class="card" style="margin-top:16px" id="mh"><div class="ct">Meter history — live twin vs ghost twin</div><div style="height:320px"><canvas id="mhc"></canvas></div></div>
<h2 style="font-size:18px;margin:24px 0 10px">Campus buildings with waste or meters</h2><div class="grid g4">${B.filter(b=>b.meters.length||b.waste_t).map(buildingCard).join("")}</div>`;icons();showMeter("Point_7")};
window.showMeter=async p=>{const s=await api("/api/series/"+p);const old=charts.find(c=>c.canvas.id=="mhc");if(old){old.destroy();charts=charts.filter(c=>c!==old)}
chart("#mhc",{type:"line",data:{labels:s.dates,datasets:[{label:"Ghost twin",data:s.ghost,borderColor:C.ghost,borderDash:[5,4],borderWidth:1.5,pointRadius:0},{label:"Live twin (actual) · "+s.name,data:s.actual,borderColor:C.forest,borderWidth:1.8,pointRadius:0}]},
options:{maintainAspectRatio:false,interaction:{mode:"index",intersect:false},scales:axes(s.unit+"/day")}});document.getElementById("mh").scrollIntoView({behavior:"smooth",block:"center"})};
P.energy=async()=>{const e=await api("/api/energy");const s=S||await api("/api/summary");
page().innerHTML=head("zap","Energy","Electricity across both campuses — live twin (meters) against the ghost twin (expected for the real weather and calendar).")+
`<div class="split"><div><div class="card"><div class="ct">Trend · campus electricity (Loyola + EV/GM/MB/TD) · daily kWh</div><div style="height:300px"><canvas id="e1"></canvas></div></div>
<div class="card" style="margin-top:14px"><div class="ct">By meter · kWh per day · ${s.month}</div><div style="height:260px"><canvas id="e2"></canvas></div></div>
<div class="card" style="margin-top:14px"><div class="row" style="justify-content:space-between"><div class="ct">Daily profile · average weekday kW</div><select id="pp" class="pill">${Object.keys(e.profile).map(p=>`<option value="${p}">${p}</option>`).join("")}</select></div><div style="height:260px"><canvas id="e3"></canvas></div></div></div>
<div><div class="eyebrow">Selection</div><div class="note" style="margin:6px 0 14px">Entire campus · ${e.by_meter.length} electricity meters.</div>
<div class="card"><div class="ct">⚡ Energy intelligence</div><div class="stat">${f1(s.intel.energy.value)} MWh</div><div class="note">${s.month} · ${f1(s.intel.energy.delta)}% vs ghost twin</div>
<div class="kv"><span>Verified (your actions)</span><b>${f0(s.intel.energy.verified_mwh)} MWh</b><span>Ground-truth test</span><b>${f0(s.intel.energy.groundtruth_mwh)} MWh</b></div></div>
<div class="card" style="margin-top:12px"><div class="ct">Monthly · MWh</div><div style="height:220px"><canvas id="e4"></canvas></div></div></div></div>`;
const ms=await api("/api/meters");$("#pp").innerHTML=Object.keys(e.profile).map(p=>`<option value="${p}">${esc((ms.find(m=>m.point==p)||{}).name||p)}</option>`).join("");
chart("#e1",{type:"line",data:{labels:e.dates,datasets:[{label:"Ghost twin",data:e.ghost,borderColor:C.ghost,borderDash:[5,4],borderWidth:1.4,pointRadius:0},{label:"Live twin (actual)",data:e.actual,borderColor:C.forest,borderWidth:1.8,pointRadius:0}]},options:{maintainAspectRatio:false,interaction:{mode:"index",intersect:false},scales:axes("kWh/day")}});
chart("#e2",{type:"bar",data:{labels:e.by_meter.map(m=>m.name),datasets:[{label:"kWh/day",data:e.by_meter.map(m=>m.per_day),backgroundColor:e.by_meter.map(m=>({Optimal:C.forest,Warning:C.amber,Critical:C.coral}[m.status]||C.muted)),borderRadius:6}]},options:{maintainAspectRatio:false,plugins:{legend:{display:false}},scales:axes("kWh/day")}});
chart("#e4",{type:"bar",data:{labels:e.months,datasets:[{label:"Actual",data:e.m_actual.map(v=>v/1000),backgroundColor:C.forest,borderRadius:4},{label:"Ghost twin",data:e.m_ghost.map(v=>v/1000),backgroundColor:C.sage,borderRadius:4}]},options:{maintainAspectRatio:false,scales:axes("MWh")}});
const prof=p=>{const d=e.profile[p];const o=charts.find(c=>c.canvas.id=="e3");if(o){o.destroy();charts=charts.filter(c=>c!==o)}
chart("#e3",{type:"line",data:{labels:d.hours.map(h=>String(h).padStart(2,"0")+":00"),datasets:[{label:d.base_label,data:d.base,borderColor:C.ghost,borderDash:[5,4],pointRadius:0},{label:d.now_label,data:d.now,borderColor:C.forest,borderWidth:2.4,pointRadius:0,fill:true,backgroundColor:"rgba(36,80,63,.06)"}]},options:{maintainAspectRatio:false,scales:axes("kW")}})};
$("#pp").onchange=()=>prof($("#pp").value);prof(Object.keys(e.profile)[0]);icons()};
P.water=async()=>{const w=await api("/api/water");
page().innerHTML=head("droplet","Water","Municipal water supply meter — daily use, with the days the meter was not measuring removed.")+
`<div class="split"><div><div class="card"><div class="ct">Trend · daily m³</div><div style="height:300px"><canvas id="w1"></canvas></div></div>
<div class="card" style="margin-top:14px"><div class="ct">Monthly · m³</div><div style="height:240px"><canvas id="w2"></canvas></div></div></div>
<div><div class="card"><div class="ct">💧 Water model</div><div class="stat">${w.accuracy.Accuracy}</div><div class="note">typical daily error ${f1(w.accuracy["Typical daily error %"])}% — ${w.reliable?"usable for savings":"above the ASHRAE Guideline 14 limit, so no water saving is ever claimed"}.</div></div>
<div class="card" style="margin-top:12px"><div class="ct">Data faults found</div><table class="t">${w.faults.map(f=>`<tr><td>${f.from} → ${f.to}</td><td>${f.days} d</td><td>${esc(f.rule)}</td></tr>`).join("")}</table></div></div></div>`;
chart("#w1",{type:"line",data:{labels:w.dates,datasets:[{label:"Ghost twin",data:w.ghost,borderColor:C.ghost,borderDash:[5,4],pointRadius:0,borderWidth:1.3},{label:"Actual",data:w.actual,borderColor:C.water,pointRadius:0,borderWidth:1.8}]},options:{maintainAspectRatio:false,interaction:{mode:"index",intersect:false},scales:axes("m³/day")}});
chart("#w2",{type:"bar",data:{labels:w.months,datasets:[{label:"m³",data:w.monthly,backgroundColor:C.water,borderRadius:4}]},options:{maintainAspectRatio:false,plugins:{legend:{display:false}},scales:axes("m³")}});icons()};
P.waste=async()=>{const w=await api("/api/waste");const y=w.yearly;
page().innerHTML=head("recycle","Waste Management","Real hauling invoices — tonnes, cost and diversion. Bulky waste is shown but left out of the diversion rate (Concordia: split unknown).")+
`<div class="grid g4">${y.filter(r=>r["Diversion %"]!=null).map(r=>`<div class="card ${r==y.filter(q=>q["Diversion %"]!=null).slice(-1)[0]?"sage":""}"><div class="ct">Diversion ${r.period}</div><div class="stat">${f0(r["Diversion %"])}%</div><div>${r["Invoices from"]} → ${r.to}</div></div>`).join("")}
<div class="card dark"><div class="ct">Landfill CO₂e · baseline year</div><div class="stat">${f0(w.landfill_base_co2e)} t</div><div style="color:#cfe0c9">${f0(w.landfill_base_t)} t × EPA factor</div></div></div>
<div class="split" style="margin-top:14px"><div><div class="card"><div class="ct">Trend · tonnes per month</div><div style="height:300px"><canvas id="ws1"></canvas></div></div>
<div class="card" style="margin-top:14px"><div class="ct">By bin location · tonnes (all years)</div><div style="height:260px"><canvas id="ws2"></canvas></div></div></div>
<div><div class="card"><div class="ct">Cost per tonne (as invoiced)</div><div class="kv">${Object.entries(w.cost).map(([k,v])=>`<span>${esc(k)}</span><b>$${f0(v)}</b>`).join("")}</div></div>
<div class="card" style="margin-top:12px"><div class="ct">Data honesty</div><div class="note">${w.no_weight} pickups had no recorded weight — left out, never guessed. "Building" = where the bin or compactor sits.</div></div></div></div>
<div class="card" style="margin-top:14px"><div class="ct">Tonnes per 12-month period</div><div class="scroll"><table class="t"><tr>${Object.keys(y[0]).map(k=>`<th>${esc(k)}</th>`).join("")}</tr>${y.map(r=>`<tr>${Object.values(r).map(v=>`<td>${v==null?"—":typeof v=="number"?f1(v):esc(v)}</td>`).join("")}</tr>`).join("")}</table></div></div>`;
chart("#ws1",{type:"bar",data:{labels:w.months,datasets:Object.entries(w.streams).map(([k,v])=>({label:k,data:v,backgroundColor:w.colors[k]||C.muted,borderRadius:2}))},options:{maintainAspectRatio:false,scales:{x:{stacked:true,grid:{display:false}},y:{stacked:true,grid:{color:C.grid},title:{display:true,text:"t"}}}}});
const bb=w.by_building,keys=Object.keys(bb[0]).filter(k=>!["Building","Total t"].includes(k));
chart("#ws2",{type:"bar",data:{labels:bb.map(r=>r.Building),datasets:keys.map(k=>({label:k,data:bb.map(r=>r[k]),backgroundColor:w.colors[k]||C.muted}))},options:{indexAxis:"y",maintainAspectRatio:false,scales:{x:{stacked:true,grid:{color:C.grid}},y:{stacked:true,grid:{display:false}}}}});icons()};
P.ewaste=async()=>{const d=await api("/api/ewaste");let rows=d.rows.length?d.rows:[Object.fromEntries(d.cols.map(c=>[c,""]))];
const render=()=>{$("#ewt").innerHTML=`<tr>${d.cols.map(c=>`<th>${c}</th>`).join("")}<th></th></tr>`+rows.map((r,i)=>`<tr>${d.cols.map(c=>`<td><input data-i="${i}" data-c="${c}" value="${esc(r[c]??"")}" style="width:100%;border:1px solid var(--border);border-radius:5px;padding:5px;font:12px 'DM Sans',system-ui,sans-serif" ${c=="condition"?'list="conds"':""} ${c=="purchase_date"?'placeholder="YYYY-MM-DD"':""}></td>`).join("")}<td><button class="btn sm warn" onclick="delRow(${i})">✕</button></td></tr>`).join("");
document.querySelectorAll("#ewt input").forEach(el=>el.oninput=()=>{rows[+el.dataset.i][el.dataset.c]=el.value})};
window.delRow=i=>{rows.splice(i,1);if(!rows.length)rows.push(Object.fromEntries(d.cols.map(c=>[c,""])));render()};
page().innerHTML=head("cpu","E-waste Management","List your department's real devices. GreenLedger flags devices past their expected life and routes each one as India's E-Waste (Management) Rules, 2022 require: keep in use, a registered refurbisher, or a registered recycler.")+
`<div class="card"><div class="row" style="justify-content:space-between"><div class="ct">Asset register — type real devices (expected life = CPCB EPR average life for the EEE code)</div>
<div class="row"><a class="btn sm" href="https://eprewastecpcb.in" target="_blank">CPCB EPR portal ↗</a><a class="btn sm" href="/files/ewaste_register_TEMPLATE.csv">Template CSV</a><label class="btn sm">Upload CSV<input type="file" id="ewf" accept=".csv" hidden></label></div></div>
<datalist id="conds"><option>working</option><option>faulty</option><option>non-functional</option></datalist>
<div class="scroll"><table class="t" id="ewt"></table></div><div class="row" style="margin-top:10px"><button class="btn" id="addr">+ Add device</button><button class="btn pri" id="plan">Make the e-waste plan</button></div><div id="ewmsg" class="msg"></div></div>
<div id="ewres" style="margin-top:14px"></div>`;
render();$("#addr").onclick=()=>{rows.push(Object.fromEntries(d.cols.map(c=>[c,""])));render()};
const show=p=>{if(!p||!p.length){$("#ewres").innerHTML=`<div class="card note">No plan yet. Concordia's open data has no e-waste records, so nothing is shown rather than invented.</div>`;return}
const leave=p.filter(r=>r["leaving campus"]);const past=p.filter(r=>r.age_years>=r.expected_life_years).length;
$("#ewres").innerHTML=`<div class="grid g4"><div class="card"><div class="ct">Devices</div><div class="stat">${p.length}</div></div><div class="card"><div class="ct">Past expected life</div><div class="stat" style="color:var(--amber)">${past}</div></div>
<div class="card dark"><div class="ct">Hand over (registered)</div><div class="stat" style="color:var(--lime)">${leave.length}</div><div style="color:#cfe0c9">${f0(leave.reduce((a,r)=>a+(r.weight_kg||0),0))} kg recorded</div></div>
<div class="card"><div class="ct">Embodied CO₂e kept in use</div><div class="stat">${f0(p.filter(r=>r.route=="Registered refurbisher").reduce((a,r)=>a+(r.embodied_kgco2e||0),0))} kg</div><div>from manufacturer PCF sheets</div></div></div>
<div class="split" style="margin-top:14px"><div class="card"><div class="ct">Plan</div><div class="scroll"><table class="t"><tr><th>Device</th><th>Type</th><th>Dept</th><th>Age</th><th>Life</th><th>Condition</th><th>Route</th><th>Why</th></tr>
${p.map(r=>`<tr><td>${esc(r.device_id)}</td><td>${esc(r.device_type)}</td><td>${esc(r.department)}</td><td>${f1(r.age_years)} y</td><td>${f1(r.expected_life_years)} y</td><td>${esc(r.condition)}</td><td><b>${esc(r.route)}</b></td><td class="note">${esc(r.why)}</td></tr>`).join("")}</table></div></div>
<div class="card"><div class="ct">Routes</div><div style="height:260px"><canvas id="ewc"></canvas></div></div></div>`;
const g={};p.forEach(r=>g[r.route]=(g[r.route]||0)+1);chart("#ewc",{type:"doughnut",data:{labels:Object.keys(g),datasets:[{data:Object.values(g),backgroundColor:[C.forest,C.sage,C.amber,C.coral,C.muted]}]},options:{maintainAspectRatio:false,cutout:"62%"}})};
show(d.plan);
$("#plan").onclick=async()=>{const r=await post("/api/ewaste",{rows});$("#ewmsg").textContent=r.plan?`✔ Plan made for ${r.plan.length} devices — also added to the Carbon footprint.`:"Add at least one device.";show(r.plan)};
$("#ewf").onchange=async ev=>{const t=await ev.target.files[0].text();const L=t.trim().split(/\r?\n/),h=L[0].split(",");rows=L.slice(1).map(l=>{const v=l.split(",");return Object.fromEntries(d.cols.map(c=>[c,v[h.indexOf(c)]??""]))});render();$("#ewmsg").textContent=`Loaded ${rows.length} rows — press "Make the e-waste plan".`};icons()};
P.carbon=async()=>{let gas=0;const draw=async()=>{const c=await api("/api/carbon?gas="+gas);
page().innerHTML=head("cloud","Carbon Footprint","GHG Protocol scopes, measured — not estimated. Avoided emissions count only VERIFIED results, at the low end.")+
`<div class="grid g4"><div class="card dark"><div class="ct">Measured · ${c.base_period}</div><div class="stat" style="color:var(--lime)">${f0(Object.values(c.scopes).reduce((a,v)=>a+(v[c.periods.indexOf(c.base_period)]||0),0))} t CO₂e</div><div style="color:#cfe0c9">${gas?"incl. gas at documented unit":"electricity + landfill"}</div></div>
<div class="card"><div class="ct">Scope 1 · gas (unit unconfirmed)</div><div class="stat">${f0(c.gas_lo)}–${f0(c.gas_hi)} t</div><div>cubic feet … documented m³ — either way the biggest source</div></div>
<div class="card"><div class="ct">Scope 2 · electricity</div><div class="stat">${f1((c.scopes["Scope 2 · electricity"]||[])[c.periods.indexOf(c.base_period)])} t</div><div>Québec hydro, 0.5 kg/MWh</div></div>
<div class="card sage"><div class="ct">Same electricity on India's grid</div><div class="stat">${f0(c.lines.filter(l=>l.period==c.base_period&&/electricity/.test(l.scope)).reduce((a,l)=>a+l.amount,0)*c.india_factor/1000)} t</div><div>CEA ${c.india_factor} kg/kWh</div></div></div>
<div class="split" style="margin-top:14px"><div class="card"><div class="row" style="justify-content:space-between"><div class="ct">Trend · t CO₂e by 12-month period</div><label class="note"><input type="checkbox" id="gas" ${gas?"checked":""}> include gas at documented m³ (log scale)</label></div><div style="height:320px"><canvas id="c1"></canvas></div></div>
<div class="card"><div class="ct">Verified avoided emissions</div><table class="t"><tr><th>Action</th><th>Type</th><th>kg CO₂e</th></tr>${c.avoided.map(a=>`<tr><td>${a.action} · ${esc(a.meter)}</td><td class="note">${a.type}</td><td>${f0(a["kg CO2e avoided"])}</td></tr>`).join("")}</table></div></div>
<div class="card" style="margin-top:14px"><div class="ct">Emission factors & sources</div><table class="t"><tr><th>What</th><th>Factor</th><th>Source</th></tr>${c.factors.map(f=>`<tr><td>${esc(f.What)}</td><td>${esc(f.Factor)}</td><td class="note">${esc(f.Source)}</td></tr>`).join("")}</table></div>
<div class="card" style="margin-top:14px"><div class="row" style="justify-content:space-between"><div class="ct">Inventory lines</div><button class="btn sm pri" onclick="dl()">Download carbon report</button></div><div class="scroll"><table class="t"><tr><th>Period</th><th>Scope</th><th>Source</th><th>Amount</th><th>t CO₂e</th><th>Data</th></tr>
${c.lines.map(l=>`<tr><td>${esc(l.period)}</td><td>${esc(l.scope)}</td><td>${esc(l.source)}</td><td>${f0(l.amount)} ${esc(l.unit)}</td><td>${l["t CO2e"]==null?"—":f1(l["t CO2e"])}</td><td class="note">${esc(l.data)}</td></tr>`).join("")}</table></div></div>`;
const col={"Scope 2 · electricity":C.forest,"Scope 3 · waste to landfill":C.sage,"Scope 1 · natural gas":C.coral,"Scope 3 · e-waste":C.amber};
chart("#c1",{type:"bar",data:{labels:c.periods,datasets:Object.entries(c.scopes).map(([k,v])=>({label:k,data:v,backgroundColor:col[k]||C.muted,borderRadius:3}))},options:{maintainAspectRatio:false,scales:{x:{stacked:true,grid:{display:false}},y:{stacked:true,type:gas?"logarithmic":"linear",grid:{color:C.grid},title:{display:true,text:"t CO₂e"}}}}});
$("#gas").onchange=e=>{gas=e.target.checked?1:0;kill();draw()};icons()};await draw()};
window.dl=async()=>{const r=await post("/api/report");location.href="/files/carbon_report.csv"};
// ---------------- LIVE ROOM ----------------
let STREAM=null,TIMER=null,BUSY=false;
function stopCam(){if(TIMER){clearInterval(TIMER);TIMER=null}if(STREAM){STREAM.getTracks().forEach(t=>t.stop());STREAM=null}}
P.room=async()=>{const st=await api("/api/room");
page().innerHTML=head("video","Live Room Control","The camera counts people → fan / lights / AC switch automatically (ASHRAE 55 comfort, ASHRAE 90.1 vacancy). The live twin is this room with GreenLedger; the ghost twin is the same room where nobody switches anything off.")+
`<div class="split"><div><div class="grid g2"><div class="card"><div class="row" style="justify-content:space-between"><div class="ct">Camera</div><div class="row"><button class="btn sm pri" id="cs">Start camera</button><button class="btn sm" id="cx">Stop</button><label class="btn sm">Test photo<input type="file" id="ph" accept="image/*" hidden></label></div></div>
<video id="vid" class="cam" autoplay playsinline muted></video><canvas id="cap" hidden></canvas></div>
<div class="card"><div class="ct">What the AI sees</div><img id="ann" class="cam" alt=""></div></div>
<div class="grid g4" id="devs" style="margin-top:14px"></div><div id="dec" class="verdict v-wait" style="margin-top:12px"><b>Decision:</b> start the camera.</div>
<div class="card" style="margin-top:14px"><div class="ct">Live twin vs ghost twin · this session</div><div style="height:260px"><canvas id="rc"></canvas></div></div>
<div class="card" style="margin-top:14px"><div class="ct">Switching log</div><div class="scroll" style="max-height:260px"><table class="t" id="rl"></table></div></div></div>
<div><div class="card dark"><div class="ct">Energy saved this session</div><div class="stat" id="rk" style="color:var(--lime)">—</div><div id="rk2" style="color:#cfe0c9">type nameplate kW below</div>
<div class="kv" style="color:#fff"><span style="color:#cfe0c9">Device-minutes off</span><b id="rdm">0</b><span style="color:#cfe0c9">CO₂ (India grid)</span><b id="rco">—</b></div></div>
<div class="card" style="margin-top:12px"><div class="ct">Settings</div>
<label class="note"><input type="checkbox" id="auto" checked> Auto-control (switch devices)</label><br><label class="note"><input type="checkbox" id="demo" checked> Demo timing: off 30 s after the room empties</label>
<div class="field" style="margin-top:8px">Otherwise off after empty (min) — ASHRAE 90.1<input id="vac" value="20"></div>
<div class="field" style="margin-top:8px">Detection confidence <span id="cv">0.50</span><input type="range" id="conf" min="0.25" max="0.9" step="0.05" value="0.5"></div>
<div class="field" style="margin-top:8px">Room temperature °C (empty = live outdoor)<input id="rt" placeholder="e.g. 29"></div>
<div class="row" style="margin-top:8px"><div class="field" style="flex:1">City (live weather)<input id="city" value="Bengaluru"></div><button class="btn sm" id="wb">↻</button></div>
<div class="field" style="margin-top:8px">Outdoor 7-day mean °C (only if offline)<input id="od"></div>
<div class="ct" style="margin-top:12px">Nameplate kW (from each label)</div><div class="row"><div class="field" style="flex:1">Fan<input id="kf" placeholder="0.075"></div><div class="field" style="flex:1">Lights<input id="kl"></div><div class="field" style="flex:1">AC<input id="ka"></div></div>
<div class="row" style="margin-top:10px"><button class="btn sm" id="rs">↺ New session</button></div><div class="note" id="sw" style="margin-top:8px"></div></div></div></div>`;
const vals=()=>({auto:$("#auto").checked,demo:$("#demo").checked,vacancy:$("#vac").value,conf:$("#conf").value,room_temp:$("#rt").value,outdoor:$("#od").value,kw_fan:$("#kf").value,kw_lights:$("#kl").value,kw_ac:$("#ka").value});
$("#conf").oninput=()=>$("#cv").textContent=(+$("#conf").value).toFixed(2);
const upd=r=>{if(r.annotated)$("#ann").src=r.annotated;const nm={Fan:"🌀 Fan",Lights:"💡 Lights",AC:"❄️ AC"};
$("#devs").innerHTML=`<div class="card"><div class="ct">People in the room</div><div class="stat">${r.people}</div><div>median of last 3 frames · this frame ${r.raw}</div></div>`+["Fan","Lights","AC"].map(d=>`<div class="card"><div class="ct">${nm[d]}</div><div class="dev"><div class="sw ${r.live[d]?"on":""}" onclick="sw('${d}',${!r.live[d]})"></div><b>${r.live[d]?"ON":"OFF"}${d=="AC"&&r.live.AC&&r.setpoint?" · "+r.setpoint+" °C":""}</b></div><div class="note" style="margin-top:6px">ghost twin: ${r.ghost[d]?"ON":"OFF"}</div></div>`).join("");
if(r.reason)$("#dec").innerHTML=`<b>Decision:</b> ${esc(r.reason)}<div class="note">Temperature used: ${esc(r.temp_source||"—")} · ${r.weather.trm!=null?`outdoor ${f1(r.weather.now)} °C now, 7-day mean ${f1(r.weather.trm)} °C (${esc(r.weather.city)})`:"live weather not loaded"} · ${esc(r.switch)}</div>`;
$("#rk").textContent=r.kwh>0?f1(r.kwh*1000)+" Wh":"0 Wh";$("#rdm").textContent=f1(r.dev_min);$("#rco").textContent=r.kwh>0?f1(r.india_g)+" g":"—";$("#sw").textContent=r.switch;
$("#rl").innerHTML=`<tr><th>Time</th><th>Device</th><th>Action</th><th>People</th><th>Why</th><th>How</th></tr>`+r.log.map(l=>`<tr><td>${l.time}</td><td>${l.device}</td><td><b>${l.action}</b></td><td>${l.people}</td><td class="note">${esc(l.reason)}</td><td class="note">${esc(l.how)}</td></tr>`).join("");
const H=r.history,kw=H.ghost_kw.some(v=>v>0),old=charts.find(c=>c.canvas.id=="rc");const ds=[{label:"Ghost twin (no GreenLedger)",data:kw?H.ghost_kw:H.ghost_on,borderColor:C.ghost,borderDash:[6,4],stepped:true,pointRadius:0,fill:false},
{label:"Live twin (GreenLedger)",data:kw?H.live_kw:H.live_on,borderColor:C.forest,borderWidth:2.5,stepped:true,pointRadius:0,fill:"-1",backgroundColor:"rgba(205,228,106,.35)"},{label:"People",data:H.people,borderColor:C.water,stepped:true,pointRadius:0,yAxisID:"y2"}];
if(old){old.data.labels=H.t;old.data.datasets=ds;old.options.scales.y.title.text=kw?"kW":"devices ON";old.update("none")}else chart("#rc",{type:"line",data:{labels:H.t,datasets:ds},options:{maintainAspectRatio:false,animation:false,scales:{x:{grid:{display:false},ticks:{maxTicksLimit:8}},y:{beginAtZero:true,grid:{color:C.grid},title:{display:true,text:kw?"kW":"devices ON"}},y2:{position:"right",beginAtZero:true,grid:{display:false},title:{display:true,text:"people"}}}}})};
window.sw=async(d,on)=>upd(await post("/api/room/switch",{device:d,on}));
const send=async dataUrl=>{if(BUSY)return;BUSY=true;try{upd(await post("/api/room/frame",{...vals(),image:dataUrl}))}finally{BUSY=false}};
const grab=()=>{const v=$("#vid");if(!v||!v.videoWidth)return;const c=$("#cap"),w=640,h=Math.round(640*v.videoHeight/v.videoWidth);c.width=w;c.height=h;c.getContext("2d").drawImage(v,0,0,w,h);send(c.toDataURL("image/jpeg",.7))};
$("#cs").onclick=async()=>{try{STREAM=await navigator.mediaDevices.getUserMedia({video:{width:640},audio:false});$("#vid").srcObject=STREAM;TIMER=setInterval(grab,2000)}catch(e){$("#dec").innerHTML="<b>Camera not available:</b> "+esc(e.message)+" — use “Test photo”."}};
$("#cx").onclick=stopCam;$("#rs").onclick=async()=>{const old=charts.find(c=>c.canvas.id=="rc");if(old){old.destroy();charts=charts.filter(c=>c!==old)}upd(await post("/api/room/reset"))};
$("#wb").onclick=async()=>upd(await api("/api/weather?city="+encodeURIComponent($("#city").value)));
$("#ph").onchange=ev=>{const fr=new FileReader();fr.onload=()=>send(fr.result);fr.readAsDataURL(ev.target.files[0])};
upd(st);icons()};
// ---------------- RECOMMENDATIONS ----------------
P.recs=async()=>{const R=await api("/api/recs");let filt="open";const today=new Date().toISOString().slice(0,10);
const ic={Electricity:"zap","Natural gas":"flame",Waste:"recycle",Water:"droplet"};
const render=()=>{const list=R.filter(r=>filt=="all"||r.status==filt);$("#rl").innerHTML=list.map(r=>`<div class="rec" id="r_${r.id}"><div class="h" onclick="this.parentNode.classList.toggle('open')">
<div class="ic"><i data-lucide="${ic[r.area]||"lightbulb"}"></i></div><div style="flex:1"><div class="t1">${r.priority} priority · ${esc(r.where)}</div><div class="t2">${esc(r.next)}</div><div class="t3">Potential (not verified): ${esc(r.potential)}</div></div>
<span class="badge b-${r.status}">${r.status=="open"?"waiting":r.status}</span><i data-lucide="chevron-down" class="chev" style="width:18px"></i></div>
<div class="body"><div>${esc(r.finding)}</div>${r.status!="open"?`<div class="note" style="margin-top:6px">${r.status} by ${esc(r.by)||"—"} ${r.reason?"· "+esc(r.reason):""} ${r.action_id?"· logged as "+r.action_id+" (see Live vs Ghost twin)":""}</div>`:
`<div class="row" style="margin-top:10px"><div class="field">Your name / role<input id="by_${r.id}"></div>${r.meter?`<div class="field">Date it was done<input id="dt_${r.id}" value="${today}"></div>`:""}<div class="field" style="flex:1">Reason (required to reject)<input id="rs_${r.id}"></div>
<button class="btn pri" onclick="dec('${r.id}','approved')">✅ Approve</button><button class="btn" onclick="dec('${r.id}','rejected')">✋ Reject</button></div>
<div class="note">${r.meter?"Approving logs it in the action register; the ghost twin verifies it after 4–8 weeks of meter data.":"Waste actions have no meter — track them with the diversion rate."}</div><div class="msg" id="m_${r.id}"></div>`}</div></div>`).join("")||`<div class="card note">Nothing here.</div>`;icons()};
page().innerHTML=head("lightbulb","AI Recommendations")+`<div class="row" style="justify-content:space-between;margin-bottom:14px"><div class="note">Sized from each building's own data — what another Concordia meter already does, or what each 1 °C / 1% is worth. A person approves or rejects every one.</div>
<select id="ft" class="pill"><option value="open">Open recommendations</option><option value="approved">Approved</option><option value="rejected">Rejected</option><option value="all">All recommendations</option></select></div><div id="rl"></div>`;
$("#ft").onchange=e=>{filt=e.target.value;render()};
window.dec=async(id,d)=>{const r=await post("/api/recs/"+id,{decision:d,by:($("#by_"+id)||{}).value,reason:($("#rs_"+id)||{}).value,date:($("#dt_"+id)||{}).value});
if(r.error){$("#m_"+id).className="msg err";$("#m_"+id).textContent=r.error;return}$("#m_"+id).textContent=r.action_id?`✔ Approved → ${r.action_id}: ${r.verdict}`:"✔ Saved";setTimeout(()=>P.recs(),900)};render()};
// ---------------- ANALYTICS ----------------
P.analytics=async()=>{const A=await api("/api/alerts");
page().innerHTML=head("line-chart","Analytics","What-if projections, alerts and unexplained changes.")+
`<div class="card"><div class="ct">🎚 What if we… (projection from each building's own model and last year's real weather)</div>
<div class="grid g4">${[["heat","Lower heating setpoint by",0,3,.5,1,"°C"],["cool","Raise cooling setpoint by",0,3,.5,1,"°C"],["off","Use less on weekends & holidays",0,50,5,10,"%"],["base","Cut always-on electricity",0,20,1,1,"%"]].map(s=>`<div class="field">${s[1]} <b id="v_${s[0]}">${s[5]} ${s[6]}</b><input type="range" id="${s[0]}" min="${s[2]}" max="${s[3]}" step="${s[4]}" value="${s[5]}"></div>`).join("")}</div>
<div class="row" style="margin-top:10px"><div class="field">Electricity tariff per kWh (optional)<input id="tar"></div><label class="note"><input type="checkbox" id="wg"> show gas CO₂ at documented m³ (unconfirmed)</label></div>
<div class="grid g4" id="wi" style="margin-top:12px"></div><div style="height:240px;margin-top:10px"><canvas id="wic"></canvas></div><div class="note">Projection — not a saving until it is done and verified on the Live vs Ghost twin page.</div></div>
<div class="grid g2" style="margin-top:14px"><div class="card"><div class="ct">🚨 Alerts — used more than ever before for that weather & day (≈${f1(A.expected_false)} expected by chance)</div>
${A.alerts.map(a=>`<div style="border-left:3px solid var(--coral);padding:6px 10px;margin:8px 0"><b>${a.id} · ${esc(a.meter)}</b> <span class="note">${a.from} → ${a.to}</span><div class="note">${esc(a["what happened"])}</div></div>`).join("")||"<div class='note'>None.</div>"}</div>
<div class="card"><div class="ct">🔍 Discoveries — sustained drops nobody has explained</div><div class="scroll"><table class="t"><tr><th>Meter</th><th>From</th><th>To</th><th>Less than ghost twin</th></tr>${A.discoveries.map(d=>`<tr><td>${esc(d.meter)}</td><td>${d.from}</td><td>${d.to}</td><td>${esc(d["less than expected"])} (${f1(d["%"])}%)</td></tr>`).join("")}</table></div></div></div>`;
const go=async()=>{["heat","cool","off","base"].forEach(k=>$("#v_"+k).textContent=$("#"+k).value+" "+({heat:"°C",cool:"°C"}[k]||"%"));
const r=await api(`/api/whatif?heat=${$("#heat").value}&cool=${$("#cool").value}&off=${$("#off").value}&base=${$("#base").value}&tariff=${encodeURIComponent($("#tar").value)}&gas=${$("#wg").checked?1:0}`);
$("#wi").innerHTML=`<div class="card dark"><div class="ct">Electricity / year</div><div class="stat" style="color:var(--lime)">${f0(r.kwh/1000)} MWh</div><div style="color:#cfe0c9">${f1(r.co2_e)} t CO₂ (Québec) · ${f0(r.india_t)} t on India's grid</div></div>
<div class="card"><div class="ct">Gas / year</div><div class="stat">${f0(r.m3)}</div><div>${r.co2_g!=null?f0(r.co2_g)+" t CO₂ at documented m³":"meter units · CO₂ once the unit is confirmed"}</div></div>
<div class="card"><div class="ct">Electricity cost / year</div><div class="stat">${r.cost!=null?f0(r.cost):"—"}</div><div>${r.cost!=null?"your tariff × kWh":"type your tariff"}</div></div>
<div class="card sage"><div class="ct">Per day</div><div class="stat">${f0(r.kwh/365)} kWh</div><div>electricity, averaged over the year</div></div>`;
const o=charts.find(c=>c.canvas.id=="wic");if(o){o.destroy();charts=charts.filter(c=>c!==o)}
chart("#wic",{type:"bar",data:{labels:r.rows.map(x=>x.meter+" ("+x.unit+")"),datasets:[["heating",C.coral],["cooling",C.water],["weekends",C.sage],["base load",C.forest]].map(([k,c])=>({label:k,data:r.rows.map(x=>x[k]),backgroundColor:c}))},options:{indexAxis:"y",maintainAspectRatio:false,scales:{x:{stacked:true,grid:{color:C.grid}},y:{stacked:true,grid:{display:false}}}}})};
["heat","cool","off","base","tar","wg"].forEach(k=>$("#"+k).addEventListener("input",go));$("#wg").onchange=go;await go();icons()};
// ---------------- REPORTS ----------------
P.reports=async()=>{const R=[["Campus resource report","HTML · every page in one document","greenledger_report"],["Labelled dataset","ZIP of CSVs · daily readings, ghost twin, faults, alerts","greenledger_concordia"],["Carbon footprint statement","CSV · scopes 1-2-3 with sources","carbon_report.csv"],["Verification ledger","CSV · live vs ghost twin results","verification.csv"],["Recommendations","CSV · with potential sizes","recommendations.csv"],["E-waste plan","CSV · after you make a plan","ewaste_plan.csv"]];
page().innerHTML=head("file-text","Reports")+`<div class="grid g2">${R.map(r=>`<div class="card row" style="justify-content:space-between;align-items:center"><div class="row" style="align-items:center"><i data-lucide="file-text" style="width:18px"></i><div><h3>${r[0]}</h3><div class="note">${r[1]} · real data</div></div></div><button class="btn pri sm" onclick="exp('${r[2]}')">⬇ Export</button></div>`).join("")}</div><div class="msg" id="rm"></div>`;
window.exp=async key=>{$("#rm").textContent="Building…";const r=await post("/api/report");const f=r.files.find(x=>x.startsWith(key));if(!f){$("#rm").textContent="Not available yet (make an e-waste plan first).";return}$("#rm").textContent="✔ "+f;location.href="/files/"+f};icons()};
// ---------------- SETTINGS ----------------
P.settings=async()=>{const s=await api("/api/settings");const tb=rows=>`<div class="scroll"><table class="t"><tr>${Object.keys(rows[0]||{}).map(k=>`<th>${esc(k)}</th>`).join("")}</tr>${rows.map(r=>`<tr>${Object.values(r).map(v=>`<td>${typeof v=="number"?f1(v):esc(v)}</td>`).join("")}</tr>`).join("")}</table></div>`;
page().innerHTML=head("settings-2","Settings & evidence",`Baseline ${s.baseline[0].slice(0,10)} → ${s.baseline[1].slice(0,10)} · monitoring ${s.monitoring[0].slice(0,10)} → ${s.monitoring[1].slice(0,10)}`)+
`<div class="card"><div class="ct">Data sources</div>${s.sources.map(x=>`<div><a href="${x.url}" target="_blank">${esc(x.name)}</a> ${x.licence?"· "+x.licence:""}</div>`).join("")}</div>
<div class="card" style="margin-top:12px"><div class="ct">Meters</div>${tb(s.meters)}</div>
<div class="card" style="margin-top:12px"><div class="ct">Which model is each ghost twin? (12 hidden months)</div>${tb(s.model_check)}</div>
<div class="card" style="margin-top:12px"><div class="ct">Model accuracy (ASHRAE Guideline 14)</div>${tb(s.accuracy)}</div>
<div class="card" style="margin-top:12px"><div class="ct">Bad data found (excluded with reason) · plus ${s.incomplete_days} incomplete days</div>${tb(s.faults)}</div>
<div class="card" style="margin-top:12px"><div class="ct">Every setting and its source</div>${tb(s.parameters)}</div>`;icons()};
// ---------------- ROUTER ----------------
$("#nav").innerHTML=NAV.map(([k,l,i])=>`<a href="#/${k}" data-k="${k}"><i data-lucide="${i}"></i>${l}</a>`).join("");
async function route(){const k=(location.hash.replace("#/","")||"overview");kill();document.querySelectorAll("#nav a").forEach(a=>a.classList.toggle("on",a.dataset.k==k));
const n=NAV.find(x=>x[0]==k);$("#ttl").textContent=n?n[1]:"Overview";page().innerHTML='<div class="note">Loading…</div>';
try{if(!S&&k!="overview")S=await api("/api/summary");await (P[k]||P.overview)()}catch(e){page().innerHTML=`<div class="card"><b>Something went wrong:</b> ${esc(e.message)}</div>`;console.error(e)}window.scrollTo(0,0);icons()}
window.addEventListener("hashchange",route);
api("/api/buildings").then(B=>{$("#blist").innerHTML=B.map(b=>`<option value="${esc(b.code+" · "+b.name)}">`).join("")});
$("#q").onchange=()=>{window.PICK=$("#q").value;if(location.hash=="#/twin")route();else location.hash="#/twin"};
api("/api/summary").then(s=>{S=s;$("#belldot").style.display=s.alerts?"block":"none"});
icons();route();
</script>
</body>
</html>
'''
print(f'Web page ready ({len(WEB_HTML):,} characters)')


## Cell 15 · The data API
FastAPI routes that serve the page and every number on it from the cells above (meters, ghost twins, verification, recommendations, waste, e-waste, carbon, live room).

In [ ]:
# ---- The web app's data API (FastAPI). Every number the pages show comes from the cells above. ----
import base64, io, math
from fastapi import FastAPI, Body
from fastapi.responses import HTMLResponse, JSONResponse, FileResponse, Response
from PIL import Image as _PILImage


def J(x):
    """Make anything JSON-safe (numpy, pandas, NaN → null)."""
    if isinstance(x, dict):
        return {str(k): J(v) for k, v in x.items()}
    if isinstance(x, (list, tuple, set)):
        return [J(v) for v in x]
    if isinstance(x, pd.DataFrame):
        return J(x.to_dict(orient="records"))
    if isinstance(x, pd.Series):
        return J(x.tolist())
    if isinstance(x, (pd.Timestamp, datetime.datetime, datetime.date)):
        return x.isoformat()[:19]
    if isinstance(x, pd.Period):
        return str(x)
    if isinstance(x, (np.integer,)):
        return int(x)
    if isinstance(x, (np.floating, float)):
        return None if (x != x or x in (math.inf, -math.inf)) else float(x)
    if isinstance(x, np.bool_):
        return bool(x)
    if isinstance(x, np.ndarray):
        return J(x.tolist())
    return x


def ok(x):
    return JSONResponse(J(x))


def _mname(p):
    return METER_INFO[p][2]


ELEC_CAMPUS = CAMPUS_TOTAL["electricity"]
_m_ok = [m for m in pd.period_range(DAYS[0], DAYS[-1], freq="M")
         if DAILY.loc[m.start_time:m.end_time.normalize(), ELEC_CAMPUS].notna().sum().min() >= 20]
LATEST = _m_ok[-1]
L0, L1 = LATEST.start_time, LATEST.end_time.normalize()
TYPICAL_MISS = TRUST.groupby("point")["gl_%"].apply(lambda s: float(np.sqrt((s ** 2).mean()))).to_dict()


def _monthly(p, kind="actual"):
    y = DAILY[p] if kind == "actual" else EXPECTED[p].reindex(DAYS).where(DAILY[p].notna())
    return y.resample("MS").sum(min_count=1)


def _campus_monthly(res, kind="actual"):
    return sum(_monthly(p, kind).fillna(0) for p in CAMPUS_TOTAL[res])


def meter_status(p, a=L0, b=L1):
    """vs the ghost twin in the period: Optimal (normal or below) · Warning (above normal) · Critical (alert in period)."""
    y = DAILY[p].loc[a:b].dropna()
    al = ALERT_TABLE[(ALERT_TABLE.point == p) & (pd.to_datetime(ALERT_TABLE["to"]) >= a) & (pd.to_datetime(ALERT_TABLE["from"]) <= b)]
    if p not in RELIABLE or not len(y):
        return "Unrated", None
    e = EXPECTED[p].reindex(y.index).sum()
    d = float((y.sum() - e) / e * 100)
    if len(al):
        return "Critical", d
    if d > TYPICAL_MISS.get(p, 0):
        return "Warning", d
    return "Optimal", d


def summary():
    em_a, em_g = _campus_monthly("electricity"), _campus_monthly("electricity", "ghost")
    wa = _monthly("Point_10")
    last12 = pd.date_range(L0 - pd.DateOffset(months=11), L0, freq="MS")
    e_now, g_now = float(em_a.get(L0, np.nan)), float(em_g.get(L0, np.nan))
    w_now = float(wa.get(L0, np.nan)); w_ly = float(wa.get(L0 - pd.DateOffset(years=1), np.nan))
    full = WASTE_YEARLY.dropna(subset=["Diversion %"])
    cb = CARBON[CARBON.period == _period(BASE_START)]
    base_t = float(cb["t CO2e"].sum())
    st = {p: meter_status(p) for p in DAILY}
    per_day = {p: float(DAILY[p].loc[L0:L1].mean()) for p in DAILY if METER_INFO[p][1] == "electricity"}
    top = max(per_day, key=per_day.get)
    best = max(OFFDAY, key=lambda q: OFFDAY[q]["cut"]) if OFFDAY else None
    W, kwh, m3, ce, cg = what_if(1, 1, 10, 1)
    _seen, _rows = set(), []
    for _, r in RECS[~RECS.priority.str.startswith("1")].iterrows():   # one per kind of action, biggest first
        if r["next step"] not in _seen:
            _seen.add(r["next step"]); _rows.append(r)
    opp = pd.DataFrame(_rows).head(3)
    S = TRUST_SUMMARY
    trust_pct = (S["tests"] - S["gl_saving"] - S["gl_increase"]) / max(S["tests"], 1) * 100
    ver = [(k, v) for k, v in VERIFIED.items() if v["verdict"] == "VERIFIED SAVING"]
    gt = sum(v["counted"] for k, v in ver if "Ground-truth" in str(ACTIONS.set_index("action_id")["type"].get(k, "")))
    act = sum(v["counted"] for k, v in ver if "Ground-truth" not in str(ACTIONS.set_index("action_id")["type"].get(k, "")))
    _, dev_min, _H = room_savings()
    return {
        "campus": "Concordia University · Montréal", "month": LATEST.strftime("%B %Y"),
        "kpis": [
            {"key": "energy", "label": "Energy Consumption", "value": e_now / 1000, "unit": "MWh", "dark": True,
             "delta": (e_now - g_now) / g_now * 100 if g_now else None, "delta_label": "vs ghost twin",
             "spark": [float(em_a.get(d, np.nan)) for d in last12]},
            {"key": "water", "label": "Water Usage", "value": w_now, "unit": "m³",
             "delta": (w_now - w_ly) / w_ly * 100 if w_ly == w_ly and w_ly else None, "delta_label": "vs same month last year",
             "spark": [float(wa.get(d, np.nan)) for d in last12]},
            {"key": "waste", "label": "Waste Diverted", "value": float(full["Diversion %"].iloc[-1]) if len(full) else None, "unit": "%",
             "accent": True, "delta": float(full["Diversion %"].iloc[-1] - full["Diversion %"].iloc[0]) if len(full) > 1 else None,
             "delta_label": f"pts since {full.index[0]}" if len(full) else "",
             "spark": [float(v) for v in full["Diversion %"].tolist()]},
            {"key": "carbon", "label": "Carbon Emissions", "value": base_t, "unit": "t CO₂e",
             "delta": None, "delta_label": f"measured, {_period(BASE_START)} · gas pending unit",
             "spark": [float(v) for v in CARBON_BY_PERIOD.fillna(0).sum(axis=1).tolist()]},
        ],
        "insights": {
            "ring": trust_pct, "ring_label": "Trust score",
            "ring_text": f"{S['tests'] - S['gl_saving'] - S['gl_increase']} of {S['tests']} normal months correctly not counted",
            "top": {"name": _mname(top), "value": f"{per_day[top] / 1000:,.0f} MWh/day"},
            "best": {"name": _mname(best), "value": f"{OFFDAY[best]['cut']:.0f}% less on weekends"} if best else None,
            "potential": {"value": f"≈ {kwh / 365:,.0f} kWh/day", "text": "1 °C setpoints · 10% weekends · 1% always-on (projection)"},
            "opps": [{"title": r["next step"], "where": r["where"], "value": r["potential (not verified)"]} for _, r in opp.iterrows()],
        },
        "intel": {
            "energy": {"value": e_now / 1000, "delta": (e_now - g_now) / g_now * 100 if g_now else None,
                       "verified_mwh": act / 1000, "groundtruth_mwh": gt / 1000},
            "water": {"value": w_now, "ly": w_ly, "reliable": "Point_10" in RELIABLE,
                      "stuck_days": int((FAULTS[(FAULTS.point == "Point_10") & (FAULTS.rule == "not measuring")]).shape[0])},
            "carbon": {"base_t": base_t, "gas_lo": GAS_FT3_T, "gas_hi": GAS_DOC_T,
                       "india_t": float(cb[cb.scope.str.contains("electricity")].amount.sum()) * INDIA_GRID_KG_PER_KWH / 1000},
            "waste": {"tonnes": float(WASTE_BASE.sum()), "split": {STREAM_LABEL[k]: float(v) for k, v in WASTE_BASE.items()}},
            "room": {"people": ROOM["people"], "on": [d for d in DEVICES if ROOM["live"][d]], "dev_min": dev_min},
        },
        "status": {p: {"status": s, "delta": d, "name": _mname(p), "res": METER_INFO[p][1]} for p, (s, d) in st.items()},
        "alerts": len(ALERT_TABLE), "discoveries": len(DISCOVERY_TABLE),
        "recs_open": len(RECS) - (DECISIONS.rec_id.nunique() if len(DECISIONS) else 0),
        "trust": S, "headline": HEADLINE,
    }


def buildings():
    B = BUILDINGS.copy()
    B["code"] = B["Building"].astype(str)
    wt = WASTE.groupby("Building").tonnes.sum()
    wl = WASTE[WASTE.Waste_Type == "Landfill"].groupby("Building").tonnes.sum()
    out = []
    for _, r in B.iterrows():
        c = r.code
        meters = [p for p, v in METER_INFO.items() if (v[3] == "LOY campus" and r.Campus == "LOY") or c in v[3].split(",")]
        stats = [meter_status(p) for p in meters]
        order = {"Critical": 3, "Warning": 2, "Optimal": 1, "Unrated": 0}
        status = max([s for s, _ in stats], key=lambda s: order[s]) if stats else "No meter"
        wcode = "GN" if c == "GNA" else c
        out.append({"code": c, "name": r["Building Long Name"], "campus": r.Campus, "address": r.Address,
                    "lat": float(r.Latitude), "lon": float(r.Longitude), "status": status,
                    "meters": [{"point": p, "name": _mname(p), "res": METER_INFO[p][1]} for p in meters],
                    "waste_t": float(wt.get(wcode, 0)), "landfill_t": float(wl.get(wcode, 0))})
    return out


def meters():
    rows = []
    for p in DAILY:
        s, d = meter_status(p)
        acc = ACC.set_index("point").loc[p]
        rows.append({"point": p, "name": _mname(p), "res": METER_INFO[p][1], "unit": RESOURCE[METER_INFO[p][1]]["unit"],
                     "buildings": METER_INFO[p][3] or "not stated", "per_day": float(DAILY[p].loc[L0:L1].mean()),
                     "status": s, "delta": d, "accuracy": acc["Accuracy"], "cv": float(acc["Typical daily error %"]),
                     "model": acc["Predictions by"], "learned": acc["What the model learned"],
                     "alerts": int((ALERT_TABLE.point == p).sum())})
    return rows


def series(p):
    y, e = DAILY[p], EXPECTED[p].reindex(DAYS)
    return {"name": _mname(p), "unit": RESOURCE[METER_INFO[p][1]]["unit"], "dates": [d.strftime("%Y-%m-%d") for d in DAYS],
            "actual": y.tolist(), "ghost": e.tolist(), "reliable": p in RELIABLE,
            "baseline": [BASE_START.strftime("%Y-%m-%d"), BASE_END.strftime("%Y-%m-%d")],
            "alerts": ALERT_TABLE[ALERT_TABLE.point == p][["id", "from", "to", "what happened"]],
            "discoveries": DISCOVERY_TABLE[DISCOVERY_TABLE.point == p][["from", "to", "less than expected", "%"]],
            "events": [d.strftime("%Y-%m-%d") for d in sorted(set(ACTIONS[ACTIONS.point == p].date))]}


def hourly_profile(p):
    """Average weekday kW by hour: latest month vs the same month in the baseline year."""
    h = HOURLY[p]
    wk = ~h.index.normalize().isin(DAYS[OFF_DAY.values])
    cur = h[(h.index >= L0) & (h.index <= L1 + pd.Timedelta(hours=23)) & wk].groupby(lambda t: t.hour).mean()
    a1 = L0 - pd.DateOffset(years=1)
    base = h[(h.index >= a1) & (h.index < a1 + pd.DateOffset(months=1)) & wk].groupby(lambda t: t.hour).mean()
    return {"hours": list(range(24)), "now": [cur.get(i) for i in range(24)], "base": [base.get(i) for i in range(24)],
            "now_label": LATEST.strftime("%b %Y"), "base_label": a1.strftime("%b %Y")}


def verification(aid, window=42):
    a = ACTIONS.set_index("action_id").loc[aid]
    r = VERIFIED.get(aid) if int(window) == 42 and aid in VERIFIED else verify(a.point, a.date, int(window), ACTIONS[ACTIONS.action_id != aid])
    out = {"action": aid, "title": a.title, "type": a.type, "meter": _mname(a.point), "unit": r["unit"], "verdict": r["verdict"],
           "explain": explain(r), "date": a.date.strftime("%Y-%m-%d")}
    if "saving_%" not in r:
        return out
    T0 = r["date"]
    pre = DAILY[a.point].loc[T0 - pd.Timedelta(days=56): T0 - pd.Timedelta(days=1)].dropna()
    pre = pre[TEMP.reindex(pre.index).notna()]
    gpre = expected(r["model"], pre.index)
    e, y = r["series_exp"], r["series_act"]
    half = r["high_%"] - r["saving_%"]
    out.update({k: r[k] for k in ["saving_%", "low_%", "high_%", "p", "naive_%", "expected", "actual", "counted", "counted_co2_kg", "typical_miss"]})
    out.update({"pre_dates": [d.strftime("%Y-%m-%d") for d in pre.index], "pre_actual": pre.tolist(), "pre_ghost": gpre.tolist(),
                "dates": [d.strftime("%Y-%m-%d") for d in e.index], "ghost": e.tolist(), "actual": y.reindex(e.index).tolist(),
                "band_hi": (e * (1 + half / 100)).tolist(), "band_lo": (e * (1 - half / 100)).tolist(),
                "cum": (e - y.reindex(e.index)).cumsum().tolist(), "fakes": list(r["fakes"]), "model": "gradient boosting" if r["kind"] == "ml" else "change-point"})
    return out


def register():
    rows = []
    for _, a in ACTIONS.iterrows():
        r = VERIFIED.get(a.action_id)
        rows.append({"id": a.action_id, "date": a.date.strftime("%Y-%m-%d"), "meter": _mname(a.point), "point": a.point,
                     "title": a.title, "type": a.type, "verdict": r["verdict"] if r else "—",
                     "saving": r.get("saving_%") if r else None, "counted": r.get("counted") if r else None, "unit": r["unit"] if r else ""})
    return rows


def recs():
    dec = DECISIONS.drop_duplicates("rec_id", keep="last").set_index("rec_id") if len(DECISIONS) else pd.DataFrame()
    out = []
    for _, r in RECS.iterrows():
        d = dec.loc[r.id] if len(dec) and r.id in dec.index else None
        out.append({"id": r.id, "priority": r.priority[4:], "level": r.priority[0], "area": r.area, "where": r["where"],
                    "finding": r.finding, "potential": r["potential (not verified)"], "next": r["next step"],
                    "meter": isinstance(r["_point"], str),
                    "status": "open" if d is None else d["decision"], "by": "" if d is None else d["by"],
                    "reason": "" if d is None else d["reason"], "action_id": "" if d is None else d["action_id"]})
    return out


def waste_data():
    M = WASTE_MONTHLY.copy()
    return {"months": [d.strftime("%Y-%m") for d in M.index],
            "streams": {STREAM_LABEL.get(c, c): M[c].tolist() for c in M.columns},
            "colors": {STREAM_LABEL.get(k, k): v for k, v in STREAM_COLOR.items()},
            "by_building": WASTE_BY_BUILDING.reset_index().rename(columns=STREAM_LABEL),
            "yearly": WASTE_YEARLY.rename(columns=STREAM_LABEL).reset_index(),
            "cost": {STREAM_LABEL.get(k, k): float(v) for k, v in WASTE_COST_PER_T.items()},
            "no_weight": NO_WEIGHT, "landfill_base_t": LANDFILL_BASE_T, "landfill_base_co2e": LANDFILL_BASE_CO2E,
            "types": WASTE_TYPES}


def carbon_data(gas=False):
    C = carbon_by_period(gas)
    return {"periods": list(C.index), "scopes": {c: C[c].tolist() for c in C.columns}, "lines": CARBON,
            "factors": CARBON_FACTORS, "avoided": avoided_table(), "gas_lo": GAS_FT3_T, "gas_hi": GAS_DOC_T,
            "india_factor": INDIA_GRID_KG_PER_KWH, "base_period": _period(BASE_START)}


def settings():
    return {"parameters": PARAMETERS, "model_check": MODEL_CHECK.drop(columns="point"), "accuracy": ACC.drop(columns="point"),
            "faults": fault_periods(FAULTS[FAULTS.rule != "incomplete day"]),
            "incomplete_days": int((FAULTS.rule == "incomplete day").sum()), "meters": METERS.drop(columns=["point"]),
            "baseline": [BASE_START, BASE_END], "monitoring": [MON_START, DAYS[-1]],
            "sources": [{"name": "Concordia University open data (Facilities)", "url": "https://www.concordia.ca/web/open-data.html", "licence": "CC BY 4.0"},
                        {"name": "Concordia API documentation", "url": "https://github.com/opendataConcordiaU/documentation"},
                        {"name": "Open-Meteo (live weather)", "url": "https://open-meteo.com"},
                        {"name": "Esri World Imagery (aerial map tiles)", "url": "https://www.esri.com"}]}


# ---------------- room (camera) ----------------
def room_state(reason=""):
    kwh, dev_min, H = room_savings()
    return {"people": ROOM["people"], "raw": ROOM["raw"], "live": ROOM["live"], "ghost": ROOM["ghost"], "setpoint": ROOM["setpoint"],
            "reason": reason, "temp_source": ROOM["temp_source"], "weather": {k: WEATHER[k] for k in ("city", "now", "trm", "error")},
            "switch": "smart switch (webhook)" if DEVICE_WEBHOOK else "virtual switch — no smart plug connected",
            "kwh": kwh, "dev_min": dev_min, "india_g": kwh * INDIA_GRID_KG_PER_KWH * 1000,
            "history": {"t": [h["time"].strftime("%H:%M:%S") for h in ROOM["history"][-300:]],
                        "people": [h["people"] for h in ROOM["history"][-300:]],
                        "live_kw": [h["live_kw"] for h in ROOM["history"][-300:]], "ghost_kw": [h["ghost_kw"] for h in ROOM["history"][-300:]],
                        "live_on": [h["live_on"] for h in ROOM["history"][-300:]], "ghost_on": [h["ghost_on"] for h in ROOM["history"][-300:]]},
            "log": ROOM["log"][-40:][::-1]}


def _f(x):
    try:
        return float(x) if x not in (None, "", "null") else None
    except (TypeError, ValueError):
        return None


API = FastAPI(title="GreenLedger")


@API.get("/", response_class=HTMLResponse)
def _index():
    return HTMLResponse(WEB_HTML)


@API.get("/api/summary")
def _summary():
    return ok(summary())


@API.get("/api/buildings")
def _buildings():
    return ok(buildings())


@API.get("/api/meters")
def _meters():
    return ok(meters())


@API.get("/api/series/{p}")
def _series(p: str):
    return ok(series(p))


@API.get("/api/energy")
def _energy():
    a, g = _campus_monthly("electricity"), _campus_monthly("electricity", "ghost")
    daily_a = sum(DAILY[p].fillna(0) for p in ELEC_CAMPUS).where(DAILY[ELEC_CAMPUS].notna().all(axis=1))
    daily_g = sum(EXPECTED[p].reindex(DAYS).fillna(0) for p in ELEC_CAMPUS).where(daily_a.notna() & TEMP.notna())
    return ok({"dates": [d.strftime("%Y-%m-%d") for d in DAYS], "actual": daily_a.tolist(), "ghost": daily_g.tolist(),
               "months": [d.strftime("%Y-%m") for d in a.index], "m_actual": a.tolist(), "m_ghost": g.tolist(),
               "by_meter": [m for m in meters() if m["res"] == "electricity"], "profile": {p: hourly_profile(p) for p in ELEC_CAMPUS + ["Point_7"]},
               "baseline": [BASE_START.strftime("%Y-%m-%d"), BASE_END.strftime("%Y-%m-%d")], "events": [d.strftime("%Y-%m-%d") for d in EVENT_DATES]})


EVENT_DATES = sorted(set(ACTIONS.date))


@API.get("/api/water")
def _water():
    s = series("Point_10")
    m = DAILY["Point_10"].resample("MS").sum(min_count=1)
    s.update({"months": [d.strftime("%Y-%m") for d in m.index], "monthly": m.tolist(),
              "faults": fault_periods(FAULTS[(FAULTS.point == "Point_10") & (FAULTS.rule != "incomplete day")]),
              "accuracy": ACC.set_index("point").loc["Point_10"].to_dict()})
    return ok(s)


@API.get("/api/waste")
def _waste():
    return ok(waste_data())


@API.get("/api/carbon")
def _carbon(gas: int = 0):
    return ok(carbon_data(bool(gas)))


@API.get("/api/actions")
def _actions():
    return ok(register())


@API.post("/api/actions")
def _add_action(body: dict = Body(...)):
    try:
        d = pd.Timestamp(str(body.get("date", "")).strip())
        p, title = body.get("point"), str(body.get("title", "")).strip()
        assert p in DAILY and title
    except Exception:
        return ok({"error": "Give a meter, a date (YYYY-MM-DD) and what was done."})
    aid = add_action(d, p, title, _f(body.get("claimed")))
    VERIFIED[aid] = verify(p, d, 42, ACTIONS[ACTIONS.action_id != aid])
    return ok({"id": aid, "verdict": VERIFIED[aid]["verdict"]})


@API.get("/api/verify/{aid}")
def _verify(aid: str, window: int = 42):
    return ok(verification(aid, window))


@API.get("/api/trust")
def _trust():
    T = TRUST.copy(); T["meter"] = T.point.map(_mname)
    return ok({"summary": TRUST_SUMMARY, "rows": T.drop(columns="point")})


@API.get("/api/recs")
def _recs():
    return ok(recs())


@API.post("/api/recs/{rid}")
def _decide(rid: str, body: dict = Body(...)):
    decision = body.get("decision")
    if decision == "rejected" and not str(body.get("reason", "")).strip():
        return ok({"error": "Give a reason for rejecting — it is kept in the log."})
    d = None
    if decision == "approved" and str(body.get("date", "")).strip():
        try:
            d = pd.Timestamp(str(body["date"]).strip())
        except Exception:
            return ok({"error": "Date must be YYYY-MM-DD"})
    aid = decide_rec(rid, decision, body.get("by", ""), body.get("reason", ""), d)
    if aid:
        VERIFIED[aid] = verify(ACTIONS.set_index("action_id").loc[aid, "point"], d, 42, ACTIONS[ACTIONS.action_id != aid])
    return ok({"action_id": aid, "verdict": VERIFIED[aid]["verdict"] if aid else None})


@API.get("/api/whatif")
def _whatif(heat: float = 1, cool: float = 1, off: float = 10, base: float = 1, tariff: str = "", gas: int = 0):
    W, kwh, m3, ce, cg = what_if(heat, cool, off, base)
    if cg is None and gas:
        cg = m3 * GAS_KG_CO2_PER_M3 / 1000
    t = _f(tariff)
    return ok({"rows": W, "kwh": kwh, "m3": m3, "co2_e": ce, "co2_g": cg, "india_t": kwh * INDIA_GRID_KG_PER_KWH / 1000,
               "cost": kwh * t if t else None})


@API.get("/api/alerts")
def _alerts():
    return ok({"alerts": ALERT_TABLE.drop(columns="point").assign(point=ALERT_TABLE.point), "discoveries": DISCOVERY_TABLE,
               "expected_false": EXPECTED_FALSE_ALERTS})


@API.get("/api/ewaste")
def _ewaste():
    return ok({"cols": EWASTE_COLS, "rows": (EWASTE[EWASTE_COLS] if EWASTE is not None else pd.DataFrame(columns=EWASTE_COLS)),
               "plan": EWASTE, "help": EWASTE_HELP})


@API.post("/api/ewaste")
def _ewaste_plan(body: dict = Body(...)):
    global EWASTE
    df = pd.DataFrame(body.get("rows", []), columns=EWASTE_COLS)
    df = df[df.device_id.astype(str).str.strip() != ""]
    if not len(df):
        EWASTE = None
        refresh_ewaste_carbon()
        return ok({"plan": None})
    EWASTE = ewaste_plan(df)
    EWASTE.to_csv(os.path.join(OUT_DIR, "ewaste_plan.csv"), index=False)
    refresh_ewaste_carbon()
    return ok({"plan": EWASTE})


@API.get("/api/settings")
def _settings():
    return ok(settings())


@API.get("/api/room")
def _room():
    return ok(room_state())


@API.post("/api/room/frame")
def _room_frame(body: dict = Body(...)):
    img = _PILImage.open(io.BytesIO(base64.b64decode(body["image"].split(",")[-1]))).convert("RGB")
    frame = np.asarray(img)
    kw = {"Fan": _f(body.get("kw_fan")), "Lights": _f(body.get("kw_lights")), "AC": _f(body.get("kw_ac"))}
    vac = DEMO_VACANCY_MIN if body.get("demo", True) else (_f(body.get("vacancy")) or VACANCY_MIN_DEFAULT)
    ann, n, raw, reason = process_frame(frame, _f(body.get("room_temp")), bool(body.get("auto", True)), vac,
                                        _f(body.get("outdoor")), _f(body.get("conf")) or CONF_DEFAULT, kw)
    buf = io.BytesIO(); _PILImage.fromarray(ann).save(buf, format="JPEG", quality=70)
    st = room_state(reason)
    st["annotated"] = "data:image/jpeg;base64," + base64.b64encode(buf.getvalue()).decode()
    return ok(st)


@API.post("/api/room/switch")
def _room_switch(body: dict = Body(...)):
    dev, on = body.get("device"), bool(body.get("on"))
    if dev in DEVICES:
        with ROOM["lock"]:
            ROOM["live"][dev] = on
            if on:
                ROOM["ghost"][dev] = True
            _log(datetime.datetime.now(), dev, "ON" if on else "OFF", "manual switch", send_command(dev, on))
    return ok(room_state(f"Manual: {dev} {'ON' if on else 'OFF'}"))


@API.post("/api/room/reset")
def _room_reset():
    global ROOM
    ROOM = _new_room()
    return ok(room_state("New session started."))


@API.get("/api/weather")
def _weather(city: str = "Bengaluru"):
    fetch_weather(city)
    return ok(room_state("Weather updated." if WEATHER["trm"] is not None else f"Weather not reachable: {WEATHER['error']}"))


@API.post("/api/report")
def _report():
    h, z = build_report()
    c = os.path.join(OUT_DIR, "carbon_report.csv"); CARBON.to_csv(c, index=False)
    v = os.path.join(OUT_DIR, "verification.csv"); verification_table().to_csv(v, index=False)
    r = os.path.join(OUT_DIR, "recommendations.csv"); RECS.drop(columns=["_value", "_point"]).to_csv(r, index=False)
    files = [h, z, c, v, r] + ([os.path.join(OUT_DIR, "ewaste_plan.csv")] if EWASTE is not None else [])
    return ok({"files": [os.path.basename(f) for f in files]})


@API.get("/files/{name}")
def _file(name: str):
    path = os.path.join(OUT_DIR, os.path.basename(name))
    if not os.path.exists(path):
        return Response(status_code=404)
    return FileResponse(path, filename=os.path.basename(path))


print(f"API ready · {len(API.routes)} routes · latest full month: {LATEST.strftime('%B %Y')}")


## Cell 16 · Open GreenLedger
Prints a **Colab link** (works in your browser, camera included) and a **public link** through a free Cloudflare tunnel (for judges' phones). Keep this cell's runtime alive during the demo.

**Demo-day safety:** run everything before going on stage; screen-record the Live Room working as a backup.

In [ ]:
# ---- Open GreenLedger ----
# Starts the web app and gives you links:
#   • Colab link — opens in your browser (camera works; only you can open it)
#   • Public link — via a free Cloudflare tunnel, for judges' phones (needs internet at the venue)
import threading, subprocess, re, time, socket
import uvicorn

PORT = int(os.environ.get("GL_PORT", "7860"))


def _free(port):
    with socket.socket() as s:
        return s.connect_ex(("127.0.0.1", port)) != 0


if _free(PORT):
    _srv = uvicorn.Server(uvicorn.Config(API, host="0.0.0.0", port=PORT, log_level="warning"))
    threading.Thread(target=_srv.run, daemon=True).start()
    time.sleep(2)
print(f"GreenLedger is running on port {PORT}.")

if os.environ.get("GL_NO_LAUNCH") != "1":
    try:                                                        # 1) Colab's own secure link
        from google.colab.output import eval_js
        print("▶ Open (Colab link):", eval_js(f"google.colab.kernel.proxyPort({PORT})"))
    except Exception:
        print(f"▶ Open: http://localhost:{PORT}")
    try:                                                        # 2) public link for other devices
        CF = os.path.join(OUT_DIR, "cloudflared")
        if not os.path.exists(CF):
            urllib.request.urlretrieve("https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", CF)
            os.chmod(CF, 0o755)
        _cf = subprocess.Popen([CF, "tunnel", "--url", f"http://localhost:{PORT}", "--no-autoupdate"],
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        _t0 = time.time(); _url = None
        while time.time() - _t0 < 40 and not _url:
            m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", _cf.stdout.readline())
            _url = m.group(0) if m else None
        print("▶ Public link (phones, judges):", _url or "not available — use the Colab link")
    except Exception as e:
        print("Public link not available:", e)
